IMPORTS + CONFIG

In [ ]:
import ast
import json
import os
import time
from collections import Counter, defaultdict
from pathlib import Path

import nbformat
import requests
from secret import GITHUB_TOKEN
from experiments import largest_data_transformations
import numpy as np


In [ ]:

# -----------------------------------
# GITHUB CONFIG
# -----------------------------------


HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}

# Better than generic ipynb search
#QUERYS = 'extension:ipynb train_test_split OR StandardScaler'#'extension:ipynb "import pandas"'
QUERYS = [
    'extension:ipynb train_test_split',
    'extension:ipynb sklearn.preprocessing',
    'extension:ipynb LabelEncoder',
    'extension:ipynb OneHotEncoder',
    'extension:ipynb pandas.read_csv',
    'extension:ipynb RandomForestClassifier',
    'extension:ipynb XGBClassifier',
    'extension:ipynb feature engineering',
    'extension:ipynb data cleaning',
    'extension:ipynb data cleansing',
    'extension:ipynb date prep',
    'extension:ipynb exploration',
    'extension:ipynb EDA'
    ]
MAX_NOTEBOOKS_PER_QUERY = 200

SAVE_DIR = Path("notebooks")#Path("notebooks_sanity_test")
SAVE_DIR.mkdir(exist_ok=True)

In [ ]:
transformations = list(largest_data_transformations.keys())

EXTRACT GITHUB DATA

In [ ]:
# -----------------------------------
# SEARCH GITHUB NOTEBOOKS
# -----------------------------------

def search_notebooks(query, page=1):

    url = "https://api.github.com/search/code"

    params = {
        "q": query,
        "per_page": 100,
        "page": page,
    }

    r = requests.get(
        url,
        headers=HEADERS,
        params=params,
    )
    print(r.status_code)
    print(r.text)
    if r.status_code != 200:

        print("GitHub API ERROR")
        print(r.text)

        return []

    data = r.json()

    return data.get("items", [])

# -----------------------------------
# DOWNLOAD NOTEBOOK
# -----------------------------------

def github_raw_url(html_url):

    raw = html_url.replace(
        "github.com",
        "raw.githubusercontent.com"
    )

    raw = raw.replace("/blob/", "/")

    return raw


def download_notebook(item):

    raw_url = github_raw_url(
        item["html_url"]
    )

    try:

        r = requests.get(raw_url)

        if r.status_code != 200:

            print("FAILED:", raw_url)
            return False

        # verify notebook JSON

        try:

            notebook_json = r.json()

        except Exception:

            print(
                "NOT JSON:",
                raw_url
            )

            return False

        # notebook sanity check

        if "cells" not in notebook_json:

            print(
                "NO CELLS:",
                raw_url
            )

            return False

        repo_name = (
            item["repository"]["full_name"]
            .replace("/", "__")
        )

        filename = item["name"]

        out_path = (
            SAVE_DIR /
            f"{repo_name}__{filename}"
        )

        with open(
            out_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                notebook_json,
                f,
            )

        return True

    except Exception as e:

        print("ERROR:", e)

        return False

CRAWL

In [ ]:
# -----------------------------------
# CRAWL NOTEBOOKS
# -----------------------------------

def crawl_notebooks():

    downloaded = 0
    page = 1
    downloaded_notebooks = set()
    for query in QUERYS:
        print("\n*************\nQUERY:", query)
        while downloaded < MAX_NOTEBOOKS_PER_QUERY:

            print(f"\nPAGE {page}")

            items = search_notebooks(query, page)

            if not items:
                print("No more results")
                break

            for item in items:
                repo_name = (
                    item["repository"]["full_name"]
                    .replace("/", "__")
                )
                filename = item["name"]
                notebook_name = f"{repo_name}__{filename}"
                if notebook_name in downloaded_notebooks:
                    print("seen this notebook!")
                    continue

                downloaded_notebooks.add(notebook_name)
                success = download_notebook(item)

                if success:

                    downloaded += 1

                    print(
                        f"Downloaded {downloaded}"
                    )

                if downloaded >= MAX_NOTEBOOKS_PER_QUERY:
                    break

                # avoid rate limits
                time.sleep(0.2)

            page += 1

        downloaded = 0
        page = 1

    print("\nDONE")

In [ ]:
# -----------------------------------
# LOAD NOTEBOOK CODE CELLS
# -----------------------------------

def extract_code_cells(notebook_path):

    try:

        nb = nbformat.read(
            notebook_path,
            as_version=4,
        )

        cells = [

            c["source"]

            for c in nb.cells

            if c.cell_type == "code"
        ]

        return cells

    except Exception as e:

        print(f"FAILED: {notebook_path}")
        print(e)

        return []

SEMANTIC RULE ENGINE

In [ ]:
# -----------------------------------
# SEMANTIC RULE ENGINE
# -----------------------------------

class SemanticPreprocessingVisitor(ast.NodeVisitor):

    def __init__(self):

        self.transforms = []

        # ---------------------------------
        # symbolic quartile tracking
        # ---------------------------------

        self.last_q1 = None
        self.last_q3 = None

    # -----------------------------------
    # HELPERS
    # -----------------------------------

    def resolve_constant(self, node):

        if isinstance(node, ast.Constant):
            return node.value

        return None

    def get_argument(
        self,
        node,
        kw_name,
        position,
    ):

        # keyword arg

        for kw in node.keywords:

            if kw.arg == kw_name:

                return self.resolve_constant(
                    kw.value
                )

        # positional arg

        if len(node.args) > position:

            return self.resolve_constant(
                node.args[position]
            )

        return None

    # -----------------------------------
    # FUNCTION CALLS
    # -----------------------------------

    def visit_Call(self, node):

        # ---------------------------------
        # attribute calls
        # ---------------------------------

        if isinstance(node.func, ast.Attribute):

            attr = node.func.attr.lower()

            # qcut

            if attr == "qcut":

                q_value = self.get_argument(
                    node=node,
                    kw_name="q",
                    position=1,
                )

                if q_value in [2, 5, 10]:

                    self.transforms.append(
                        f"bin_equal_frequency_{q_value}"
                    )

            # cut

            elif attr == "cut":

                bins_value = self.get_argument(
                    node=node,
                    kw_name="bins",
                    position=1,
                )

                if bins_value in [2, 5, 10]:

                    self.transforms.append(
                        f"bin_equal_width_{bins_value}"
                    )

            # winsorize

            elif attr == "winsorize":

                self.transforms.append(
                    "winsorize"
                )

            # explicit iqr function

            elif attr == "iqr":

                self.transforms.append(
                    "IQR"
                )

            elif attr == "zscore":
                self.transforms.append(
                    "zscore"
                )
            # ---------------------------------
            # deduplication
            # ---------------------------------

            elif attr == "drop_duplicates":
                self.transforms.append(
                    "drop_duplicates"
                )

        # ---------------------------------
        # direct function calls
        # ---------------------------------

        elif isinstance(node.func, ast.Name):

            func_name = node.func.id.lower()

            # IsolationForest

            if func_name == "isolationforest":

                self.transforms.append(
                    "isolationForest"
                )

            # winsorize

            elif func_name == "winsorize":

                self.transforms.append(
                    "winsorize"
                )

            # IQR

            elif func_name == "iqr":

                self.transforms.append(
                    "IQR"
                )

            # MinMaxScaler

            elif func_name in [
                "minmaxscaler",
                "minmax_scale",
            ]:

                self.transforms.append(
                    "norm_min_max"
                )

            elif func_name == "zscore":
                self.transforms.append(
                    "zscore"
                )

        self.generic_visit(node)

    # -----------------------------------
    # ASSIGNMENTS
    # -----------------------------------

    def visit_Assign(self, node):

        # only simple assignments

        if len(node.targets) != 1:

            self.generic_visit(node)
            return

        target = node.targets[0]

        # ---------------------------------
        # variable assignment
        # ---------------------------------

        if isinstance(target, ast.Name):

            var_name = target.id

            # ---------------------------------
            # RHS is function call
            # ---------------------------------

            if isinstance(node.value, ast.Call):

                value = node.value

                # ---------------------------------
                # attribute call
                # ---------------------------------

                if isinstance(
                    value.func,
                    ast.Attribute
                ):

                    attr = value.func.attr.lower()

                    # -------------------------
                    # quantile(.25/.75)
                    # -------------------------

                    if attr == "quantile":

                        q_value = self.get_argument(
                            node=value,
                            kw_name="q",
                            position=0,
                        )

                        # Q1

                        if q_value == 0.25:

                            self.last_q1 = var_name

                        # Q3

                        elif q_value == 0.75:

                            self.last_q3 = var_name

        # ---------------------------------
        # df["col"] = np.log(...)
        # ---------------------------------

        if isinstance(target, ast.Subscript):

            value = node.value

            if isinstance(value, ast.Call):

                if isinstance(
                    value.func,
                    ast.Attribute
                ):

                    attr = value.func.attr.lower()

                    if attr in [
                        "log",
                        "log1p",
                    ]:

                        self.transforms.append(
                            "norm_log"
                        )

        self.generic_visit(node)

    # -----------------------------------
    # BINARY OPERATIONS
    # -----------------------------------

    def visit_BinOp(self, node):

        # subtraction

        if isinstance(node.op, ast.Sub):

            left = node.left
            right = node.right

            # Q3 - Q1

            if (
                isinstance(left, ast.Name)
                and isinstance(right, ast.Name)
            ):

                left_name = left.id
                right_name = right.id

                if (
                    left_name == self.last_q3
                    and right_name == self.last_q1
                ):

                    self.transforms.append(
                        "IQR"
                    )

        self.generic_visit(node)


In [ ]:
# -----------------------------------
# EXTRACT TRANSFORMS FROM CODE
# -----------------------------------

def extract_transforms_from_code(code):

    try:
        tree = ast.parse(code)

        visitor = SemanticPreprocessingVisitor()

        visitor.visit(tree)
        return visitor.transforms

    except Exception:
        return []

In [ ]:
# -----------------------------------
# PROCESS SINGLE NOTEBOOK
# -----------------------------------

def process_notebook(notebook_path):

    cells = extract_code_cells(
        notebook_path
    )

    notebook_transforms = []

    for cell in cells:

        transforms = (
            extract_transforms_from_code(
                cell
            )
        )

        notebook_transforms.extend(
            transforms
        )

    return notebook_transforms

In [ ]:
# -----------------------------------
# ANALYZE ALL NOTEBOOKS
# -----------------------------------

def analyze_corpus(folder_names):
    if isinstance(folder_names, (str, Path)):
            folder_names = [folder_names]

    transform_counter = Counter()
    transition_counter = defaultdict(Counter)

    # 1. Collect notebook paths from ALL folders
    notebook_paths = []
    for folder in folder_names:
        save_dir = Path(folder)
        # Extend the main list with notebooks found in this specific folder
        notebook_paths.extend(list(save_dir.rglob("*.ipynb")))

    print(f"Found {len(notebook_paths)} notebooks across {len(folder_names)} folders")

    for idx, notebook_path in enumerate(notebook_paths):
        if idx % 50 == 0:
            print(f"Processing {idx}")

        transforms = process_notebook(notebook_path)
        if transforms:
            print("\n===================")
            print(notebook_path)
            print(transforms)

        # frequency counts
        transform_counter.update(transforms)

        # transitions

        for a, b in zip(transforms[:-1], transforms[1:]):
            transition_counter[a][b] += 1

    # ---------------------------------
    # transform probabilities
    # ---------------------------------

    total = sum(transform_counter.values())

    transform_probabilities = {
        t: c / total for t, c in (transform_counter.items())
    }

    # ---------------------------------
    # transition probabilities
    # ---------------------------------

    transition_probabilities = {}

    for a, next_ops in (transition_counter.items()):
        total_transitions = sum(next_ops.values())
        transition_probabilities[a] = {
            b: c / total_transitions for b, c in (next_ops.items())
        }

    return (
        transform_probabilities,
        transition_probabilities,
    )

RUN CODE

In [ ]:
crawl_notebooks()

In [ ]:
(
    transform_probabilities,
    transition_probabilities,
) = analyze_corpus("notebooks")

In [ ]:
# -----------------------------------
# PRINT TRANSFORM PROBABILITIES
# -----------------------------------
print("\n=== TRANSFORM PROBABILITIES ===\n")

for transform, prob in sorted(transform_probabilities.items(), key=lambda x: x[1], reverse=True):
    print(f"{transform:30s} {prob:.4f}")

# -----------------------------------
# PRINT TRANSITION PROBABILITIES
# -----------------------------------
print("\n=== TRANSITION PROBABILITIES ===\n")

for transform_a, transitions in (transition_probabilities.items()):
    print(f"\n{transform_a} ->")

    for transform_b, prob in sorted(transitions.items(), key=lambda x: x[1], reverse=True):
        print(f"    {transform_b:30s} {prob:.4f}")

In [ ]:
prob_dict = {}
eps = 1e-10
for transform_op in transformations:
    prob_dict[transform_op] = transform_probabilities.get(transform_op, eps)

prob_dict['zscore_clip_3'] = transform_probabilities.get('zscore', eps)
prob_dict['zscore_filter_3'] = transform_probabilities.get('zscore', eps)
print(prob_dict)


In [ ]:
import ast
import re
import pandas as pd


text = r"""
Found 8792 notebooks
Processing 0

===================
notebooks\10xac__crispdm-yabebalFantaye__du1.ipynb
['norm_min_max']

===================
notebooks\14zip__Tubes-Data-Mining-Kelompok-2__Tubes Datmin.ipynb
['drop_duplicates', 'drop_duplicates', 'IQR', 'norm_min_max']

===================
notebooks\1644heihei__Kaggle_Predicting_Loan_Payback__S5E11.ipynb
['IQR']

===================
notebooks\1644heihei__Kaggle_Predicting_Stellar_Class__stellar_classification_executed.ipynb
['norm_log', 'norm_log']

===================
notebooks\2026-1st__team-2__09_bumjun_data3_compact_l1_explainability.ipynb
['drop_duplicates', 'bin_equal_frequency_10']

===================
notebooks\21Alul21__3MTT_Machine_Learning_Capstone_project__Augustine_Alul_Agaji_Capstone_Project.ipynb
['norm_min_max']

===================
notebooks\23eg110e35-eng__k-means__kmeans_student_performance.ipynb
['drop_duplicates']

===================
notebooks\3dvvarMvb__proyecto_DS__analisis_reproducible.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']
Processing 50

===================
notebooks\56Percentt__box-office-dataset-and-modeling__01_box_office_dataset_builder.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\777tharun7__Direct-Market-Access-for-Farmers-__Untitled7.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\78f150__neural_data_science_textbook__data_cleaning.ipynb
['IQR', 'zscore']

===================
notebooks\a-kanaan__dm-practicals__practical4_data-preprocessing.ipynb
['norm_min_max']

C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:51: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:52: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:53: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:54: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:55: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:56: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:57: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:83: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:85: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:120: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:153: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:159: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:165: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:171: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:221: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:227: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:387: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:390: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:391: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:392: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:393: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:394: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:395: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\a37950456__100Day-ML-Marathon__Day_032_HW.ipynb
['norm_min_max']
Processing 100

===================
notebooks\Aathisivabalan__-AI-Based-Predictive-Modeling-for-Network-Threat-Detection__M1-DATA PREPROCESSING.ipynb
['drop_duplicates']

===================
notebooks\abawchen__kaggle-home-credit-default-risk__m_nn_10x.ipynb
['norm_min_max']

===================
notebooks\AbdelghaffourMouhsine__Mham_AWS_Car_Parts_Scraping_Project__code.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Abditivus__sklearn-group__data_cleaning_and_preparation_sk7.ipynb
['norm_log', 'norm_log', 'drop_duplicates']
Processing 150

===================
notebooks\abhijha8287__beverage_p-rice_range_predictor__file.ipynb
['drop_duplicates']

===================
notebooks\abhimanyu1805__auto-sales-analysis-powerbi-python__sales_data.ipynb
['drop_duplicates']

===================
notebooks\AbhishekNatani__Neural_network_project__LEC.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\abodh__Electricity-cost-forecasting-using-machine-learning-and-deep-learning-models__LSTM.ipynb
['norm_min_max']

===================
notebooks\Abstract-Dex__Neural_Nets__classification.ipynb
['norm_min_max']
Processing 200

===================
notebooks\AceTylercholine__npc_playground__Both_Rewarded_Pie_Plot.ipynb
['drop_duplicates']

===================
notebooks\ACMILabs__wikidata-notebooks__data_statements.ipynb
['drop_duplicates']

===================
notebooks\acmilannesta__MIMIC-III_Sepsis_Prediction__Lightgbm_Model.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'norm_min_max']

===================
notebooks\adetbekov__ydl-summer-school__stock_solution.ipynb
['norm_min_max']

===================
notebooks\Adi3220__DSBDA__1)Data Wrangling 1.ipynb
['norm_min_max']
Processing 250

===================
notebooks\AdithyaG-911__Small-Basket-Product-Recommendation-Engine__Recommendation.ipynb
['drop_duplicates']

===================
notebooks\Aditya-1663__HealthGo__tf.ipynb
['norm_min_max']

===================
notebooks\Adityarajj23__CropSense__model.ipynb
['norm_min_max']

===================
notebooks\adityasingh-0803__-Dynamic-Pricing-for-Urban-Parking-Lots__capstone_project.ipynb
['drop_duplicates']

===================
notebooks\adrianferu__BurgerMap-Bucaramanga-__1_dataCleansing.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Adwait197__ML-and-DMV-Practicals__9.ipynb
['drop_duplicates']
Processing 300

<unknown>:39: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:45: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
notebooks\AgneseNahuel__PI_ML_OPS__ML.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\agungsanjayas__SENTIMENT-ANALYSIS-OF-PUBLIC-RESPONSE-ON-TWITTER-TO-COVID-19-VACCINATION-IN-INDONESIA-SVM-METHOD__preprocessing.ipynb
['drop_duplicates']

===================
notebooks\ahadxdev__Urban_Sense_AI__CA_file.ipynb
['IQR']

===================
notebooks\ahadxdev__Urban_Sense_AI__TX_file.ipynb
['IQR']

===================
notebooks\AhmadRafliR__Literasi-Data-dan-Intelligent-Artificial__01_Data_Cleansing.ipynb
['drop_duplicates']
Processing 350

===================
notebooks\AiniNurM__Data-Analysis-with-Python__Analysis User Retention.ipynb
['zscore']

===================
notebooks\AiniNurM__Data-Analysis-with-Python__Basket Analisis Market (1).ipynb
['zscore']

===================
notebooks\aithasahith02__Predictive-analysis-of-Heart-Patients-Re-admission__code.ipynb
['drop_duplicates']

===================
notebooks\ajemily96__machine-learning-challenge__SVC.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\ajhaver4__MD_analysis__Analyze_Energies.ipynb
['drop_duplicates', 'drop_duplicates']

<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:38: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.


===================
notebooks\ajk6604__DS340w_Ecom__retail-rocket-ecommerce-recommender-system (4).ipynb
['drop_duplicates']

===================
notebooks\ajm2004__OpenAI-Sentiment-Analysis__filtering.ipynb
['drop_duplicates', 'IQR']

===================
notebooks\ajogwusalifu__Data-Analyst-Capstone-Project__M3ExploratoryDataAnalysis-lab.ipynb
['IQR']
Processing 400

===================
notebooks\akarapunzl__minds-in-motion-project__Loan_approval6.ipynb
['norm_min_max']

===================
notebooks\AKASH-C-105__auto-mpg-regression-analysis-agent-with-langchain-groq__Reg_model.ipynb
['IQR']

===================
notebooks\akashlimkar09__The_Sparks_Foundation__TSF Task 6 Prediction using Decision Tree  Algorithm.ipynb
['IQR']

===================
notebooks\akramex-dz__Haick-2024-Cloud-Latency-Anticipation-Challenge-Wining-Notebook__VotingRegLgbmRfXgboost_After_Competition_Try.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']
Processing 450

===================
notebooks\albertw__Radio__SOTA WWFF Overlap.ipynb
['drop_duplicates']

===================
notebooks\Albish04__Banking-Dataset_Classification-__02Algorithm Implementation-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\aleicer__proyecto-integrador-III-entrega-1__EA2_Telco_Limpieza.ipynb
['drop_duplicates']

===================
notebooks\alejo-perez-upc-77__TextMining-LIU__TM-L5.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:46: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
notebooks\AleSigno4__movie-recommender__preprocessing.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Alex9667__Week12__.ipynb
['norm_min_max']
Processing 500

===================
notebooks\alfitranurr__DATA-INFORMATION-KNOWLEDGE__MarketBasketAnalysis.ipynb
['zscore']

===================
notebooks\AliArabi55__Digital-Egypt-Pioneers__project22.ipynb
['IQR']

===================
notebooks\alibakh62__orderbot__Simple Bot.ipynb
['drop_duplicates']

===================
notebooks\alifzl__boston_irises__01 Project Cancer Detection.ipynb
['norm_min_max']

===================
notebooks\alio-programmer__DSBDA__4-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\alio-programmer__DSBDA__4.ipynb
['drop_duplicates']
Processing 550

===================
notebooks\allindiacoderlife__Customer-Feedback-Analysis-System__data_preprocessing.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\allu0786ansari__Exploratory_Data_Analysis__Data_Cleaning_Lab.ipynb
['drop_duplicates', 'norm_min_max', 'zscore']

<unknown>:2: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\alphardaniel020-droid__Data-analytics__titanic.ipynb
['norm_min_max']

===================
notebooks\Altaieb-Mohammed__pytorch-tutorial-YouTube-__Ml1.ipynb
['norm_min_max', 'drop_duplicates', 'norm_min_max']

===================
notebooks\alyssa-tsh__CryptoMine__new.ipynb
['norm_log']

===================
notebooks\Aman-Vishwakarma1729__Battery_Health_Insights-and_Prediction_for_Electric_Vehicles__E.ipynb
['norm_min_max']

===================
notebooks\Aman8883__CV_website__.ipynb
['drop_duplicates']
Processing 600

===================
notebooks\amenalahassa__women_poverty_insight__ft_en_ydf_model.ipynb
['norm_min_max']

===================
notebooks\AmirFaridi-2002__Pyxcel__DataAnalysis.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\AmirGadami__ReserVigil__notebook.ipynb
['drop_duplicates', 'norm_log']

===================
notebooks\amirhosseinkarimi7__predictive_analysis__q2.ipynb
['zscore', 'zscore', 'zscore', 'zscore', 'zscore', 'zscore']

===================
notebooks\amirhosseinkarimi7__predictive_analysis__q3.ipynb
['zscore', 'zscore']

===================
notebooks\AmoliR__nlp-for-book-recommendation__eda.ipynb
['drop_duplicates']

===================
notebooks\amr-yasser226__intrusion-detection-kaggle__ydata_profiling_code.ipynb
['drop_duplicates', 'IQR', 'IQR', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'drop_duplicates', 'norm_log']

<unknown>:8: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
notebooks\AmRitJain0442__American-express-prediction-models-__dnn.ipynb
['norm_log', 'bin_equal_frequency_10']

===================
notebooks\amsha16__dvc_churn_prediction_hyperparemeters__DNC.ipynb
['drop_duplicates']

===================
notebooks\AmsterdamUMC__I-care4old__TEMPLATE-CLASS-MULTI-Exercise-HC.ipynb
['drop_duplicates']

===================
notebooks\amtbuzii__Recommendation-System__Vi.ipynb
['drop_duplicates', 'norm_min_max', 'drop_duplicates', 'drop_duplicates']
Processing 650

===================
notebooks\AnantaCoder__JIS-Idea-Jam-2025__crop_model.ipynb
['norm_min_max']

===================
notebooks\anderoos__cbc-customer-segmentation__rfm_analysis_jerry.ipynb
['bin_equal_width_5', 'bin_equal_width_5', 'bin_equal_width_5']

===================
notebooks\Andreihbk__Master-EDA__1.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\Andru-1987__77695_data_science_i_flex__entrega_project_sample.ipynb
['drop_duplicates']
Processing 700

===================
notebooks\andyp14feb__IndonesiaAI_ML_Batch7_Project_04__smokerStatus_v6-MANUAL_FeatureEng.ipynb
['IQR', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\anggiearrizki__house-prix__aparna_experiment.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
notebooks\aniketkumar101__Oasis-Infobyte---Data-Science__car_price_prediction.ipynb
['drop_duplicates']

===================
notebooks\Anish62027__Machine-Learning__iris.ipynb
['norm_min_max']

===================
notebooks\AnishDhanork__CloudComp_mini__4.ipynb
['norm_min_max']
Processing 750

===================
notebooks\Anjali-Khantaal__cryogenic-digital-twin__Digital_Twin_Deployment.ipynb
['norm_min_max']

===================
notebooks\ankidvlpr__Zero-to-AI__day4_pandas_practice.ipynb
['drop_duplicates']

===================
notebooks\ankit-rathi__Data-Science-with-Python__DataSciencePipeline.ipynb
['norm_min_max']

===================
notebooks\ankitkrsingh05__sms_lucknow__Complete_Data_Preprocessing_Feature_Engineering.ipynb
['norm_min_max', 'zscore', 'IQR']

===================
notebooks\Ansell-OK__ckd__model_file.ipynb
['IQR', 'IQR', 'norm_min_max']

===================
notebooks\anthonyrodrigues443__Used-Car-Price-Prediction-Project__main2.ipynb
['drop_duplicates']

===================
notebooks\anthonySemaan01__IEA__CNN.ipynb
['norm_min_max']

<unknown>:6: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.

Processing 800

===================
notebooks\anujmshukla__Machine-Learning-Specialization-by-Deeplearning.ai__Lab_2_Deep_Learning_for_Content_Based_Filtering.ipynb
['norm_min_max']

===================
notebooks\anupkumar08__Learning-Python__detecting-parkinson-disease.ipynb
['norm_min_max']

===================
notebooks\anushaihalapathirana__xai-t1d-ms-prediction-models__SH Prediction models.ipynb
['drop_duplicates']

===================
notebooks\Anushka123Garg__Time-Series-Analysis__rnn_uni.ipynb
['norm_min_max']

===================
notebooks\AozakiHayate__Kaggle__titanic-the-only-notebook-you-need-to-see.ipynb
['bin_equal_width_5']

===================
notebooks\Ape12b__assignment_2_randomized_optimization__tutorial_examples.ipynb
['norm_min_max']

===================
notebooks\apgt60__ai-ml-course__Hands_on_Analyzing_Text_Data_Notebook.ipynb
['drop_duplicates']

===================
notebooks\aqillabf__Data-Scientist__Credit Card Fraud Detection.ipynb
['drop_duplicates', 'norm_log', 'norm_min_max']
Processing 850

===================
notebooks\archx64-ait__ID-LD-ML__sklearn_model.ipynb
['norm_min_max']

===================
notebooks\arifwidianto08__water-quality-classification__classifications.ipynb
['drop_duplicates']

===================
notebooks\Arimoro2020__Predicting_Churn_telecomUsers__01_Data_cleaning.ipynb
['drop_duplicates']

===================
notebooks\Arinatyas__Uas-pmd__Untitled2a.ipynb
['drop_duplicates']

===================
notebooks\ArishAmin__Predicting-House-Prices__Predicting House Prices.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\ArkashJ__CloudComputing__model.ipynb
['drop_duplicates']
Processing 900

===================
notebooks\arnestarnanda__Python-For-Data-Science__07 Modeling.ipynb
['norm_min_max']

===================
notebooks\arnestarnanda__Python-For-Data-Science__10 Cross Validation .ipynb
['norm_min_max']

===================
notebooks\Arpit0324__Nykaa-Analysis__Analysis.ipynb
['drop_duplicates']

===================
notebooks\Arsney091289421__RAG-Augmented-chatbot__plot_compare_reduction.ipynb
['norm_min_max']

===================
notebooks\Art1star__Cleansing_Data__Data Cleansing.ipynb
['IQR']

===================
notebooks\arthi0__Afame-Technologies__HR_data.ipynb
['drop_duplicates']

===================
notebooks\ArTish100__new-repo__two-checkpoint.ipynb
['norm_min_max']

===================
notebooks\ArtyomShabunin__SMOPA-25__lesson_10.ipynb
['drop_duplicates']

===================
notebooks\Arunn1011__Machine-Learning-Algorithms-from-Scratch__knn.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\Arv-98__Feynn-Labs-EV-Market-Segmentation__ev.ipynb
['norm_min_max']

<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.


===================
notebooks\aryam643__Data_Analysis_manipulation__RFM_Analysis.ipynb
['bin_equal_width_5', 'bin_equal_width_5', 'bin_equal_width_5']

===================
notebooks\aryan02420__BITS-F464-Machine-Learning__g11.ipynb
['isolationForest', 'isolationForest']
Processing 950

===================
notebooks\AryanRajeshK__VPN-Fraud-Detection__dt.ipynb
['drop_duplicates']

===================
notebooks\Asad-Afridi__NAVTTC-AI-Course__w4d3 - Introduction_to_Pandas.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\Ashaduzzaman12__Machine_learning__06_Performance_Enhancement_and_Feature_Engineering.ipynb
['norm_min_max', 'norm_log']

===================
notebooks\Asmaa-khorkhash__Renewable-Energy-APPs__Seq2Seq_model.ipynb
['norm_min_max']

===================
notebooks\Asmaelmn__Human_Activity_Recognition__eda.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\asmahanmohamed__asmahanmohamed__M3ExploratoryDataAnalysis-lab (4).ipynb
['IQR', 'IQR', 'IQR', 'IQR', 'IQR', 'IQR']

===================
notebooks\astridwalle__python_jupyter_basics__3_ML.ipynb
['norm_min_max']

===================
notebooks\asumanulusoy__recipe_recommender__vm.ipynb
['norm_min_max']

===================
notebooks\Asv53__Afame-Technologies__HR Data Analysis.ipynb
['drop_duplicates']
Processing 1000

===================
notebooks\atharvabhoite7__Farming_Assistant_Hack-AI-Thon__disease-detection.ipynb
['norm_min_max']

===================
notebooks\athena-masc__Codecademy__Cleaning US Census Data.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\atikahlestar__Data-Analysis__Project_4_User_Segmentation.ipynb
['zscore']

<unknown>:1: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\avartan007__deeplearning__4.ipynb
['norm_min_max']

===================
notebooks\Avvvvvvie__MLDM__L01_Data_Cleaning.ipynb
['drop_duplicates']
Processing 1050

===================
notebooks\AyanGairola__GDSC-BVP__model-nn-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\ayushd150__mlpractice__oppe2p1.ipynb
['norm_min_max']

===================
notebooks\Azri-oss__Deep_Learning_Project_29__processing_data.ipynb
['drop_duplicates']

===================
notebooks\b1060t__Levodopa_Parkinson_MRI__xgb.ipynb
['drop_duplicates', 'zscore']
Processing 1100

<unknown>:2: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:61: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
notebooks\bahanivissiley__fraude_detection_machine_learning__sn.ipynb
['IQR', 'drop_duplicates']

===================
notebooks\BangYudiss__Seleksi-Nasional-Nabil__2.ipynb
['norm_log']

===================
notebooks\basmalaeltabakh__Zag-AI__Task-part1.ipynb
['drop_duplicates']

===================
notebooks\BATspock__deeplearning__NAP.ipynb
['norm_min_max']
Processing 1150

===================
notebooks\bdhruv671__Music-Recommendation-System__p.ipynb
['drop_duplicates']

===================
notebooks\bekeodangyeuqn__Anime_Recommander__model-checkpoint.ipynb
['norm_min_max']

===================
notebooks\BelowzeroA__ComposeUniversity__DA.ipynb
['norm_min_max']

===================
notebooks\beluticona__licentiate-thesis-repo__2.2-mbto-single-optimized-estimators.ipynb
['drop_duplicates']

===================
notebooks\ben1234560__AiLearning-Theory-Applying__2_建模_建筑能源利用率预测.ipynb
['norm_min_max']

<unknown>:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.


===================
notebooks\Benjaxmen__prediccion-puntaje__implementacion_rf.ipynb
['norm_min_max']

===================
notebooks\BenouaklilHodhaifa__Machine_learning_TPs__TP01_Boukacem_Benouaklil_v1-checkpoint.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\Berniceboateng775__Heart-disease-project__age_group_mortality_preprocessing.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 1200

===================
notebooks\Beshoy-Atef-Adel__Beshoy-Atef-Adel--1-house-prices-advanced-regression-techniques__1.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\betr0dalf__TIMO__TIMO_NovikovDV_prac5.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\better-data-science__TensorFlow__002_TensorFlow_Regression.ipynb
['zscore', 'norm_min_max']

===================
notebooks\BgeeDB__expression-annotations-documents__SRP254063.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Bhagas-Dewa__Mini-Portofolio-Properti---Rakamin-Academy__Homework RTC (1).ipynb
['drop_duplicates', 'IQR']

===================
notebooks\bhattacharyasaikat__spam-detection__spam-detection.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\BhavaniPaili__FMML-LAB-1__Regression_Lab_2.ipynb
['norm_min_max']

===================
notebooks\bhnum__mlops-threats__1. dataset.ipynb
['drop_duplicates']
Processing 1250

===================
notebooks\bigzhao__Aliyun_Security_Rank_38th__RNN.ipynb
['norm_min_max']

===================
notebooks\bijaygautamcode__Apple-Stock-Forecast__Final.ipynb
['drop_duplicates']

===================
notebooks\bilhalvadiego__ds-course__Aula_07_pandas.ipynb
['drop_duplicates']

===================
notebooks\BILIM488__projet_ML_2025-__SVRegressor.ipynb
['zscore']

===================
notebooks\biof509__biof509-fall2018__Week3.ipynb
['norm_min_max']

===================
notebooks\BiomedSciAI__biomed-multi-omic__cellxgene_mouse_dataset_split.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Biswajit-17__ml-journey__Level 2 - Scaling, Encoding, Data Preparation.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\bkleyn__restaurant_inspections__data_prep.ipynb
['drop_duplicates', 'norm_log']

===================
notebooks\bkty1122__com6003_cancer_classifier__IT_stacking.ipynb
['norm_min_max']

===================
notebooks\blankwatermelon__kenney02-CS506-ExtraCredit__2.ipynb
['norm_log', 'isolationForest']
Processing 1300

<unknown>:43: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:45: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:46: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:65: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:42: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:87: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
notebooks\borhanitrash__BhashaBodh__bnlp_sylhet_to_chittagong_mbart_50.ipynb
['drop_duplicates']

===================
notebooks\bradwicklund__Springboard__1.0-bjw-relax-inc-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\brandao34__DAATP__5_XGboost_Processamento.ipynb
['norm_min_max']
Processing 1350

===================
notebooks\Breinich__EnvironmentAnalysis__data_cleansing.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\brendonwp__Anomaly_Detection__P1_M2_Full_Solution_082323-checkpoint.ipynb
['norm_min_max']

===================
notebooks\brendonwp__Anomaly_Detection__P1_M4_Full_Solution_082323.ipynb
['norm_min_max', 'isolationForest', 'isolationForest', 'isolationForest']

===================
notebooks\brenwildt__Kaggle_House_Prices__CatBoost.ipynb
['norm_log']

===================
notebooks\BrunoFCastro__ADMF01__a.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 1400

===================
notebooks\busramkurnaz__Collective-Learning__augmentation_llm.ipynb
['drop_duplicates']

===================
notebooks\calcoafrancisca__ML-Group52__ProjectML.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\calebsiow0228__Credit-Approval-Prediction__ML.ipynb
['drop_duplicates']

===================
notebooks\camillaysm__final-project-bootcamp__User_Retention.ipynb
['zscore']
Processing 1450

===================
notebooks\caovy-univers__rootedremedies__model_performance_feature_test.ipynb
['drop_duplicates']

===================
notebooks\Capstone-B10-2022__Training_Experiments__Expt1_other_models.ipynb
['norm_min_max']

===================
notebooks\carlomazzaferro__neoantigen__Immune Stealth MultiProt Analysis From Prot List - Combinatorial Search - New Proteins.ipynb
['drop_duplicates']

===================
notebooks\CarltonLobo__SA-hackathon2__c1.ipynb
['IQR']

===================
notebooks\carotinoid__course-library__sub3, 4.ipynb
['norm_min_max']

<unknown>:109: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:110: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:111: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.

Processing 1500

===================
notebooks\Chaan0210__study-ai__datamining_project.ipynb
['norm_log', 'norm_log']

===================
notebooks\chaewoncutie__ADV-ML-tests__GMM.ipynb
['drop_duplicates']

===================
notebooks\Chaitra-Bhat383__100-Days-of-Machine-Learning__Prescribing_Drugs_using_Consumer_Reviews.ipynb
['drop_duplicates']

===================
notebooks\Chakrapani2122__Data_Science_Foundation_Final_Project__Final_Project.ipynb
['drop_duplicates']

===================
notebooks\charakajg__uom-student-performance-analytics__preprocess_xapi_dataset.ipynb
['norm_min_max']

===================
notebooks\charangt-ai__flood-prediction-pipeline__j.ipynb
['norm_log']
Processing 1550

===================
notebooks\charvibannur__100-Days-of-Machine-learning__Prescribing_Drugs_using_Consumer_Reviews.ipynb
['drop_duplicates']

===================
notebooks\chathumiamarasinghe__ANOVA-Test__ODIN.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\chdoig__scipy2015-blaze-bokeh__2. Blaze.ipynb
['drop_duplicates']

===================
notebooks\Chihiro1998__HVAC_DATA__data_cleaning.ipynb
['zscore', 'zscore']

<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
notebooks\chirchir92__machine-learning-challenge__LR.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\chirchir92__machine-learning-challenge__RF.ipynb
['norm_min_max', 'norm_min_max']
Processing 1600

===================
notebooks\Chu-c-git__Automated_Trading_System__LSTM_single_stock.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\chulminkw__PerfectGuide__2.5 데이터_전처리.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\ChungWasawat__data-notes__Basic_Pandas_&_Polars.ipynb
['drop_duplicates', 'IQR', 'norm_min_max', 'norm_min_max']

===================
notebooks\CJsGit-tech__FinancialBERT-Project__Modeling-Model_TF-IDF_LM_Dictionary_part2.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 1650

===================
notebooks\cmagliano__Proj__WineQualityPrediction.ipynb
['norm_min_max']

===================
notebooks\cmendonsa__brfss-diabetes-trends__2_data_preparation.ipynb
['drop_duplicates']

<unknown>:1: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<unknown>:56: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:57: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.


===================
notebooks\cod3astro__kaggle_ML_competition__kaggle_podcast.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\codehacken__CognitiveQuery__LDA imdb.ipynb
['bin_equal_frequency_2']

===================
notebooks\codevet210__ConsumerComplaintClassifier__Consumer_complaint_classify.ipynb
['drop_duplicates']
Processing 1700

===================
notebooks\CollaboratoryColumbiaClinic__genetics__genetic.ipynb
['norm_min_max']

===================
notebooks\cooperleong00__NCCCU2019-Big-Data-Algorithm-Rank2__avg.ipynb
['norm_min_max']
Processing 1750

===================
notebooks\crowley409__Project-4__model_fitting.ipynb
['norm_min_max']

<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:51: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:55: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:62: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:66: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:74: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
notebooks\CU-ESIIL__CulturalES_WildfireRx__01_Process_Data.ipynb
['drop_duplicates']

===================
notebooks\CumulusCycles__Python_for_Data_Science_and_Machine_Learning__demo.ipynb
['drop_duplicates']
Processing 1800

===================
notebooks\daffaaprilio__masters_thesis__reviewing_scoring_approach.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\damiangajd-db__nbks--90__regularized-linear-models.ipynb
['norm_log', 'norm_log']

===================
notebooks\damiangajd-db__nbks-193__regularized-linear-models.ipynb
['norm_log', 'norm_log']

===================
notebooks\damiangajd-db__nbks-810-no__stacked-regressions-top-4-on-leaderboard.ipynb
['norm_log']

===================
notebooks\DangThiKiemHong__MachineLearning_DeepLearningForcastStockPriceAndMacroeconomics__HPG_sLSTM.ipynb
['norm_min_max']

===================
notebooks\danieleciciani96__tesi_mlops__lstm.ipynb
['norm_min_max']
Processing 1850

===================
notebooks\danimataonrails__python_basics_4_analists__2_python_data.ipynb
['drop_duplicates']

===================
notebooks\daphrut__lab-experiment__1_2_0_check_unique_values.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\Darkprogrammerpb__DeepLearningProjects_when_I_was_a_noob__OMP House Prices.ipynb
['norm_min_max']

===================
notebooks\darpham__open-disclosure-data__data_processing2.ipynb
['drop_duplicates']

===================
notebooks\dartwinshu__rakamin-digital-festival-data-science__Analyze the Behavior of Loan Property Customers.ipynb
['drop_duplicates', 'IQR']

===================
notebooks\darurauf__permasalahan_institusi_pendidikan__Permasalahan_Institusi_Pendidikan.ipynb
['norm_min_max']

===================
notebooks\Data-Science-Community-SRM__Cryptocurrency-Price-Prediction__ts.ipynb
['norm_min_max']

===================
notebooks\datablazor__MPS-Analytics-CPS-Northeastern-Univ__12.06.20.ipynb
['norm_min_max']

===================
notebooks\DataThinkers__Machine-Learning-Projects-Code__Predicting Employee Churn Using Machine Learning (1).ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\datawithalvin__Crude-Oil-Price-Forecasting__build-function.ipynb
['norm_log', 'norm_min_max', 'drop_duplicates']
Processing 1900

===================
notebooks\davew-msft__notebooks-everywhere__DataPreparation.ipynb
['norm_log', 'drop_duplicates']

===================
notebooks\DavidCharles473__3101-AIHT-Proj_229796_Team_2-Public-Transportation-Efficiency-Analysis__Code with Explanation.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Davidelvis__Data_Science_Portfolio__Sales_prediction_rofiah-checkpoint.ipynb
['IQR']

===================
notebooks\DavidnBui__DataStormers__Andrew_testing.ipynb
['drop_duplicates']

<unknown>:39: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:80: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\Dawittsegaye12__movie-recommendation-system__eda.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\de69p__CLV_Prediction_for_EcomX_Retailers__final_model.ipynb
['norm_min_max']

===================
notebooks\deanakbar__Final-Project__DataProcessing.ipynb
['zscore', 'norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 1950

===================
notebooks\debadridtt__IIIT-Delhi-PostGradDiploma-CS-AI__AML Module Assignment 1-checkpoint.ipynb
['drop_duplicates', 'drop_duplicates', 'zscore', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\Debbizz__Diabetes-Prediction-Using-Machine-Learning__Diabetes-Prediction-Project-checkpoint.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\DeepMathukiya__FloodAiHackthon__p3.ipynb
['norm_min_max']

===================
notebooks\DeepMathukiya__FloodAiHackthon__p8.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\Deezzir__DataScience-Python__eda.ipynb
['drop_duplicates']

===================
notebooks\Delyespadon__Employee-performance-and-productivity-__Employee performance Anlysis .ipynb
['drop_duplicates']
Processing 2000

===================
notebooks\Desoky231__bike-store-etl__cleaning.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\desyka-s__DQLab__Data_Science_in_Telco_Data_Cleansing.ipynb
['drop_duplicates', 'drop_duplicates', 'IQR']

===================
notebooks\Devanshi-Y__RTAM_ML_Module__UPDATED_sensor_anomaly_detection.ipynb
['isolationForest']

===================
notebooks\devBOX03__Amazon-Fine-Food-Review__k-NN on Amazon Fine Food Review.ipynb
['drop_duplicates']
Processing 2050

===================
notebooks\dhruv-pandit__bigDataLabsIMS25_26__lab10_bda_pipelines_student.ipynb
['norm_min_max']

===================
notebooks\dhruvg029__Data-Science-Bootcamp__1_knn_implementation.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\dianakhajieva__IBM-Data-Analyst-Capstone-Project__Hands-on Lab 10 Normalizing Data.ipynb
['drop_duplicates']

===================
notebooks\DianeSoHungry__ShallowMachineLearningCodeItOut__kNN.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\Diishasing__MAXIMUM_REVENUE-Statistical-project__max_revenue.ipynb
['drop_duplicates', 'IQR']
Processing 2100

===================
notebooks\dimitreOliveira__KaggleCareerCon2019__[61th iteration] - LSTM New val - Add ft 2.ipynb
['norm_min_max']

===================
notebooks\DindaFeb__Proyek-Capstone-Analis-Data-IBM__Modul 3 Exploratory DataAnalysis.ipynb
['IQR']

===================
notebooks\Dipnil07__Deep-Learning-based-Feature-Extraction-with-sMRI-data-in-Neuroimaging-Genetics-for-Alzheimer-s-Disea__Whole_image_classification_final.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\dishaphalle27__AMAZON-PRODUCT-Review---Sentiment-Analysis-__sa.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\DistrictDataLabs__03-censusables__20150822.ipynb
['drop_duplicates', 'drop_duplicates']

<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\dityas__SensorWeb__ansi_regression-preprocessing.ipynb
['norm_min_max']

===================
notebooks\DivyaBansal__FortyFiveDays__ML.ipynb
['norm_min_max', 'drop_duplicates']

===================
notebooks\Diyorbek-MY__House_Price_prediction__Data_Preparing_For_ML(3).ipynb
['norm_min_max']
Processing 2150

===================
notebooks\doms911__titanic-ml__03_feature_engineering.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
notebooks\Donguk-Kim-kr__BigData-Practice__17_seaborn.ipynb
['drop_duplicates']

===================
notebooks\Dowee2__March-Madness-Predictor__Book.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\dragosandreibobu__fiicode-2026-ai-bank-telemarketing-prediction__improved.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\drlphysics__Real_Estate_ML_Project__sfr_data_optimization_II.ipynb
['IQR', 'norm_log']

<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:42: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.

Processing 2200

===================
notebooks\drshahizan__Python-big-data__bigData.ipynb
['drop_duplicates']

===================
notebooks\DSRoCCO__modelos_seguros_privacidad_TPT__run.ipynb
['drop_duplicates', 'IQR']

===================
notebooks\Dugi000__kaggle__new-0515_0.63387.ipynb
['IQR']

===================
notebooks\dumindagamage__House-Price-Analysis__02_data_trasformation_and_loading.ipynb
['norm_log']

===================
notebooks\durupudiruthvika__Machine-Learning-Lab02__A7.ipynb
['norm_min_max']
Processing 2250

===================
notebooks\e-kirkland__datascience__Client Command Assessment.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\e1548423__Team23_IT5006_Predictive_Policing_AY2526Sem2__Retrain_Inference_Engine_UI.ipynb
['drop_duplicates']

<unknown>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:59: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<unknown>:59: SyntaxWarning: "\#" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\#"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.


===================
notebooks\EarthByte__MPM_Lachlan_Laterite__MPM_Hypogene_Features.ipynb
['norm_min_max']

===================
notebooks\Edch23__examendojo__intento1.ipynb
['norm_min_max']

===================
notebooks\eduardodazac__Modelos-Regresion-Supervisados__Modelo de regresion lineal.ipynb
['IQR']
Processing 2300

===================
notebooks\egecjdemir__how_football_teams_play__create_ball_gain_df_h1.ipynb
['drop_duplicates']

===================
notebooks\ehtisham-sadiq__Final-Year-Project-Material-__Emotions detection from Tweets Data.ipynb
['drop_duplicates']

===================
notebooks\ekovegeance__datascience-nb__3-data-cleaning.ipynb
['IQR', 'IQR', 'IQR']

===================
notebooks\eldesokye__Naive_Bayes_project__NB.ipynb
['bin_equal_frequency_10']

===================
notebooks\electricmechanism__python-machine-learning-projects__Model_prediction_2.ipynb
['drop_duplicates']

===================
notebooks\EliAndrade__CoinGeckoAPIML__ML.ipynb
['norm_min_max']

===================
notebooks\ElizabetDA__VK_practice__VK.ipynb
['norm_log']

===================
notebooks\elizabeththrall__MLforPChem__Cyanine_Dye_Regression_Tutorial_Instructor.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\ellaTTTT__Python__【Week07】Data Science basics (3) DataPreprocessing.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

<unknown>:8: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:3: SyntaxWarning: invalid decimal literal
<unknown>:7: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.

Processing 2350

===================
notebooks\EmincanY__Machine-Learning__MultipleLinearRegression.ipynb
['norm_min_max']

===================
notebooks\Emma03299__Energy-Usage-and-Prediction__ML.ipynb
['norm_min_max']

===================
notebooks\EmreTYucel__Iphone_Price_Prediction__EDA_V1-checkpoint.ipynb
['drop_duplicates', 'IQR', 'IQR', 'drop_duplicates']

===================
notebooks\EnricRovira__TFM_DNN_Recomendator__11_Recommendator.ipynb
['drop_duplicates']

===================
notebooks\enuguru__DataScienceLevelOne__preprocessing.ipynb
['norm_min_max']

===================
notebooks\enumerbs__Project3-Group6__etl_part4.ipynb
['drop_duplicates']

===================
notebooks\ERA-Software__computational-data-analysis__T3_from_missing_to_insights_solutions.ipynb
['IQR', 'IQR']
Processing 2400

===================
notebooks\Erick-Faster__challenge-ml__3-previsao-falhas.ipynb
['drop_duplicates']

===================
notebooks\ericshenggle__PandasVSPyspark__main.ipynb
['drop_duplicates']

===================
notebooks\Ertugrul-Kurubal__Data-Scientist__Word N Gram And Sentence In Each Other ReDe.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\estcr__Machine-Learning-Project__main.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\estherguiuhernandez__UCI_prediction_final_project__data_cleaning_step2.ipynb
['IQR']
Processing 2450

===================
notebooks\Fahmi-IT__CSI4142_A4__A4.ipynb
['drop_duplicates']
Processing 2500

===================
notebooks\faniloo08__ANNPrediction__Prediction.ipynb
['norm_min_max', 'norm_min_max']

<unknown>:12: SyntaxWarning: "\j" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\j"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\L" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\L"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\L" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\L"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\L" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\L"? A raw string is also an option.


===================
notebooks\FatimaMumtaz86__IBM-Data-Analyst-Capstone-Project__Hands-on Lab 9 - Imput Missing Values-v1 (1).ipynb
['drop_duplicates']

===================
notebooks\felipemegale__simuvent-clean__010_current_speed.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 2550

===================
notebooks\Felix-Lin-0907__YB_Data_Analysis__exercise_outlier.ipynb
['norm_log', 'IQR']

===================
notebooks\felpscunha__projects_datascience__portoseguro.ipynb
['zscore']

===================
notebooks\fenago__datawrangling__Activity_1_01_Structure_Quality_Investigations.ipynb
['drop_duplicates']

===================
notebooks\ferdinandputra86__Emotion-Detection-NB-SVM__Sampling.ipynb
['norm_min_max']

===================
notebooks\Ferris-Solutions__goalspotter_public__topic_detection_modeling.ipynb
['drop_duplicates']

===================
notebooks\flashstep11__IoMT-Proto__02_hypertension_model.ipynb
['drop_duplicates']

===================
notebooks\flatiron-school__ds-k-nearest_neighbors-kbo33__knn.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\flatiron-school__ds-k-nearest_neighbors-kvm32__knn.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\flatiron-school__ds-k-nearest_neighbors-kvo32__knn.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\flatiron-school__ds-k-nearest_neighbors-opw32__knn.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\flatiron-school__ds-k-nearest_neighbors__knn.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 2600

<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:52: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.


===================
notebooks\FlavioFRibeiro__hotel_reviews_nlp_analysis__Crete_Reviews.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\FlowBoat123__ML_BTL__xgboost-v3 (1).ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\fmedrano2019__TraderJoes__AE.ipynb
['norm_min_max']

===================
notebooks\fosetorico__pH_level_forecasting__2. Model_Training.ipynb
['IQR']

===================
notebooks\FranciscoAlves124__AC__coty.ipynb
['drop_duplicates']

===================
notebooks\FrankyFresh17__first-working-ML-model__train_model_lightgbm.ipynb
['drop_duplicates']
Processing 2650

===================
notebooks\fyakkan__Predicting-Heart-Disease__04_shallow_nn.ipynb
['drop_duplicates']

===================
notebooks\g-troiani__AI-Fall-25-Project-Phase-5__project5_final_integrated_system_over_9000 (1).ipynb
['drop_duplicates']

===================
notebooks\g0900971__Analytics_Capstone_Projects__Data_Manipulation_with_Pandas.ipynb
['drop_duplicates']

===================
notebooks\GabMeng__Estimating-permanent-price-impact__RL.ipynb
['norm_log']

===================
notebooks\gabriel1200__shot_data__series_gamelevel-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\GaggeraVinodh__datascience__preprocessing.ipynb
['norm_min_max']

===================
notebooks\Gamana__DataScience__pandas_guide.ipynb
['drop_duplicates']

===================
notebooks\gamboaalejandro__ML-Vocational-Interest-Project__preprocess.ipynb
['drop_duplicates']

===================
notebooks\ganeshbmc__MLP_project__select_features.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.


===================
notebooks\garvit-agarwal-purdue__VERA-Dataset__rw_family_classification_pipeline.ipynb
['drop_duplicates']

===================
notebooks\gaurav-maheshwari-sada__ExoplanetExploration__svm.ipynb
['norm_min_max']

===================
notebooks\gaurshivangi__CS550_Assignment1__q4.ipynb
['norm_min_max']
Processing 2700

===================
notebooks\Gayatri8-sys__Machine-Learning__NN.ipynb
['IQR']

===================
notebooks\geopan2000__Exploring-Mental-Health-Data__Mental-Health-Data-3.ipynb
['IQR', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\George9822__CICIDS_2017and2018_IntrusionDetectionSystem__PartII_dask.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\ghn9zh__ds3001-HW5__assignment_knn.ipynb
['norm_min_max']

===================
notebooks\Ghostvulture__TradeMaster2026__LBM.ipynb
['bin_equal_frequency_5']

===================
notebooks\GianMan89__finding_donors__finding_donors.ipynb
['norm_min_max']

===================
notebooks\Gibranfaktiananwar__Loan-Eligibility-Prediction-using-Naive-Bayes_Mini-Python-Projects-For-Data-Science-__Notebook_Loan Eligibility Prediction.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\GihanAyesh__CS4622-ML-Challenge__ML_Challenge.ipynb
['norm_min_max']
Processing 2750

===================
notebooks\Giri1426__3101-AIHT-Proj_229796_Team_2-Public-Transportation-Efficiency-Analysis__Code with Explanation.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\GIRIJAOK__Data-Analytics-Project__M3ExploratoryDataAnalysis-lab.ipynb
['IQR']

===================
notebooks\GiuseppeZappia__Quantum_Classification_on_Wine_dataset__CLASSIFICATORI_NON_QUANTISTICI.ipynb
['norm_min_max']

===================
notebooks\Gj00110__testrepo__M2DataWrangling-lab (1).ipynb
['drop_duplicates']

===================
notebooks\gkrishna247__AgriCastV01__data_cleansing.ipynb
['drop_duplicates']

===================
notebooks\gmshroff__aicourse__learning1.ipynb
['norm_min_max']

===================
notebooks\GnanaDeepika29__ddos-detection-mitigation-system__model_training.ipynb
['norm_log', 'isolationForest']

===================
notebooks\Gnanas458__Tourism_experience_analytics__classification.ipynb
['IQR', 'norm_min_max']
Processing 2800

===================
notebooks\GoloMarcos__MVAEs-FakeNews__Multimodal_LIME_OCL.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\GOMEZBORIS6__deforestation-streamlite-app__projetdeforestationMAG3.ipynb
['drop_duplicates']

===================
notebooks\gonaloppcc__projetoDAA__random_forest_tensor_flow.ipynb
['drop_duplicates']

===================
notebooks\google-research__google-research__acs_fit_models.ipynb
['drop_duplicates']

===================
notebooks\GordeevKV__fuzzy-giggle__ML_2_HW_Гордеев К.В-checkpoint.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\GoshaSerbin__ML-notebooks__HW1.ipynb
['norm_min_max']

===================
notebooks\gowprial__ML-Assignment__ML_Assignment_9_Feature_Engineering.ipynb
['norm_log', 'norm_log', 'norm_min_max']

===================
notebooks\gplinkage__Core-Python__Pandas_data_Cleaning.ipynb
['drop_duplicates']

===================
notebooks\Gracezu__Neural-Network-Deep-Learning-Training-and-Evaluation__CNN_LSTM_STOCKPRICEPR.ipynb
['norm_min_max']

<unknown>:8: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\graphistry__pygraphistry__splunk_demo_public.ipynb
['drop_duplicates']
Processing 2850

===================
notebooks\greyluo__News-Recommender__FM.ipynb
['norm_min_max']

===================
notebooks\gsu-ds__campus-burglary-risk-prediction__01_wrangler.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\GuilhermeGML__Analise-Valorant-ESport__3 - Aplicação de ML-China.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\GunikaSharma__Zomato-Discount-Cohort-CLV-Analysis-Food-Delivery-Platform__01_data_cleaning.ipynb
['drop_duplicates']

===================
notebooks\gustavopierre__flight_delay_project__Flight_Delay.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\GustavoValenca__DataScience-Placas-de-Video__regressao.ipynb
['norm_min_max', 'norm_min_max']
Processing 2900

<unknown>:9: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
notebooks\hafizharis246__thyroid_disease_recurrence_classifier__thyroid-dataset-classification-model.ipynb
['drop_duplicates']

===================
notebooks\hamaadshah__gan_tensorflow__gan.ipynb
['norm_min_max']

===================
notebooks\hamnasz__Class-Assignments__SALARY REGRESSION.ipynb
['IQR']

===================
notebooks\hamza-aziz-ai__heart-disease-risk-prediction__Heart_Disease_MLOps_Assignment_Report.ipynb
['drop_duplicates']

===================
notebooks\HamzaaAkmal__Data-Analysis-Project-First-Semester__analysis.ipynb
['drop_duplicates']

===================
notebooks\HANEENAVP__HANEENA-V-P__Haneena_CLASSIFICATION.ipynb
['drop_duplicates']

===================
notebooks\Hansaka2001__IPL-Score-Prediction-using-Deep-Learning__code.ipynb
['norm_min_max']

===================
notebooks\HanyMedhat10__YouTube-comments-Segment-analysis__app.ipynb
['drop_duplicates']
Processing 2950

===================
notebooks\Haosam__HPthingy__ML.ipynb
['norm_min_max']

===================
notebooks\haoylle__19_CourseRegistration-Prediction__EDA.ipynb
['drop_duplicates']

===================
notebooks\Happily-Coding__TimeSeriesForecasterStoreSales__exploration2.ipynb
['drop_duplicates']

===================
notebooks\Harddik02__ML_mini_project__Mini_Project.ipynb
['drop_duplicates']

===================
notebooks\Hardfive__ProBettor__modelling.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\harishhirthi__AAAI-2024__Zero-shot-hourly_Bareilly.ipynb
['norm_min_max']

===================
notebooks\Harmanp456__cinemind__movie.ipynb
['drop_duplicates']

<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
notebooks\harsh-kakadiya1__Autosweep__1.ipynb
['drop_duplicates', 'IQR', 'norm_min_max']

===================
notebooks\Harsha1811819__retailpulse_zidio__07_checkpoint.ipynb
['drop_duplicates']

===================
notebooks\HarshCasper__NeoAlgo__Income_Classification.ipynb
['zscore']
Processing 3000

===================
notebooks\haufjan__TimeGAN-PyTorch__timegan.ipynb
['norm_min_max']

===================
notebooks\Hebrink__capstone-ui-newton__flask-ui-skeleton.ipynb
['drop_duplicates']

===================
notebooks\heena-parveen23__Content-Based-Book-Recommender__eda.ipynb
['drop_duplicates']

===================
notebooks\hellonish__AML-System__AML_Baseline_and_MultiGNN_Replication.ipynb
['norm_log']
Processing 3050

===================
notebooks\hemant3580__Final_Year_Project__LSTM_power_analysis.ipynb
['norm_min_max']

<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:99: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\X" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\X"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\X" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\X"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:38: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\hgmhd7__LEGACY-Wine-O-Vation-project__UPDATED_final_model_training.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\hieundx__ML-pytorch-sklearn__chapter 4 - data preprocessing.ipynb
['norm_min_max']

===================
notebooks\HiPatil__Machine-Deep-Learning__finding_donors.ipynb
['norm_min_max']
Processing 3100

===================
notebooks\hiteshmr-7637103__lendingClub__lending_club_eda.ipynb
['zscore']

===================
notebooks\hjooh__Mixed-Reality-Cybersecurity__trying_pca.ipynb
['norm_min_max']

===================
notebooks\hluu01__DSC180A_B09_2__FullModelPipeline.ipynb
['drop_duplicates']

===================
notebooks\horkydorky__Data-Analysis__salesanalysis-checkpoint.ipynb
['drop_duplicates']
Processing 3150

===================
notebooks\howtodie123__howtodie123__ML.ipynb
['IQR']

===================
notebooks\HOXOMInc__feature-engineering-book__9.ipynb
['drop_duplicates']

<unknown>:9: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
notebooks\htetaunglynn94__coursera__holab_1_regression_tts.ipynb
['norm_min_max']

===================
notebooks\hulseyvincentr__WCC_MachineLearning__04-Keras-Project-Exercise-Solutions.ipynb
['norm_min_max']

===================
notebooks\hurhu__recommendation-pytorch__FFM.ipynb
['norm_min_max']

===================
notebooks\hurhu__recommendation-pytorch__NFM.ipynb
['norm_min_max']
Processing 3200

===================
notebooks\Hussain0327__risk_modeling__03_feature_engineering.ipynb
['norm_log', 'norm_log']

===================
notebooks\Hussainaquib__Deep-Learning__rnn-gated-recurrent-unit.ipynb
['norm_min_max']

===================
notebooks\HuynhDucPhu2502__Data-Analysis-Learning-Projects__22653551_HuynhDucPhu_TH_Tuan02.ipynb
['drop_duplicates']

===================
notebooks\huzaiffff__Network-Intrusion-Detection-System__5_3 LightgbmWithADASYN.ipynb
['drop_duplicates']

===================
notebooks\hwittlieff__DSC550__Titanic Model Building Project Sample.ipynb
['norm_log', 'norm_log']

===================
notebooks\HyeongseopSo0914__my-project__dacon-enterprise.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\hyun-min-park__phishing-email-detection__mlp.ipynb
['drop_duplicates']

===================
notebooks\Hyunkio__LG_Aimers_LightGBM__main.ipynb
['norm_log', 'norm_log']

===================
notebooks\Hyunzuny__classes__04_데이터_전처리.ipynb
['norm_min_max', 'norm_min_max']

<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\y" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\y"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:125: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:129: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


===================
notebooks\ianfrederickk__clickbait-detection__clickbait-bert.ipynb
['drop_duplicates']
Processing 3250

===================
notebooks\icta-tecaji__python-machine-learning-public__05_Example_Pipelines_usage_CLEAN.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\Idank96__Chronic_Kidney_Disease_models__maman22.ipynb
['norm_min_max']

===================
notebooks\iftikhar200__ml-tip-prediction-scalers__ai.ipynb
['norm_min_max']

===================
notebooks\ijessicachen__introdatascience__dataprep.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\ikramul2012__Credit-card-fraud-detection__v2.ipynb
['norm_min_max']
Processing 3300

===================
notebooks\Imcyj123__hw2-M11223041__KNN-checkpoint.ipynb
['norm_min_max']

===================
notebooks\imengu__tf26__a.ipynb
['drop_duplicates', 'norm_log']

<unknown>:49: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.


===================
notebooks\inEXASCALE__llm-abba__peft_lora_embedding_semantic_similarity_inference.ipynb
['drop_duplicates']

===================
notebooks\informrohit1__DataScience-Minor__Ds_work.ipynb
['drop_duplicates']

===================
notebooks\Intelligent-molecular-systems__LLM_finetuning_for_biochemistry__xgboost_baseline.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 3350

===================
notebooks\invincible1786__ICU_datathon__pre.ipynb
['norm_log', 'norm_log']

===================
notebooks\iqbalzayn01__data_quality_with_python__Data_Quality.ipynb
['IQR', 'IQR', 'drop_duplicates']

===================
notebooks\irfanhasib0__Machine-Learning__ANN_From_Scratch_modular_class-exp-mod-checkpoint.ipynb
['norm_min_max']

===================
notebooks\Ironhack-Data-Madrid-Julio-2023__apuntes_clase__3.1 - Data Cleaning.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\Isabella373__hw5-330__hw5.ipynb
['norm_log', 'norm_log']

===================
notebooks\Ishasingh2025__NYCCongestion__Six Datasets cleansing.ipynb
['drop_duplicates', 'IQR', 'drop_duplicates', 'IQR', 'drop_duplicates', 'IQR', 'IQR', 'drop_duplicates', 'IQR', 'drop_duplicates', 'IQR']

===================
notebooks\ishitabansal21__Banking-Churn__DT_Modelling.ipynb
['norm_min_max']

===================
notebooks\Iskken__Echoes-of-Longevity-Research__data_collection.ipynb
['drop_duplicates']
Processing 3400

===================
notebooks\IsmaDDamara__Optimizing-Customer-Strategy-with-RFM-Analysis__RFM_Segmentation.ipynb
['zscore']

<unknown>:25: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:79: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:129: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.


===================
notebooks\ItsDarker__Cyber-Attack-Classification-Using-Supervised-ML__M1.ipynb
['drop_duplicates']

===================
notebooks\iupui-soic__wound-forecast__1_Merging_Big_WE.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\ivanarielcaceres__timeseries-lstm-keras__timeseries-prediction.ipynb
['norm_min_max']

===================
notebooks\iwangmoeslem__CNN-Malware-Detection__CNN.ipynb
['norm_min_max']
Processing 3450

===================
notebooks\jacky0405__100Days-ML-Marathon__Day_022_HW.ipynb
['norm_min_max']

===================
notebooks\JaGuzmanT__Logistic-Regression-to-predict-the-risk-of-death-in-Covid-19-Patients__Feature selection and model.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\Jainivas03__AI-Enhanced-Personalized-Career-Guidance-System__Main.ipynb
['drop_duplicates']

===================
notebooks\jalvord1__nfl_sentiment__final loop.ipynb
['drop_duplicates']
Processing 3500

<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:54: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:86: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:91: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:95: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:97: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:110: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:126: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\=" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\="? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:109: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\JamesRestall__machine_learning__Model_Churn-checkpoint.ipynb
['norm_min_max']

===================
notebooks\Jana1805__ML-Assignment__linear_regression.ipynb
['drop_duplicates']

===================
notebooks\janice880624__3rd-ML100Days__Day_031_HW.ipynb
['norm_min_max']

===================
notebooks\jansiddiqui__Women-Safety__Random_Forest_Women_Safety.ipynb
['bin_equal_frequency_10', 'bin_equal_frequency_10']

===================
notebooks\jasonliu1990__brainchild__txbranch_debit_credit_ratio_0.082a.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\jathinreddy__jakevdp_PythonDataScienceHandbook__03.09-Pivot-Tables.ipynb
['bin_equal_frequency_2']

===================
notebooks\javierreansyah__ML-Battle-Royale__mlnew.ipynb
['drop_duplicates']
Processing 3550

===================
notebooks\Jayk5__ML_mini_project__Mini_Project.ipynb
['drop_duplicates']

===================
notebooks\jcmartinezs__llm_engineering__end_of_week_assesment.ipynb
['drop_duplicates', 'IQR', 'drop_duplicates']

<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id 'fd1ddc61' detected. Corrected to '5f67dc0c'.
  validate(nb)
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id 'fd1ddc61' detected. Corrected to '016b4d01'.
  validate(nb)
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id 'fd1ddc61' detected. Corrected to 'fdd0b6da'.
  validate(nb)
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id 'fd1ddc61' detected. Corrected to '433144af'.
  validate(nb)
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id 'fd1ddc61' detected. Corrected to '016c4746'.
  validate(nb)
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id 'fd1ddc61' detected. Corrected to '28e7d628'.
  validate(nb)
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.

Processing 3600

===================
notebooks\jengler__nbks-605mb__regularized-linear-models.ipynb
['norm_log', 'norm_log']

===================
notebooks\Jenil7828__Sem-VII__Practical2a.ipynb
['drop_duplicates']

===================
notebooks\jermynyeo__fake-news-detection__Compiled.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\Jeromexsu__business_data_analysis__analysis.ipynb
['drop_duplicates']

===================
notebooks\jeron-williams__Easy_Visa_Classification_Hypertuning_Bagging_Boosting__Copy_of_EasyVisa_Full_Code_Notebook.ipynb
['IQR']

===================
notebooks\Jerry-britto__Job-Analysis__Job_Analysis.ipynb
['drop_duplicates']

===================
notebooks\jessyca-ferreira__mth-ids-implementation__improvement_classe2_holdout.ipynb
['norm_min_max']

===================
notebooks\Jh-wanderer__class__04_preprocessing.ipynb
['norm_min_max', 'norm_min_max']
Processing 3650

===================
notebooks\jharrisong830__cs513-final-project__main.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\JHC90__Basic-DataScience-Skills__01-Deep-Nets-mit-TF-Abstraktionen-checkpoint.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\ji3g4m6zo6__100Day-ML-Marathon__Day_028_HW.ipynb
['norm_min_max']

<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.


===================
notebooks\jianjhihlai__2nd-ML100Days__Day_016_HW.ipynb
['norm_min_max']

===================
notebooks\jiaolong1988__Machine_Learning__finding_donors-checkpoint.ipynb
['norm_min_max']
Processing 3700

===================
notebooks\jingyuanchan__Real-time-video-anomaly-detection__Optical_Flow_Ang.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\jkarpen__Springboard_Projects__json_exercise_jkarpen.ipynb
['drop_duplicates']

===================
notebooks\jm55__Evaluation-and-Comparison-of-Boosted-ML-Models-in-Behavior-Based-Malware-Detection__[TEST] Dataset.ipynb
['drop_duplicates']

===================
notebooks\joaoaleite__mmo__test.ipynb
['norm_min_max']
Processing 3750

===================
notebooks\johirul398__Machine-Learning-for-Heart-Attack-Prediction__machine-learning-for-heart-attack-prediction.ipynb
['bin_equal_width_5', 'bin_equal_width_5', 'bin_equal_width_5', 'bin_equal_width_5', 'zscore']

===================
notebooks\johnnyyang722__fraud_detection_app__DTSC 691 Project Notebook-Final.ipynb
['isolationForest']

===================
notebooks\johnpyp__stonks__AI.ipynb
['norm_min_max']

===================
notebooks\jonrtaylor__twitch__FN_with_OLS.ipynb
['norm_min_max']

===================
notebooks\JoshuaChoa__IYKRA__Practice_Case_2.ipynb
['IQR', 'IQR', 'IQR']
Processing 3800

===================
notebooks\Joyfreaky__Ashrae-Energy-Prediction-III-21-22__RNN_Dense_Final.ipynb
['drop_duplicates', 'norm_log', 'norm_log', 'drop_duplicates', 'norm_log']

===================
notebooks\Jp1823__Projeto_DAA__Data_Analysis.ipynb
['IQR']

===================
notebooks\jpioug__predictionio-template-kaggle-house-prices__eda.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
notebooks\jpmbrito123__DAA__SupportVectorMachine.ipynb
['IQR']

<unknown>:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.


===================
notebooks\juanbracho__UT_Module23__lstm_model_training.ipynb
['norm_min_max']
Processing 3850

===================
notebooks\Juliansan__LLM_Engineering__end_of_week_assesment.ipynb
['drop_duplicates', 'IQR', 'drop_duplicates']

===================
notebooks\junxi-haoyi__PythonDataScience__03.09-Pivot-Tables.ipynb
['bin_equal_frequency_2']

===================
notebooks\jupotter37__JupOtter__HOXOMInc_feature-engineering-book_9.ipynb
['drop_duplicates']

===================
notebooks\Jvelasquez980__Aprendizaje-Automatico-MQ__QML.ipynb
['norm_min_max']

===================
notebooks\jxplanet0__ekt_sur__sur.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\jyothimanoj12__Afame-Technologies__HR Data Analysis.ipynb
['drop_duplicates']

===================
notebooks\k0nig1__llm_engineering__end_of_week_assesment.ipynb
['drop_duplicates', 'IQR', 'drop_duplicates']

===================
notebooks\ka-sa-004__SUMMIFY__Sentiment_analysis.ipynb
['drop_duplicates']

===================
notebooks\kaan4dev__microsoftCourseML__df.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 3900

===================
notebooks\kacp-i__BCU-Work__Pre_Processing.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\kahalandkh__ZSL_Master__2_initial_data_cleaning.ipynb
['drop_duplicates', 'drop_duplicates']

<unknown>:26: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:28: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:38: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:42: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\kaiquevalentim__fair-launch-analytics__v1v2_download.ipynb
['drop_duplicates']

===================
notebooks\kalebsampaco__libro-machine-learning__03.09-Pivot-Tables.ipynb
['bin_equal_frequency_2']

===================
notebooks\kanakrajarora__Insure-Pro__code copy.ipynb
['drop_duplicates']

===================
notebooks\Kancherla-Amulya__AIML-2303A51242__adm_lab_06.ipynb
['drop_duplicates']

===================
notebooks\KanikaGupta-22978__Japan_Internship__2 - Forecasting.ipynb
['norm_min_max']
Processing 3950

===================
notebooks\karisamarykopecek__545ML__inclass_04_21_Kopecek.ipynb
['norm_log', 'norm_log']

===================
notebooks\kartikeVr__Python-basics__Iris.ipynb
['drop_duplicates']

===================
notebooks\katherinezhao123__DIMACS_REU__peft_lora_embedding_semantic_similarity_inference.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.

Processing 4000

===================
notebooks\Kay-Sap5__Yangon_House_Price__Steps_for_cleaning.ipynb
['drop_duplicates', 'IQR', 'IQR']

===================
notebooks\Kaybhee__Internship_hamoyeHQ__sec2_tag.ipynb
['norm_min_max']

===================
notebooks\KayChansiri__Demo_GBT__GBT.ipynb
['IQR']

===================
notebooks\Kemalavsever__Alzheimer-detection__detection.ipynb
['IQR']

===================
notebooks\kershrita__Heart-Attack-Prediction-Model__3 - Linear SVM.ipynb
['drop_duplicates']

===================
notebooks\keshav-rathor__Pytorch-tutorial__L4_LSTM.ipynb
['norm_min_max']
Processing 4050

===================
notebooks\kevin-291__startup-health-scoring-model__neural_net_tensorflow.ipynb
['norm_min_max']

===================
notebooks\KFMBB__T5_Weekly_Tasks__Weekly_Project_Khalid_AlBakr.ipynb
['IQR']

===================
notebooks\kgpark88__visionai__DNN.ipynb
['norm_min_max']

===================
notebooks\khalidumar29__video-game-sales-prediction__main.ipynb
['IQR']

===================
notebooks\KhanhVHM17__AIO-Exercise__Sentiment_Analysis.ipynb
['drop_duplicates']

===================
notebooks\kibindy__DMW_Lab2__Scratch_Francis_v3.ipynb
['drop_duplicates']

===================
notebooks\kijen28__P4DS_22G1__exploring_data.ipynb
['IQR']
Processing 4100

===================
notebooks\kiranteja2005__IIT-Ropar-Minor-in-AI-for-Content-Recommendation-System__eda.ipynb
['drop_duplicates']

===================
notebooks\Kirisannn__IBM-Data-Analyst-Capstone-Project__(Lab 7) Removing Duplicates-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\KishorKumarParoi__AI-Engineering__end_of_week_assesment.ipynb
['drop_duplicates', 'IQR', 'drop_duplicates']

===================
notebooks\kiwimaya__XGBoost__SF.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\kmsanjar47__Ml-Session-7__Batch_18_Class_11_Handling_Imbalanced_Class_&_Cross_Validation.ipynb
['drop_duplicates']

<unknown>:12: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\konderal333__HGT-2022-EmDomArDon__bert2bert.ipynb
['drop_duplicates']

===================
notebooks\konfuckyus__anime-recommender__dataset1.ipynb
['drop_duplicates', 'norm_min_max']
Processing 4150

===================
notebooks\kov225__Projects__03_user_segmentation.ipynb
['norm_min_max']

===================
notebooks\krotkikhmaxim__rsm_hackathon_2026__Untitled3-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\krunalsalunkhe23__Afame-Technologies__HR_Data_Analysis.ipynb
['drop_duplicates']
Processing 4200

===================
notebooks\kruth-s__Data-Engg-Lab__ETL.ipynb
['drop_duplicates']

<unknown>:33: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:45: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:48: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:53: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
notebooks\KryakPingvi__Exam__K.ipynb
['drop_duplicates', 'IQR']

===================
notebooks\krzysiekniburski__Network-Traffic-Classification__ANN.ipynb
['norm_min_max']

===================
notebooks\kshilin__Fintech-AD-ML__ML-06-03-Encoder.ipynb
['norm_min_max']

===================
notebooks\kurtsenol__machine-learning__customer-churn-prediction.ipynb
['norm_log']

===================
notebooks\kutikova2016__see-me__R.ipynb
['drop_duplicates']

===================
notebooks\kvv1618__neuralnetworks__model.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\kwierman__CustomerChurn__05_predictions.ipynb
['norm_log', 'norm_log']

===================
notebooks\KwintaJ__MachineLearningClasses__Jan_Kwinta_exercises_4.ipynb
['norm_min_max']
Processing 4250

===================
notebooks\Lacenedihia__Data-Science-Challenges-__LoanDefaultPrediction.ipynb
['norm_log', 'norm_log']

===================
notebooks\lairifangtang__Typhoon-Forecast-based-on-LSTM__台风预测.ipynb
['drop_duplicates']

===================
notebooks\Lake-Commander__premium-prediction-model__a.ipynb
['norm_log', 'norm_min_max']

===================
notebooks\Laksh-Mendpara__Football_Match_Outcome_Prediction__Supervised Learning Models (1).ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\Lalit767__Expedia_Case_Study__datacleansing.ipynb
['drop_duplicates']

===================
notebooks\LandinGabriel13__Introducci-n-a-Pandas__Introducción a Pandas.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\lara-62__CSE472_MachineLearning__1905062.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\Lau-Tisca__FlyRank_ML_1__w05_model.ipynb
['drop_duplicates', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\LauraAva__LauraAva__DATA_CLEANSING.ipynb
['IQR']
Processing 4300

===================
notebooks\LavishBhatia-Projects__ML-Project__Model.ipynb
['drop_duplicates']

===================
notebooks\LCAlloyance__ml_Alloyance__main-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\lcroash__spam_filter__Spam_Classifier - body extract ref-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\leanaco__udacity-courses__finding_donors-checkpoint.ipynb
['norm_min_max']

===================
notebooks\learn-co-students__hbs-ds-060120__pipelines_roc_auc-enkeboll.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\LeeDohoun__Learn__RNN.ipynb
['norm_min_max']
Processing 4350

===================
notebooks\LenaNevel__CAPSTONE__03_cleaning_data.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\Leomutz__NIDS1__2025_06_March_UNSW_NB15_dataset_.ipynb
['norm_min_max']

===================
notebooks\LEON-JU__CS182-Final-project__SVM.ipynb
['bin_equal_frequency_5']

===================
notebooks\lexoz-bedra__rank_model_vk__vk.ipynb
['drop_duplicates']

===================
notebooks\lg960214__2022-lguplus-AI-Ground__[EDA]Get_Pairs.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 4400

===================
notebooks\LilianYou__Maze_Learning_Neural_Analysis__mvpa_pipeline_test-checkpoint.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\liulin7576__The-structure-of-data-and-Algorithm__house_price_kernel.ipynb
['norm_log']

<unknown>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


===================
notebooks\LizJnuzyt__zyt__trainEDA_v0508.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\ljm524__esaa24-1__esaa_hw0322.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 4450

===================
notebooks\llh139__Data-Analytics-Portfolio__Vacation Preference Prediction Classification Model.ipynb
['norm_min_max']

===================
notebooks\lourenco500__Data-Mining-25-26__lab02_data_exploration-checkpoint.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\LRTGithub2023__DTSA5511IntroToDeepLearning__week3KaggleMPRev1.ipynb
['drop_duplicates']
Processing 4500

<unknown>:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:51: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.


===================
notebooks\lucasnunesilveira__estudo__Semana3-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\LuckyBoy587__Statistical-Methods__05_Data_Preprocessing.ipynb
['zscore']

===================
notebooks\LugoBlogger__SI-201-542-forecasting-technique__week-13.ipynb
['norm_min_max']

===================
notebooks\luisgh87__Project_Madrid_Pedalea__data_cleansing.ipynb
['drop_duplicates']

===================
notebooks\luisjbranco__Pieran_Data_Learning__RNN_multivariate_timeseries.ipynb
['norm_min_max']

===================
notebooks\luisppereira18__copilot-flight-hackathon__manage-flight-data.ipynb
['drop_duplicates']

===================
notebooks\LukasOttenhof__JupOtter__AliciaFrame_Public-Python-Notebooks_LinkPrediction.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 4550

===================
notebooks\lukegbenson__parking_lot_analysis__lot_feature_analysis.ipynb
['norm_log']

===================
notebooks\lunamoonsun__QSPR-STUDY__kNN for (Q)SPR.ipynb
['drop_duplicates']

<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
notebooks\Luxflamy__STAT-628-Module-3__dep_delay_nn.ipynb
['norm_log']

===================
notebooks\lyang24__KaggleComp__WLF.ipynb
['norm_log']

===================
notebooks\M00N7682__Youtube-View-Predict__youtube_pred(category) (1).ipynb
['norm_log']

===================
notebooks\m0hitchaudhary__Music-Recommendation-System__p.ipynb
['drop_duplicates']

===================
notebooks\m1guelperez__jupylab_cli__royisland_price-this-house.ipynb
['norm_min_max']

===================
notebooks\m9tadeo__real-estate-price-prediction__data_analysis.ipynb
['drop_duplicates']
Processing 4600

===================
notebooks\maaz0511__health-prediction-project__thyroid.ipynb
['drop_duplicates']

===================
notebooks\maciad__movie-genre-prediction__create_dataset.ipynb
['drop_duplicates']

===================
notebooks\MaCoZu__dsr__custom_encoder.ipynb
['norm_min_max']

===================
notebooks\Mageshwaran18__Multi_Modal_Brain_Tumor_Segmentation_2023__Data_Processing.ipynb
['norm_min_max']

===================
notebooks\Maham-j__Data-Mining-and-Machine-Learning__Dataset Preprocessing.ipynb
['norm_min_max']

===================
notebooks\mail2mhossain__practical_data_science__9_Otto_Group_Product_Classification_Scaling_Transforming_Model_Based_Feature.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 4650

===================
notebooks\mambo06__TCL__XDG.ipynb
['norm_min_max']

===================
notebooks\ManchineLearningENN__Hw1_ML__HW.ipynb
['norm_min_max']

===================
notebooks\mangalcodes__ML_learning__1.ipynb
['norm_min_max']

===================
notebooks\manisaiprasad__Communicate-Data-Findings__exploration.ipynb
['IQR']

===================
notebooks\Manogna-595__Gemstones-Price-Predictor__gemstones.ipynb
['drop_duplicates', 'IQR', 'IQR', 'IQR', 'IQR', 'IQR', 'IQR', 'IQR']

===================
notebooks\manojdon777__HPCAP_ALL__S03c_ santender_xgb_pt3_wip.ipynb
['norm_log']

===================
notebooks\ManonYa09__Statistics_with_Python_G7__02. Working with normal Distribution v2 2.ipynb
['zscore', 'zscore', 'IQR']
Processing 4700

===================
notebooks\MANTHAN137__Machine-Learning-Lab__4.ipynb
['norm_min_max']

===================
notebooks\manwestc__Contingences-MexicoDF__SuperTML.ipynb
['norm_min_max']

===================
notebooks\marckleyman-ucb__Capstone_Final_Project__Update Output copy.ipynb
['drop_duplicates']

===================
notebooks\marie202__ML_jupyter_reader__3_Feature_scales.ipynb
['norm_log']

===================
notebooks\Marklieflat__Course_codes_grad__Trial2.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 4750

===================
notebooks\marlene-alcobendas__Project6_MachineLearning__ML.ipynb
['drop_duplicates']

===================
notebooks\marouaneguemimi__iav_perso__MLP.ipynb
['norm_log', 'norm_min_max']

<unknown>:30: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
notebooks\martinaainembabazi__GROUP-D-DATA-SCIENCE-PROJECT-ON-CUSTOMER-SEGMENTATION__customer_lifecycle.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_width_5', 'bin_equal_width_5']

===================
notebooks\mate7lord__Analisis-Retensi-Pengguna-Mengapa-Loyalitas-Pelanggan-Penting-dalam-E-Commerce__User_Segmentation.ipynb
['zscore']

===================
notebooks\MateoGomezTamayo__Data-Science-knowledge-base__etl_quiz_complete.ipynb
['drop_duplicates']

===================
notebooks\MathiasMoelgaard__SongRetrieval__wasabi_scrape.ipynb
['drop_duplicates']
Processing 4800

===================
notebooks\mattslyons__JLPS_capstone_project__2nd_stage_cv_func.ipynb
['norm_log']

<unknown>:1: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.


===================
notebooks\maviator__recommendation_system__RS.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\MaxKwen2__LSTM-Code__UNVR.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\mayuresh0711__Data-Science-Portfolio__Assignment_09_Data_Preprocessing.ipynb
['norm_min_max', 'norm_log']

===================
notebooks\mbaezpy__hsbi-nlp-2025__P01_Pandas.ipynb
['drop_duplicates']
Processing 4850

===================
notebooks\mbugyis__Fraud_Detection_Project__fraud_project1.ipynb
['drop_duplicates']

===================
notebooks\mbura98__Lawrence_Mbura17__ad.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.


===================
notebooks\MCHERONO137__assignmentrepo__M2DataWrangling-lab.ipynb
['drop_duplicates']

===================
notebooks\MDS7202__MDS7202__11_Feature_Engineering_Part_I.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\MechaKaradi__Spatial-Crime-Prediction__Responsible_data_analytics_kb.ipynb
['drop_duplicates']

===================
notebooks\MelisSezer__air_quality_prediction__au.ipynb
['norm_log']
Processing 4900

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\mhmmdziyadd14__BizSight__NB.ipynb
['norm_min_max']

===================
notebooks\MHoffmannAC__nfl_project__classification.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\michael-ngx__deep-learning__Data_Imputation.ipynb
['norm_min_max']

===================
notebooks\Michelle-Watson__aten_ai_classifier_ner__Aten_Classifier_v2_1.ipynb
['drop_duplicates']

===================
notebooks\MichoelSnow__boardgame-library-recommender__get_ratings.ipynb
['drop_duplicates']
Processing 4950

<unknown>:10: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.


===================
notebooks\mikayla-bryant__insurance-churn-analysis__Insurance Customer Churn Analysis-checkpoint.ipynb
['zscore', 'zscore']

===================
notebooks\mikerabs__StuffLocation__Stuff_File.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\MimoHasPurpose__100DaysOfMachineLearning__titanic-using-pipeline-checkpoint.ipynb
['norm_min_max']
Processing 5000

===================
notebooks\minhnion__spotify_trends__Explore_data copy.ipynb
['drop_duplicates']

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
notebooks\MinkiGao__TectonicDiscrimination__code.ipynb
['norm_min_max']

===================
notebooks\mirioeggmann__ost__choropleth_map.ipynb
['drop_duplicates']

===================
notebooks\misbahsy__APMonitor-do__DeepLearning.ipynb
['norm_min_max']

===================
notebooks\misclassified__super-reviews__03.Super Reviews-checkpoint.ipynb
['drop_duplicates', 'drop_duplicates']

<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\!" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\!"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:45: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:57: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:70: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:71: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\MJ029__Notes__Data-Preprocessing.ipynb
['drop_duplicates']

===================
notebooks\Mjorni__ML_Trading__nm.ipynb
['norm_min_max']
Processing 5050

===================
notebooks\mlepennec-ensae__formation_cepe__correction_sep25.ipynb
['norm_min_max']

===================
notebooks\mmdrezazarei__Supervised_Learning_Projects__adaBoostRegressor.ipynb
['IQR']

===================
notebooks\mnrclab__Modul3_Data_Cleaning_1__03 DATA CLEANING & PREP - Handling Outlier.ipynb
['zscore']

===================
notebooks\MobeenQaisrani__Academic-Projects__C3_W2_RecSysNN_Assignment.ipynb
['norm_min_max']

===================
notebooks\mohamadouhayatouabbassi-glitch__Deploiement-Modele-ML-Gradio-Prediction-du-CA__Projet_deploiement_modele_ML_Gradio_Mohamadou_Hayatou_Abbassi (2).ipynb
['IQR']

===================
notebooks\MohamedAliKhedr__automate__M2DataWrangling_lab.ipynb
['drop_duplicates']
Processing 5100

===================
notebooks\Mohammadhariswani__Network-Traffic-Classification-DISSERTATION__TomekLinks Random_forest.ipynb
['norm_min_max']

===================
notebooks\MOHAN-DATTA-24__AI-ENABLED-CAR-PARKING-USING-OPENCV__SB_Assignment_2.ipynb
['IQR', 'IQR', 'norm_min_max']

===================
notebooks\Mohan-this-side__Classify.ai__classification_project_test_session_123_20251014_012041.ipynb
['IQR', 'zscore']

<unknown>:39: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
notebooks\Mohitderek__CelebalAssignments__CelebalAssignment1.ipynb
['drop_duplicates']

===================
notebooks\mojc__titanic__ESP.ipynb
['bin_equal_width_5']

===================
notebooks\moky1477__AgroSense__Crop_Recommendation_Model (1).ipynb
['norm_min_max']

===================
notebooks\monishar2006__apexplanet-data-analytics__EDA_Task1.ipynb
['drop_duplicates', 'IQR', 'IQR']

===================
notebooks\moranenzo__Hickaton-24__processing-X_train-withdrawal.ipynb
['drop_duplicates']
FAILED: notebooks\Moreira-Ruan__machine-learning__revisão-prova.ipynb
Unsupported nbformat version 5
Processing 5150

===================
notebooks\mosalov__Notebook_For_AI_Main__#6 Pyslar.ipynb
['norm_min_max']

===================
notebooks\mosalov__Notebook_For_AI_Main__Галеев task4.ipynb
['norm_min_max']

===================
notebooks\mosalov__Notebook_For_AI_Main__Петров - задание 4.ipynb
['norm_min_max']

===================
notebooks\MosesKKhoza__Crop-Yield-Estimate__mk.ipynb
['IQR', 'norm_log', 'norm_log', 'norm_log']

<unknown>:13: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
notebooks\Mrfrktmrck19__Istanbul_Earthquake__ensemble.ipynb
['drop_duplicates', 'norm_log']
Processing 5200

===================
notebooks\MrunaliTupsoundar__idgaf__14.ipynb
['IQR']

<unknown>:16: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.


===================
notebooks\mskim94__seminar__seminar.ipynb
['norm_min_max', 'drop_duplicates']

<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
notebooks\MuhammadHanzala-13__PkW-Scraper-DataMining__eda.ipynb
['norm_log']

===================
notebooks\muhdadbachmid__Descriptive-Analysis-of-Earthquake-Indonesia__Descriptive_Analysis_of_Earthquake_Indonesia.ipynb
['drop_duplicates', 'IQR']

===================
notebooks\Mukhammadkodir27__Python_Machine_Learning__Collections.ipynb
['drop_duplicates']

===================
notebooks\MukhlisMaulanaA__data-analytics-python__data-cleansing.ipynb
['zscore']
Processing 5250

<unknown>:7: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.


===================
notebooks\myscratchbooks__Python-and-R__Dimensionality Reduction in Python.ipynb
['norm_min_max']

===================
notebooks\mzhyui__wutong__agent copy 8-0.ipynb
['norm_log', 'norm_log']

===================
notebooks\NabilMik__Applied-ML-Project-Credit-Card-Campaigns__AML_MC.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\NadiaHirwa__DataEngineering__Lesson 2.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 5300

===================
notebooks\Nagpal45__Customer-Segmentation__v.ipynb
['norm_min_max']

===================
notebooks\nakul2707__XpertSim__model2_11.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\Namir-Khan__MLL__Supervised_Learning_and_K_Nearest_Neighbors_Exercises-checkpoint.ipynb
['norm_min_max']

===================
notebooks\NandaCj__data-science-class__Optimizers-checkpoint.ipynb
['norm_min_max']

===================
notebooks\NandiniPD__Data_Engineer_Lab_Programs__ETL.ipynb
['drop_duplicates']

===================
notebooks\NarimanYou__Data-Analytics-efforts__Advanced_Python_1_Thursday_Lesson_Completed.ipynb
['drop_duplicates']

===================
notebooks\narz0001__Titanic_ML_Kaggle__Titanic ML.ipynb
['norm_log']

===================
notebooks\Nashra-Tazmeen__python__5.ipynb
['IQR']

<unknown>:1: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.

Processing 5350

===================
notebooks\NaufalArsyaputraPradana__Data_Mining-A11.2022.14606__Analisis_Klasterisasi_Pada_Transaksi_Penjualan_Menggunakan_Algoritma_K_Means_Clustering.ipynb
['norm_min_max']

===================
notebooks\nawaf-alageel__Research-Methods__API.ipynb
['drop_duplicates']

===================
notebooks\nchamara91__MLModalClimateDisease__Climate_Disease_Model.ipynb
['IQR']

===================
notebooks\ndmch3w__ML_AppliedStat__pj.ipynb
['zscore', 'zscore']

===================
notebooks\Negm2000__RNN-Neural-Net-From-Scratch__CH3.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\nehashukla91__-Python---Data-Structure__Feature_Engineering.ipynb
['norm_min_max']

===================
notebooks\nelioasousa__iiot_threat_detec__exp01__basic_decision_tree.ipynb
['norm_min_max']

===================
notebooks\netanelazran11__BiomedSciAI_ACL_Project__cellxgene_mouse_dataset_split.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\neural-data-science__NESC_3505_textbook__data_cleaning.ipynb
['IQR', 'zscore']

===================
notebooks\NguyenXuanGiang30__Data_Mining__02_preprocess_feature.ipynb
['drop_duplicates']
Processing 5400

===================
notebooks\nickjwheatley__march_madness_predictor__supervised_model_selection.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\nicolemd7__Used-Car-Prices-Predictions__Used Audi Price Prediction-checkpoint.ipynb
['IQR', 'IQR', 'norm_min_max']

===================
notebooks\nifedara__Modeling-Earthquake-Damage__Modeling Earthquake damage.ipynb
['norm_min_max']

===================
notebooks\Nifoluwa__Analytics-repo___Modelling-checkpoint.ipynb
['norm_log']
Processing 5450

===================
notebooks\nilayb12__Scripts__Processor.ipynb
['drop_duplicates']

===================
notebooks\NipunPatel2004__Data-analytics--task__pr11-checkpoint.ipynb
['drop_duplicates', 'zscore', 'drop_duplicates']

<unknown>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.


===================
notebooks\niravrshah__snowflake-implicit-bpr-retail-recommender__RECOMMENDER_SYSTEM_WITH_IMPLICIT_BPR.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\NirBtw__Customer-Clustering-task__crude_oil_price_forecast.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\nischalshrestha__automatic_wat_discovery__notebook8a95ec97d7.ipynb
['bin_equal_width_5']

===================
notebooks\nishanthoo7__YBI-foundation-__Data science.ipynb
['drop_duplicates']

===================
notebooks\NishiParikh16__stability-PSCs__ANN_for_Perovskite_stability.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\nitishtalekar__ProjectsGit__Classification(root).ipynb
['norm_min_max']
Processing 5500

<unknown>:37: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\noelcodes__aiap_tech_test__5-machine-learning.ipynb
['norm_min_max']

===================
notebooks\noiseless47__icse-final__isce.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_min_max', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\NotHydra__evori-dreamwings-finalis-hackathon-kic__ml_alias_resolution.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Novly57__IA_Framework_DefiIA__analysis-checkpoint.ipynb
['drop_duplicates']

===================
notebooks\NoxMoon__inside_beauty__statistical_test_price.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 5550

===================
notebooks\NumEconCopenhagen__lectures-2019__Workflow_and_debugging.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
notebooks\nycdatasci__bootcamp009_project__Random Forest.ipynb
['bin_equal_frequency_5']

===================
notebooks\NYUDataBootcamp__Projects__Ding-Baseball+Weather.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:28: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<unknown>:89: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
notebooks\ohjho__recommendation_system__Hybrid with Lightfm.ipynb
['drop_duplicates']
Processing 5600

===================
notebooks\olgasilyutina__emopok__emopok_xgboost.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\omarabdalla869__data-analalisys-of-SFsalaries__project data.ipynb
['IQR', 'IQR']

===================
notebooks\Omvishesh__ML-Assignment__10.ipynb
['norm_log', 'IQR']

===================
notebooks\Omvishesh__ML-Assignment__2.ipynb
['drop_duplicates']

===================
notebooks\oneapi-src__oneAPI-samples__Clustering_Methods_Exercises.ipynb
['norm_log']
Processing 5650

===================
notebooks\oreo496__BUDGET_MANAGEMENT__AI_DL_MODEL.ipynb
['isolationForest']

===================
notebooks\osman-sultan__Underlying-Factors-in-Soccer-Injuries__data_preprocessing.ipynb
['drop_duplicates', 'drop_duplicates', 'isolationForest']
Processing 5700

===================
notebooks\ovenmakemeheat__spai-ss6-archive__pipeline_xgb copy 2.ipynb
['bin_equal_frequency_10']

===================
notebooks\p3choco__Project_AI__main.ipynb
['drop_duplicates']

<unknown>:9: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.

Processing 5750

===================
notebooks\Parag-khandelwal__parkinsons-disease-prediction__Parkinsons_disease_using_csv.ipynb
['norm_min_max']

===================
notebooks\parsahg2025__amazon-sales-analysis__Checkpoint_1.ipynb
['isolationForest']

===================
notebooks\parthgiramkar__Machine_Learning__With_piplelines.ipynb
['norm_min_max']

===================
notebooks\ParthRajauria__PredictingStudentGrade_EndToEndMLOPs__model_training.ipynb
['norm_log']

===================
notebooks\paulguz261__MIAD_2025_proy_final__selected_model.ipynb
['isolationForest']

===================
notebooks\pauloarayasantiago__insurance-cross-selling-prediction__super duper version 4.ipynb
['norm_min_max']

===================
notebooks\pavanjit09__HireSense__HireSense_Employee_Attrition_Analysis.ipynb
['drop_duplicates']
Processing 5800

===================
notebooks\Pawan4356__MLOPS__pipeline.ipynb
['drop_duplicates']

===================
notebooks\pawel0705__PythonTensorflowStuff__t3.ipynb
['norm_min_max']

===================
notebooks\pearlynliu__IS3107-JobLens__salary_prediction.ipynb
['norm_min_max', 'IQR', 'norm_min_max']

===================
notebooks\pebe007__Customer-Churn-Prediction-using-Streamlit__OOPS.ipynb
['drop_duplicates']

===================
notebooks\pedronatanaelfs__votes_prediction__global_votes_prediction_FULL.ipynb
['drop_duplicates']

===================
notebooks\pedrovfalcao__ProjetoAirbnb__tratamento.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.

Processing 5850

===================
notebooks\PhamKhoa96__tensorflow__data_processing.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\PharminfoVienna__Retraining_Notebook__JN.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\Phil-Dua__ECE_204__python_11.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Pieter414__projects-v2__House Value Regression.ipynb
['norm_min_max']

===================
notebooks\pinthoz__claude-notebook-skill__experiment-template.ipynb
['drop_duplicates']
Processing 5900

===================
notebooks\pooja30123__MLOps-End-to-End-Course__mynotebook.ipynb
['drop_duplicates']

===================
notebooks\PosgradoMNA__actividades-de-aprendizaje-A00819192__A00819192_MNA_IAyAA_semana_2_Actividad.ipynb
['norm_min_max']

===================
notebooks\potgieterphiline__UdemyTrainingCode__SVM1.ipynb
['norm_min_max']
Processing 5950

<unknown>:8: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\pradhyuman-yadav__Detecting-Parkinson-Disease__lab.ipynb
['norm_min_max']

===================
notebooks\PradipPantha__AI__PradipPantha_2417489_regression_final.ipynb
['IQR']

===================
notebooks\pradyuk__MLND__finding_donors.ipynb
['norm_min_max']

===================
notebooks\prajwalbang__Data-Wrangling-airline-survey-data__Project_2_Group9.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\pranavarora1895__advanced-data-visualization__Assignment6.ipynb
['drop_duplicates']

===================
notebooks\pranavvachhani__machine-learning__As.ipynb
['IQR', 'norm_min_max']

===================
notebooks\pranitashakya__DataAnalysis_Projects__DataPreprocessing_Project1.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 6000

===================
notebooks\Prathap-Ait__Cognifyz__data_cleansing.ipynb
['zscore', 'IQR', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\prathit1__CyberSecML__fl.ipynb
['isolationForest', 'isolationForest']

<unknown>:1: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.


===================
notebooks\pratikpv__predicting_bitcoin_market__Expr8-LSTM.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Bitcoin-Transaction-Price-Prediction__00.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Ethereum-Price-Prediction-Learning-PyTorch-RNN__00.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Fetal-Health-Classification__00.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Mobile-Price-Prediction__00.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Tabular-Playground-Series-Aug-2021-Clf__00.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Tabular-Playground-Series-Aug-2021-Reg__00.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Tabular-Playground-Series-May-2021__00-main.ipynb
['norm_min_max']

===================
notebooks\prdai-archive__Titanic-V4__00.ipynb
['norm_min_max']
Processing 6050

===================
notebooks\prwoolley__zero_shot_analysis__figures_scoring-functions.ipynb
['drop_duplicates', 'norm_log', 'drop_duplicates', 'bin_equal_width_10']

<unknown>:2: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\PSindhuri__Research-Project__Finding_Best_Activation.ipynb
['norm_min_max']

===================
notebooks\psuny1116__python_data_analysis__분류 분석(logistic regression, KNeighborsClassifier, decision tree, supprot vector classifier).ipynb
['norm_min_max']
Processing 6100

===================
notebooks\puzzle38__python_repository__해외_부동산_월세_예측_automl.ipynb
['norm_log', 'norm_log']

===================
notebooks\pvvkishore__NLPA_LAB_2025__Exp_5_Vectorizing_Text_TF_TF_IDF.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\pypi-ahmad__Machine-Learning-Projects__Burnout Risk Indicator Analysis.ipynb
['norm_min_max']

<unknown>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.

Processing 6150

===================
notebooks\qazidanishayub__Master_in_Data_Science_ITU__Abstractive_summariation_keras.ipynb
['drop_duplicates']

===================
notebooks\quannguyen02__Visualize-Oberlin-Energy-Usage__ML.ipynb
['norm_min_max']

===================
notebooks\Quratulain786__-R-versus-Python-EDA__RM.ipynb
['norm_min_max']

===================
notebooks\r-a-j__World-Value-Survey__tt.ipynb
['norm_min_max']

===================
notebooks\raddick__ungerrymandering__districts-cities.ipynb
['drop_duplicates']

===================
notebooks\radrogue1__RepFR_analyses__repfr_sem_crp.ipynb
['drop_duplicates']

===================
notebooks\radubauzh__Quantitative_Trading_Algorithm__LSTM.ipynb
['norm_min_max', 'norm_min_max']
Processing 6200

===================
notebooks\Rahul5655__Deep-Learning-Project__D7.ipynb
['norm_min_max']

===================
notebooks\Raidbourzam__TPs-BDM__healthcare.ipynb
['norm_min_max']

===================
notebooks\raj-deshmukh6403__dsbda__26.ipynb
['zscore']

===================
notebooks\RajaATAli__Machine-Learning-For-Early-Disease-Prediction__Random_Forest_Ensemble_Model_Diabetes_Prediction_Wider_HyperParameters.ipynb
['IQR', 'norm_log']

===================
notebooks\Rajesh727833__Laishram-Mukesh-Singh__SVM Forest Fires.ipynb
['drop_duplicates']

===================
notebooks\RajNikhil__ML_nanodegree__finding_donors-checkpoint.ipynb
['norm_min_max']

===================
notebooks\rajshah4__snowflake-notebooks__Madelon_scaling.ipynb
['norm_min_max']
Processing 6250

===================
notebooks\ramseylab__networkscompbio__class07_clustcoeff_python3_template.ipynb
['drop_duplicates']

===================
notebooks\ramseylab__networkscompbio__class08_components_python3_template.ipynb
['drop_duplicates']

<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:77: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:46: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:55: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:53: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:165: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.


===================
notebooks\RaulEcheverryLopez__Claser-Jose-Armando__Data_leakage.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 6300

===================
notebooks\rbaral__natural_language_processing__ColumnTransformer Meets NLP.ipynb
['norm_log']

===================
notebooks\rcdang__project-portfolio__Pandas_Cleaning_Checklist.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'IQR']

===================
notebooks\Reaemanz__Machine-Learning-Projects-in-Python__rnn-detailed-explanation-0-2246.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\REDDY99011__FDE__FDE_Lab_2(1RVU23CSE497).ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\regional-specter__Dispatch__jet-engine-predictive-maintenance-rul.ipynb
['norm_min_max']

<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\renasyan__datmin-tubes-mlbb__TUBES_DATMIN_MOBILE_LEGEND.ipynb
['IQR']

===================
notebooks\Renata1214__Adv_ML_Carbon_Price_Predictor_Project__Feature_engineering.ipynb
['drop_duplicates']
Processing 6350

===================
notebooks\ReynadelYolo__ML_Projects__DecisionTreeClassifier_IntroAI.ipynb
['IQR']
Processing 6400

===================
notebooks\ridgerunner03__PythonDataScienceHandbook__03.09-Pivot-Tables.ipynb
['bin_equal_frequency_2']

===================
notebooks\ridovci__Parkinsons-Disease-XGBoost__main.ipynb
['norm_min_max']

===================
notebooks\ridwaanhall__Dicoding-Machine-Learning-Intermediate__Copy_of_Ridwan_Halim_Pelatihan_Model.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\RileyLePrell__Rouge_Hat__Xg-notebook.ipynb
['IQR']

<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.


===================
notebooks\rishithaa9__Medical-Recommendation-System__ML.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\riyageorgek__Text-Classifiers__Text Classification Word2vec.ipynb
['norm_min_max']

===================
notebooks\rizkinabil__feature-weighting-effect-on-rnn-tiktok-sentiment-analysis__w2v_pretrained_svm.ipynb
['drop_duplicates']
Processing 6450

===================
notebooks\rizqihilman__rizqihilman__Data_Cleansing.ipynb
['drop_duplicates', 'IQR']

===================
notebooks\rizzyintrance__Phishing_detection__Model_implementaion.ipynb
['IQR']

===================
notebooks\rngustj9139__ML_PerfectGuide__2.5 데이터_전처리.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\robert-solomon12__MSc_Dissertation-Remote_Work_On_Mental_Health__data_preprocessing_sr.ipynb
['IQR']

===================
notebooks\RobertoL00__Neural-Data-Science-in-Python__data_cleaning.ipynb
['IQR', 'zscore']

===================
notebooks\rockerspace__DATA-ANALYST-CAPSTONE-PROJECT__M3ExploratoryDataAnalysis-lab.ipynb
['IQR']
Processing 6500

===================
notebooks\roopchandrika__DS602__Roop Chandrika Mallela_YV25690_602_week3 - homework.ipynb
['drop_duplicates']
FAILED: notebooks\Roopesht__b4_project_1__a.ipynb
cannot access local variable 'newcell' where it is not associated with a value

===================
notebooks\roshanr11__Deep-Learning-for-Trading-RNN-LSTMs-getRichWithStocks.py__redoLSTMv1-checkpoint.ipynb
['norm_min_max']

===================
notebooks\RozenAstrayChen__House-prediction__LR.ipynb
['norm_min_max']

===================
notebooks\RozenAstrayChen__House-prediction__RF.ipynb
['norm_min_max']
Processing 6550

===================
notebooks\rubenfonnegra__analitica_datos__Practicum_9.ipynb
['norm_log']

<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.


===================
notebooks\rudranis__DSBDL_CODE__13.ipynb
['IQR']

===================
notebooks\rudranis__DSBDL_CODE__14.ipynb
['IQR']

===================
notebooks\rudranis__DSBDL_CODE__15.ipynb
['IQR']

===================
notebooks\rudranis__DSBDL_CODE__16.ipynb
['IQR']

===================
notebooks\rudranis__DSBDL_CODE__26.ipynb
['IQR']

===================
notebooks\rulezcasa__AI-ML-basics__SVM.ipynb
['norm_min_max']

===================
notebooks\Rullyro__research_BotnetDetect__DT_tahapan2.ipynb
['norm_min_max']

===================
notebooks\rushilp7__housing-prices__l2.ipynb
['norm_log', 'norm_log']
Processing 6600

===================
notebooks\ruturaj0626__Environmental-Data-Analysis__Environmental-Data-Analysis.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_log', 'norm_min_max', 'norm_min_max']

===================
notebooks\rvizarreta__moritos-codas__BDT.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
notebooks\ryanapierce__fantasy-football-assistant__init_1_lag_dataset_generator.ipynb
['drop_duplicates']

===================
notebooks\S3oudd__eplTopScorers__Machine_Learning_EPL.ipynb
['norm_min_max']

===================
notebooks\SaadDamine__Machine-Learning-A-Z__Data Preprocessing.ipynb
['norm_min_max']

===================
notebooks\sabaly__TEFI-PATE__adult-checkpoint.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\sachinsachu20__GED__ST.ipynb
['drop_duplicates']
Processing 6650

===================
notebooks\sagu3628__LA-Crime__DT.ipynb
['drop_duplicates']

<unknown>:16: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.


===================
notebooks\SahilSawant0605__Excelr-Assingment__EDA2.ipynb
['norm_min_max', 'norm_log', 'isolationForest']

===================
notebooks\SaiShashank-10__ml-lab__1.ipynb
['norm_min_max']

===================
notebooks\sajedjalil__Data-Science-Pipeline-Detector__costa-tu-rica.ipynb
['norm_min_max']

===================
notebooks\sajedjalil__Data-Science-Pipeline-Detector__deep-learning-tps-december-2021.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\sajedjalil__Data-Science-Pipeline-Detector__ds-stdy-mercari-namiki.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\sajedjalil__Data-Science-Pipeline-Detector__fillna.ipynb
['norm_log']
Processing 6700

===================
notebooks\sajedjalil__Data-Science-Pipeline-Detector__fraud-detection-different-scenarios-final-model.ipynb
['norm_log']

===================
notebooks\sajedjalil__Data-Science-Pipeline-Detector__lab2-glhf.ipynb
['norm_min_max']

===================
notebooks\sajedjalil__Data-Science-Pipeline-Detector__predict-future-sales-lightgbm-framework.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\Saltizm__42-misc__code.ipynb
['IQR']
Processing 6750

===================
notebooks\samesense__pathopredictor__predict-for-missense-clinvar-union_features.ipynb
['drop_duplicates']

===================
notebooks\samuel-eric__fraud-transaction-classification__fraud_transaction_classification.ipynb
['IQR']

===================
notebooks\Samuel-ZDM__AM-Codes__Sample_Model_Evaluation_class.ipynb
['norm_min_max']

<unknown>:3: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.

Processing 6800

===================
notebooks\Sanhith30__Data-Science-And-ML-Projects__3.Mean_tfidf_W2V.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\SantiagoMain__DS3000-Project-Fall-2025__Phase_III (1)-checkpoint.ipynb
['IQR']

===================
notebooks\Santostang__box-office-prediction__3_Clean Wrong matches - Metacritic and BoxOfficeMojo.ipynb
['drop_duplicates']

===================
notebooks\SapnaSChavan__Medicare-Claim-Fraud-Detection__base_model 5.ipynb
['bin_equal_frequency_10']

===================
notebooks\Saraavana__reclamation-processing__01-prepare-data.ipynb
['drop_duplicates']

===================
notebooks\sardor014__project_andan_2023__ML.ipynb
['drop_duplicates']

===================
notebooks\saritmaitra__Customer-Churn-analysis__Calibration.ipynb
['norm_min_max']

===================
notebooks\sasirekhas__DataScienceProjects__Detecting Parkinson’s Disease – Python Machine Learning Project.ipynb
['norm_min_max']
Processing 6850

===================
notebooks\Sastraaaa__Car-Price-Analyst__car.ipynb
['zscore']

===================
notebooks\satabios__scandia__scandia-checkpoint.ipynb
['norm_min_max']

===================
notebooks\SaTr0V__RobustFraudDetection__04_final_adv_eval.ipynb
['drop_duplicates', 'norm_log']

===================
notebooks\SatyamSisodiya__ueba__ueba.ipynb
['isolationForest']

===================
notebooks\saurav-singh321__Flask_ML__spaceship titanic.ipynb
['IQR']

===================
notebooks\saust1__Project-OptiC4__1.0.0 Preprocess.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\SayamAlt__Global-Equity-Forecasting-using-LSTM__global-equity-forecasting-using-lstm.ipynb
['norm_min_max']

===================
notebooks\sayemimtiaz__kaggle-notebooks__forestfires.ipynb
['norm_log', 'norm_log']
Processing 6900

===================
notebooks\scochran3__LazyApartment__Data Model Preparation-checkpoint.ipynb
['drop_duplicates', 'norm_log']

===================
notebooks\scochran3__LazyApartment__Model Prediction-checkpoint.ipynb
['drop_duplicates']

<unknown>:3: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\B" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\B"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.


===================
notebooks\seb208__COVID-19-Case-Prediction-__Covid_19_Case_Predictions.ipynb
['norm_min_max']
Processing 6950

===================
notebooks\SECE-2022-2026__ml-project-hackathon-mukesbkumar__Credit Card Fraud Detection.ipynb
['drop_duplicates']

===================
notebooks\seifgendy__AI__Session 4 DS.ipynb
['drop_duplicates']

===================
notebooks\semi0612__DL_study__1018.ipynb
['norm_log', 'IQR', 'norm_log']

===================
notebooks\SergeiRage__Projects__Project_car_accident_Nikulin.ipynb
['drop_duplicates']

===================
notebooks\sethkipsangmutuba__Database-Management-System__Week_8.ipynb
['drop_duplicates']
Processing 7000

===================
notebooks\SHADOWZERO93__Loan_Approval_Prediction__Loan_Approval_prediction.ipynb
['norm_min_max']

===================
notebooks\shaminchokshi__Detection-of-Parkinsons-Disease-using-speech-parameters__Untitled.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\Sharafunneesa-pp__logistic_regression_implementation__logistic_regression-checkpoint.ipynb
['drop_duplicates']
Processing 7050

===================
notebooks\sharmaraja__Basic_Machine_Learning_Projects__clustering-global-country-prosperity.ipynb
['norm_min_max']

===================
notebooks\SharmaVrishab__numpy-and-pandas-for-analysis__main.ipynb
['drop_duplicates']

===================
notebooks\Shaurya8769__BROWNrepShaurya__Team7_logistic_regression_1_3.ipynb
['norm_min_max']

===================
notebooks\shayankebriti__ML-archive__C3_W2_RecSysNN_Assignment.ipynb
['norm_min_max']
Processing 7100

===================
notebooks\shigemorita__python_chemometrics_ohmsha__0901.ipynb
['isolationForest']

===================
notebooks\shigemorita__python_chemometrics_ohmsha__0902.ipynb
['isolationForest']

===================
notebooks\shigemorita__python_chemometrics_ohmsha__0903.ipynb
['isolationForest']

===================
notebooks\ShikharSrivastava-aiml__TwitterSentimentAnalysis-NaturalLanguageProcessing__bert_final.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\shimap64__Salary-Estimate-Regression__preprocessing of Salary regression .ipynb
['norm_min_max']

<unknown>:22: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.


===================
notebooks\shivampandey79__hsbiybyasbb__1.ipynb
['drop_duplicates']

===================
notebooks\Shivateja-31__Fraud_Detection_System__FrauduDetection (1).ipynb
['norm_log']

===================
notebooks\Shoaibrehmane__Predicting-Side-Effects-from-Patient-Drug-Reviews-Using-NLP-Techniques__5-SVM.ipynb
['norm_min_max']

===================
notebooks\shobith-s__AURORA-V2__meta_learning_training.ipynb
['norm_min_max']

===================
notebooks\shoelesshoe__PAICA1__data_cleansing.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 7150

===================
notebooks\ShreeTilakraj__Finale-Dashboard__M2DataWrangling-lab.ipynb
['drop_duplicates']

===================
notebooks\ShreyashDevali2518__Data-Science-Lab__ML.ipynb
['IQR']

===================
notebooks\shruti-1007__mobile-usage-insights__data_preprocessing.ipynb
['IQR', 'norm_min_max']

===================
notebooks\shuangjianxi__745__3A.ipynb
['norm_min_max']
Processing 7200

===================
notebooks\sienaroma__RE-MAX-housing-price-predictions__REMAX_Home_Price_Models.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\silpiria98__aiffel_camp__project3_breast_cancer.ipynb
['norm_min_max']

<unknown>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
notebooks\SimplyYasH19__ds__2.ipynb
['IQR', 'norm_log']
Processing 7250

===================
notebooks\singhrama__Insurers_TIC__United_Health_Data_Segregation.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\Sithik19__mpg-prediction-ai-agent__Reg_model.ipynb
['IQR']

===================
notebooks\skabone__applied-research-portfolio__Job_Change_Prediction_Data_Mining.ipynb
['norm_log']

<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
notebooks\skanderkaroui__TP2-Machine-Learning__TP3.ipynb
['IQR', 'IQR']

===================
notebooks\skedaddlers__Machine-Learning__cleanPredict.ipynb
['drop_duplicates']

===================
notebooks\Skolasta__Machine-Learning-Portfolio__TelcoChurn.ipynb
['IQR']

===================
notebooks\skywateryang__timeseries101__cp7.ipynb
['drop_duplicates']
Processing 7300

===================
notebooks\slowLEAN__TitanicT1__titanic.ipynb
['IQR']

===================
notebooks\smb-h__corporate-bankruptcy-prediction__rnn.ipynb
['norm_min_max']

<unknown>:1: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id 'bd5e6ddc' detected. Corrected to '4b665d62'.
  validate(nb)

Processing 7350

===================
notebooks\songminkyu__llm_engineering__end_of_week_assesment.ipynb
['drop_duplicates', 'IQR', 'drop_duplicates']

===================
notebooks\sonjasonja123__petnica-compfin-2025-projekat__2.ipynb
['bin_equal_frequency_10']

===================
notebooks\SonyFebri__Machine-Learning__ML1.ipynb
['norm_min_max']

===================
notebooks\Souvik2376__Data-Science-Machine-Learning-Projects__Titanic Survival Data Analysis & Classification.ipynb
['norm_log']
Processing 7400

===================
notebooks\Sri-Tulasi__VIT_Morning_Slot__SVC.ipynb
['norm_min_max']

===================
notebooks\sriratnachintapalli__Amazon-Sales-Analysis__dc.ipynb
['drop_duplicates', 'zscore']

===================
notebooks\SriYanisaa__Sistem-Rekomendasi-Hybrid-FIltering__cleaning_product.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\sssison__SecURL__lexical_filter_feature_selection.ipynb
['zscore']
Processing 7450

===================
notebooks\StarMarks01__Codes__Income Prediction dataset.ipynb
['IQR']

===================
notebooks\SteckerLecker__WorkspaceKDT__Gruppenarbeit.ipynb
['isolationForest']

===================
notebooks\stellayannn__DataScience_TongYan__Main Project_Tong Yan.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
notebooks\Stopira18__FICO-Score-Quantization-for-Credit-Scoring-A-Machine-Learning-Perspective__code.ipynb
['norm_log']

<unknown>:11: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
notebooks\stxupengyu__Air-Quality-Prediction__B3.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 7500

===================
notebooks\sud2268__Credit-card-Fraud-Detection__IEEE.ipynb
['norm_log', 'norm_log']

===================
notebooks\sudeepSubedi01__Natural-Language-Processing__Average-Word2Vec.ipynb
['drop_duplicates']

===================
notebooks\sudipta-rkmrc__RKMRC-Coding__MLP_with_diabetes_dataset (1).ipynb
['norm_min_max']

<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:42: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\sunilmadishetty__AIML-2025__week5_lab01.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\Sunisa-Yuki__movie-ratings-analysis__01-DataCheckpoint.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 7550

===================
notebooks\supawitlearn__customer-churn-prediction__04_feature.ipynb
['drop_duplicates']

===================
notebooks\Surasan01__Dengue-Forecast__AutoGluon_h1_h2_02 (1).ipynb
['norm_log', 'drop_duplicates']

===================
notebooks\SuRreal1000__capstone_know_your_ship__04_1_Model_Random_Forrest.ipynb
['norm_min_max']

===================
notebooks\sushrutghimire1__Fraud-Detection-Temporal-Correlation__temporal_correlation.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\Svelumula-tech__hello-world__Srija_EC.ipynb
['norm_min_max']

===================
notebooks\swaroopms658__AIBOM__Untitled1-checkpoint.ipynb
['norm_min_max']
Processing 7600

<unknown>:7: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\SzymonChirowski242621__linear-regression-analysis-silver-medal-challenge-block-a__data_processing.ipynb
['IQR']

===================
notebooks\tal-ladijinski__ProgrammerProfiling__feature-engineering-with-randomized-search.ipynb
['norm_min_max']
Processing 7650

===================
notebooks\Tamil-Ilakkiya1404__Canteen-Management-System__sentiment_analysis.ipynb
['drop_duplicates']

===================
notebooks\tanaykasyap__Interconnections-Yale__c14_style_Q.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\tanx1825__MINI-PROJECT-6-SEM__ann.ipynb
['norm_min_max']

<unknown>:9: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\TasneemBadry__TasneemBadry__nn.ipynb
['norm_min_max']

===================
notebooks\Tate0524__ML100day__Day_030_Feature_Selection.ipynb
['norm_min_max']
Processing 7700

===================
notebooks\tea-ok__JAMK-course-review__cleaning_and_preparation.ipynb
['drop_duplicates']

===================
notebooks\team-epoch__EPOCH_Datathon_4th__4th_miniproj_code_참새와고양이.ipynb
['norm_log']

===================
notebooks\tehtianyan__Life-Sciences-Descriptive-Statistics__Code_Holmusk - Teh Tian Yan v4.ipynb
['norm_log']

===================
notebooks\Tejadithya__stock-price-prediction-using-ML__stockprice.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\Tejas-c-0__Heart-Disease-Predictor__Heart Disease.ipynb
['drop_duplicates']

===================
notebooks\tendanimasala21__Movie-Data-Analysis__Movie_sales_project.ipynb
['drop_duplicates']

===================
notebooks\Teradata__developer-resources__ModelOps_Operationalize_v6.ipynb
['norm_min_max']

===================
notebooks\Teradata__jupyter-demos__Automatic_DataPreprocessing_tdprepview.ipynb
['norm_min_max', 'norm_min_max']
Processing 7750

===================
notebooks\Tesnime__stagePFE__AI.ipynb
['drop_duplicates']

===================
notebooks\thangnch__MIAI_Customer_Churn_Prediction__CCP.ipynb
['norm_min_max']
Processing 7800

<unknown>:6: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
notebooks\theo-futol__Tardis__tardis_eda.ipynb
['drop_duplicates']

===================
notebooks\theoboiss__GrassGrowthDisaggregation__data_analysis.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\thinlh07__IBM-Data-Analyst-Capstone-Project__Exploratory Data Analysis.ipynb
['IQR']

===================
notebooks\Thurin7__ecommerce-analytics__ecommerce_analyses_completes.ipynb
['drop_duplicates', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
notebooks\Time4045__DeviceDetect__EDA.ipynb
['drop_duplicates']
Processing 7850

===================
notebooks\TimovNiedek__azure-ml-playground__EDA.ipynb
['norm_min_max']

===================
notebooks\TimRyder9876__Python__Ch12_Machine_Learning.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\Tina-Sterite__Hotel-Analysis-with-Python__Unleashing Data Insights with Python and VS Code Mastery.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\tingruiray__Music-Trend-Analysis__data_cleansing_EDATextAnalysis.ipynb.ipynb
['drop_duplicates']

<unknown>:14: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.


===================
notebooks\TomoCid__ProyectoDeepLearning__ProyectoDeepLearning.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\TomODonn__ECGR-4105__Assignment6.ipynb
['norm_min_max']

===================
notebooks\tonywork737__data-mining__NB.ipynb
['norm_log', 'norm_log']

===================
notebooks\traceswrldd__traceswrldd__Uptrail_project_week_3.ipynb
['drop_duplicates', 'IQR']
Processing 7900

===================
notebooks\trainningjava__MaratonaBehindCode2020__Testes_desafio_2_IBM.ipynb
['norm_min_max']

===================
notebooks\TreeTechDev__biomodelml__Feature Analysis by Channel.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\TreshMom__CS-Space-contest__G.ipynb
['drop_duplicates']

===================
notebooks\tubarao312__Geoprotocol__ML.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\tuhinbidyanta__mineral-forcasting__ml.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 7950

===================
notebooks\tulip-lab__SIT742__M05C-IsolationForest.ipynb
['isolationForest', 'isolationForest']

===================
notebooks\Tuminha__Frankenstein-LLM-Fine-Tuning-with-Mistral__01_eda_dataset.ipynb
['drop_duplicates']
Processing 8000

===================
notebooks\ud204__Python-project__UpdatedMost.ipynb
['drop_duplicates']

===================
notebooks\Udacity-MachineLearning-Internship__finding_donors__finding_donors.ipynb
['norm_min_max']

===================
notebooks\UdithaMayadunna__Ensemble-Deep-Learning-Models-for-Stock-Price-Forecasting__LSTM(Ceylon_Tobacco).ipynb
['norm_min_max']

<unknown>:2: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\UmamaQayumKhan__FYP__ShapFL2.ipynb
['drop_duplicates', 'IQR']
Processing 8050

===================
notebooks\UserCDP__Hi_Paris_Data_Science_Bootcamp_2023__ML.ipynb
['drop_duplicates']

===================
notebooks\UsmanGohar__FairEnsemble__0-catboost-and-other-class-algos-with-88-accuracy.ipynb
['norm_log']

===================
notebooks\UsmanGohar__FairEnsemble__1-income-prediction-84-369-accuracy.ipynb
['norm_log']

===================
notebooks\utkarshrajputt__Outlier_Detection__Outlier_Detection_Assignment.ipynb
['IQR']

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\uzay00__KaVe__Müşteri Kayıp (Churn) Analizi-1.ipynb
['norm_min_max']

===================
notebooks\uzilon__Coursera__M3ExploratoryDataAnalysis-lab.ipynb
['IQR', 'IQR']

===================
notebooks\vahid-khazaei-nezhad__ml_lecture4011__Association_Rule_Mining_Apriori.ipynb
['drop_duplicates']
Processing 8100

===================
notebooks\VaibhavBajpaij__Python-Library__Pandas.ipynb
['drop_duplicates']

===================
notebooks\vaishnavib013__Infosys_Internship__Team 4 salary x experience (1).ipynb
['norm_log', 'norm_log', 'norm_min_max']

===================
notebooks\valdemaras-pletkus-euromonitor-com__flight-delay__manage-flight-data.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\valeriandwi__dicoding-data-scientist-expert__Data Cleansing Preprocessing PySpark.ipynb
['norm_min_max']

===================
notebooks\Varshithatangeti__FMML_2023_ASSIGNMENTS__Regression_Lab_2.ipynb
['norm_min_max']

<unknown>:21: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.

Processing 8150

===================
notebooks\VasuAdireddy__Courses__lab_jupyter_logistic_regression.ipynb
['norm_min_max']

===================
notebooks\vermasrijan__srijan-gsoc-2020__GTEx_V8_NN-regression.ipynb
['norm_min_max', 'norm_min_max']

===================
notebooks\vfamim__spotify-popularity__winner_song-checkpoint.ipynb
['norm_min_max']
Processing 8200

===================
notebooks\vighn-esh__Zomato_casestudy__EDA.ipynb
['drop_duplicates']

===================
notebooks\Vijaya-1621__Vijaya_16__ml.ipynb
['norm_min_max']

===================
notebooks\viniciussogo__EBAC__Mod_12_Tarefa_03.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
notebooks\vinodhkumargaggera__vinodhdatascinse__preprocessing.ipynb
['norm_min_max']

===================
notebooks\vipulbeniwal01__Music-Recommendation-System__p.ipynb
['drop_duplicates']

===================
notebooks\vipulnikam25__DETECTING-PARKINSON-S-DISEASE-WITH-XGBOOST__main.ipynb
['norm_min_max']

===================
notebooks\virajkalhara__ipl-ml-team-selection__ipl_ml_models.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\vishaltyagig__Machine_Learning__titanic_using_pipeline.ipynb
['norm_min_max']
Processing 8250

===================
notebooks\VISWA68__TSA__tsa_exp9 (1).ipynb
['norm_min_max']

===================
notebooks\vivianah__dataScienceNotebooks__03.09-Pivot-Tables.ipynb
['bin_equal_frequency_2']

<unknown>:16: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.


===================
notebooks\vKenjo__dm-final-proj__attrition_analysis.ipynb
['IQR']

===================
notebooks\vngne__fds-digitalent__data-cleansing.ipynb
['IQR', 'IQR', 'IQR']

===================
notebooks\VsinK14__Robust-Aggression-Detection-in-Textual-Content-Adversarial-Learning-with-GANs-__CNN.ipynb
['drop_duplicates']

===================
notebooks\Vucibatina__eur_minute_by_minute_predictor__EURPredictor.ipynb
['norm_min_max']

===================
notebooks\vwang0__DS_ML_Projects__Credit Risk Modeling - Preparation - 5-18.ipynb
['norm_log', 'norm_log', 'norm_log']
Processing 8300

<unknown>:11: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.

Processing 8350

===================
notebooks\Waihong1__peergroep15-data-driven-logistics__main.ipynb
['norm_log', 'norm_log']

===================
notebooks\wandwan__doomed__logisticRegression.ipynb
['norm_log']

===================
notebooks\wang4009kai__CSC2558Project__RL.ipynb
['drop_duplicates']

===================
notebooks\waviad__NBA-Heights-EDA__EDA Project - NBA.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\weareng__radiomics_neuroblastoma__2_models.ipynb
['norm_min_max']

===================
notebooks\webclinic017__inception__micro_NN.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 8400

===================
notebooks\Weinihsiang__Pytorch__Lab4_Data_Imputation.ipynb
['norm_min_max']

<unknown>:34: SyntaxWarning: invalid decimal literal
<unknown>:37: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:28: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:38: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:42: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.


===================
notebooks\WiktoriaPabis__Praca_mgr_przygotowania__Baza_danych_z_pracy_bazowej.ipynb
['drop_duplicates']

===================
notebooks\willy0222__ML_100day__Day_031_HW_特徵評估.ipynb
['norm_min_max']

===================
notebooks\Winfry__EnergyPredictionMachineLearning__SMART METER .ipynb
['norm_min_max']

===================
notebooks\wisam007__qiyas_wi__Spam_Classification_Lab_Guide_Commented.ipynb
['bin_equal_width_10']
Processing 8450

===================
notebooks\Xenaroxy__TIL____[01]Mini_proj_lg_obesity.ipynb
['norm_min_max']

<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.

Processing 8500

===================
notebooks\YanLiuGit__IBM-data-scientist-course__Data_Cleaning_Lab.ipynb
['drop_duplicates', 'norm_min_max', 'zscore']

===================
notebooks\yash-dange__Credit-Score-Classification-Multi-Class-__AML_Project.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\yashaur__fraud-detection-project__1 EDA.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
notebooks\Yashrajgk__ds__1_DS.ipynb
['norm_min_max']
Processing 8550

===================
notebooks\yaskyj__housing-price-regression__Housing Price Regression.ipynb
['norm_log', 'norm_log', 'norm_min_max']

<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
notebooks\yatin-t__XRP-prediction__EDA.ipynb
['drop_duplicates', 'norm_min_max']

===================
notebooks\yauheni-chekan__ML-Spring-Practical-Tasks__22_LR_JA_Yauheni_Chekan_DP.ipynb
['norm_min_max']

===================
notebooks\yellatp__Supply-Chain-Analysis-Python__Data_cleaning.ipynb
['drop_duplicates', 'IQR']

===================
notebooks\ygcahyono__hackrefinitiv__2-FTSE_Preprocessing.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\yhstory62__llm_engineering__end_of_week_assesment.ipynb
['drop_duplicates', 'IQR', 'drop_duplicates']

===================
notebooks\Yiujin__gradProject__KeyFrameExtraction_test.ipynb
['norm_min_max']

===================
notebooks\YizheWill__titanic__titanic_grid_search_cv.ipynb
['norm_min_max']

===================
notebooks\yneha70__Yarram-Neha__Tasks.ipynb
['IQR', 'IQR']
Processing 8600

===================
notebooks\yogitaaax22__eeg_website__google_colab.ipynb
['winsorize']

<unknown>:12: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.


===================
notebooks\Yop-La__prediction-valeur-cadastre__nn.ipynb
['drop_duplicates']

===================
notebooks\Yorko__mlcourse.ai__project_telecom_response_prediction_MdScntst.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
notebooks\YoRu-Cat__PyPy__8.ipynb
['drop_duplicates', 'norm_min_max', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'norm_min_max', 'drop_duplicates', 'drop_duplicates']

===================
notebooks\YounginYoon__KT_AIVLE_SCHOOL__종합실습_02_Mobile.ipynb
['norm_min_max']

===================
notebooks\YoussefEslam29__DATA-ANALYSIS__Model_1.ipynb
['IQR']

===================
notebooks\youssefhass__ml-predict-student-dropout__03_cleanse_data_students_dropout.ipynb
['IQR']
Processing 8650

===================
notebooks\yuvism__Pulpnet__Final_Submit_file.ipynb
['drop_duplicates']

===================
notebooks\yuzuponikemi__machine-learning-playground__25_categorical_variable_encoding_improved_v2.ipynb
['drop_duplicates']

===================
notebooks\YYYYMao__2nd-ML100Days__Day_028_HW.ipynb
['norm_min_max']

===================
notebooks\ZaakZoeng__SSN4PaCDM__SSN4PaCDM.ipynb
['drop_duplicates', 'drop_duplicates']

===================
notebooks\zakill96__pra__ve3.ipynb
['norm_min_max']
Processing 8700

===================
notebooks\ZamirPineda__spark_colab_package__Masterclass_ETL_Data_Quality.ipynb
['drop_duplicates']

===================
notebooks\zekoNinja__Stock-Prediction__Stocks Prediction_Vale-Optimized -Copy1.ipynb
['norm_min_max']

===================
notebooks\Zeyad-Daowd__VoiceProfiler__eda.ipynb
['drop_duplicates']

===================
notebooks\zhimin-z__Asset-Management-Topic-Modeling__RQ5.ipynb
['norm_log']

===================
notebooks\zhonghaozhan__REAL-IoT__Anomal_E_cicids2017.ipynb
['isolationForest', 'isolationForest', 'isolationForest', 'isolationForest']
Processing 8750

<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
C:\Users\danie\PycharmProjects\ATE-shifting\.venv\Lib\site-packages\nbformat\__init__.py:96: DuplicateCellId: Non-unique cell id '7ffacb27' detected. Corrected to '5fc4257d'.
  validate(nb)


===================
notebooks\zilto__IFT6390-Comp1__comp1_dev.ipynb
['drop_duplicates', 'norm_log']

===================
notebooks\zimkk__Anomaly-Detection-System__ADS.ipynb
['norm_log']

===================
notebooks\ziyapatel0169__TEAM_BROKER_AMAZON_SALES_ANALYSES__SRC.ipynb
['isolationForest']

===================
notebooks\zlatankr__Projects__Titanic Model Walk-Through-checkpoint.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'drop_duplicates']

===================
notebooks\zohaibbashir__Pakwheels.com-Webscraping-and-Data-Analysis-Visualization-using-Tableau__PW.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:79: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:129: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:62: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:63: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:62: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:61: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.

"""


# Find every line that looks like a Python list
list_strings = re.findall(r"^\[.*\]$", text, flags=re.MULTILINE)

# Convert them into actual Python lists
lists = [ast.literal_eval(s) for s in list_strings]

# print("Lists:")
# print(lists)

print(f"found {len(lists)} pipes")
# Average list size
avg_size = sum(len(lst) for lst in lists) / len(lists) if lists else 0

print(f"\nAverage list size: {avg_size:.2f}")

# Save to CSV (one row per list)
df = pd.DataFrame({
    "list": [str(lst) for lst in lists],
    "size": [len(lst) for lst in lists]
})
df.to_csv("pipelinesDataPrep.csv", index=False)

print("\nSaved to lists.csv")

KAGGLE


In [ ]:
(
    transform_probabilities_kaggle,
    transition_probabilities_kaggle,
) = analyze_corpus("kaggle_notebooks")

In [ ]:
prob_dict_kaggle = {}
eps = 1e-10
for transform_op in transformations:
    prob_dict_kaggle[transform_op] = transform_probabilities_kaggle.get(transform_op, eps)

prob_dict_kaggle['zscore_clip_3'] = transform_probabilities_kaggle.get('zscore', eps)
prob_dict_kaggle['zscore_filter_3'] = transform_probabilities_kaggle.get('zscore', eps)
print(prob_dict_kaggle)


In [ ]:
for k in prob_dict.keys():
    print(f"{k} | github: {prob_dict[k]} || kaggle: {prob_dict_kaggle[k]}")

In [ ]:
def get_dicts_stats(dict1, dict2):
    """Calculates correlation between two dicts with the same keys."""
    # Align values by the exact same key order
    keys = list(dict1.keys())
    x = np.array([float(dict1[k]) for k in keys])
    y = np.array([float(dict2[k]) for k in keys])
    print(f"min is {min([min(x), min(y)])}, max is {max([max(x),max(y)])}")
    # Calculate and return the Pearson correlation coefficient
    corr =  np.corrcoef(x, y)[0, 1]
    mae = np.mean(np.abs(x - y))
    rmsd = np.sqrt(np.mean((x - y) ** 2))

    return corr, mae, rmsd

print(get_dicts_stats(prob_dict, prob_dict_kaggle))

In [ ]:
import matplotlib.pyplot as plt
dict1 = prob_dict
dict2 = prob_dict_kaggle
keys = list(dict1.keys())
x = np.array([float(dict1[k]) for k in keys])
y = np.array([float(dict2[k]) for k in keys])

fig,ax = plt.subplots()
ax.scatter(x,y)
ax.set_yscale('log')
ax.set_xscale('log')
ax.axline((0,0),slope=1,color='grey',dashes=(3,3))
plt.show()

In [ ]:
import ast
import re
import pandas as pd




# Find every line that looks like a Python list
text = r"""
Found 5500 notebooks
Processing 0

===================
kaggle_notebooks\aaronds_us-traffic-accidents-analysis-and-prediction.ipynb
['bin_equal_width_10', 'bin_equal_width_10', 'bin_equal_width_10', 'bin_equal_width_10', 'bin_equal_width_10']

===================
kaggle_notebooks\abaojiang_lmsys-detailed-eda.ipynb
['drop_duplicates', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\abazdyrev_nn-w-o-skew.ipynb
['norm_log']

===================
kaggle_notebooks\abdallahsaadelgendy_diabetes-prediction-eda-preprocessing-models.ipynb
['drop_duplicates']

<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\abdellahaitelouli_sentiment-analysis-using-tenerflow.ipynb
['drop_duplicates']

===================
kaggle_notebooks\abdelruhmanessam_wine-quality.ipynb
['drop_duplicates']

===================
kaggle_notebooks\abdmental01_bank-churn-lightgbm-and-catboost-0-8945.ipynb
['norm_min_max']

===================
kaggle_notebooks\abdmental01_delicious-delights-exploring-online-food.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\abdmental01_embark-on-titanic-a-beginner-s-journey-through-ml.ipynb
['norm_min_max']

===================
kaggle_notebooks\abdmental01_gre-admissions-forecasting-success-with-data.ipynb
['norm_min_max']

===================
kaggle_notebooks\abdmental01_heart-disease-prediction-binary-classification.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\abdmental01_multimodel-isic.ipynb
['norm_log']
Processing 50

===================
kaggle_notebooks\abdmental01_nlp-email-spam-detection-a-beginner-s-guide.ipynb
['drop_duplicates']

===================
kaggle_notebooks\abdmental01_ps4e6-eda-modeling-optuna.ipynb
['norm_min_max']

===================
kaggle_notebooks\abdmental01_sparkle-forecast-predicting-diamond-prices.ipynb
['drop_duplicates', 'IQR', 'norm_min_max']

===================
kaggle_notebooks\abdmental01_unmasking-deception-innovations-in-credit-card.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\abdoashraf90_helthcare-diabetes-with-acuracy-99-using-knn.ipynb
['IQR']

===================
kaggle_notebooks\abdoashraf90_price-of-airline-tickets.ipynb
['IQR']

<unknown>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
kaggle_notebooks\abhaymudgal_intrusion-detection-system.ipynb
['norm_min_max']

===================
kaggle_notebooks\abhishek0032_data-science-toolkit-codes-skills-to-succeed.ipynb
['norm_min_max']

===================
kaggle_notebooks\abhishek0032_titanic-survival-prediction-feature-engineering.ipynb
['norm_min_max']

===================
kaggle_notebooks\abhishekmamidi_time-series-analysis-artificial-neural-networks.ipynb
['norm_min_max']

<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.

Processing 100

===================
kaggle_notebooks\abinanthank_crop-production-rice-and-wheat-abinanthan-k.ipynb
['norm_min_max']

===================
kaggle_notebooks\adachowicz_house-prices-random-forest-regression-analysis.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\adamml_titanic-to-beginner.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\adavydenko_titanic-solution-a-beginner-s-guide-russian.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\adhamtarek147_alzheimer-s-disease-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\adhamtarek147_recurrence-of-thyroid-disease.ipynb
['norm_min_max']

===================
kaggle_notebooks\adhang_telco-customer-churn-prediction-complete-guide.ipynb
['norm_min_max']

===================
kaggle_notebooks\adinishad_looking-for-a-job-as-data-analyst.ipynb
['drop_duplicates']

===================
kaggle_notebooks\aditimulye_imdb-5000-movie-dataset-analysis.ipynb
['drop_duplicates', 'norm_min_max']

<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\adityaecdrid_initial-eda.ipynb
['norm_log']

===================
kaggle_notebooks\adityaecdrid_translate-them-to-tamil-language-external-data.ipynb
['drop_duplicates']

===================
kaggle_notebooks\adrianoavelar_bond-calculaltion-lb-0-82.ipynb
['norm_log']

===================
kaggle_notebooks\adrienmorel97_eda-lightgbm-optuna-1-0644.ipynb
['bin_equal_width_10']

===================
kaggle_notebooks\adrienmorel97_predicting-depression-with-ensemble-learning.ipynb
['norm_log', 'isolationForest']

===================
kaggle_notebooks\advikmaniar_heart-attack-eda-prediction-with-9-model-95.ipynb
['IQR', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'IQR', 'norm_min_max']
Processing 150

===================
kaggle_notebooks\aeryan_spotify-music-analysis.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\agewerc_corporate-credit-rating-forecasting.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\agileteam_3rd-type2-3-2-baseline.ipynb
['norm_min_max']

===================
kaggle_notebooks\agileteam_insurance-starter-tutorial.ipynb
['norm_log']

===================
kaggle_notebooks\agileteam_mock-exam1-type1-1-tutorial.ipynb
['IQR']

===================
kaggle_notebooks\agileteam_py-t1-1-iqr-expected-questions.ipynb
['IQR']

===================
kaggle_notebooks\agileteam_py-t1-4-expected-questions.ipynb
['norm_log']

===================
kaggle_notebooks\agileteam_t1-23-drop-duplicates.ipynb
['drop_duplicates']
Processing 200

===================
kaggle_notebooks\agileteam_t2-3-adult-census-income-tutorial.ipynb
['norm_min_max']

===================
kaggle_notebooks\agileteam_tutorial-t1-python.ipynb
['norm_min_max']

===================
kaggle_notebooks\agodwinp_stacking-house-prices-walkthrough-to-top-5.ipynb
['bin_equal_width_10', 'norm_log']

===================
kaggle_notebooks\ahmadfadliramadhan_k-means-pada-kecepatan-internet-di-indonesia.ipynb
['norm_min_max']

===================
kaggle_notebooks\ahmadihossein_passive-learning.ipynb
['norm_log']

===================
kaggle_notebooks\ahmadpk_eda-on-box-office-data.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\ahmdayman_retail-sales-dataset.ipynb
['IQR']

===================
kaggle_notebooks\ahmedabdulhamid_4-data-transformation-using-pandas-python-for-da.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\ahmedezzatibrahem_student-performance-factors.ipynb
['norm_min_max']
Processing 250

===================
kaggle_notebooks\ahmedraafat666_credit-card-fraud-detection-model.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ahmedterry_restaurants-sales-during-covid-eda.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ahmedtronic_ann-breast-cancer.ipynb
['norm_min_max']

===================
kaggle_notebooks\ahmetcankaraolan_churn-prediction-using-machine-learning.ipynb
['bin_equal_frequency_5', 'IQR', 'bin_equal_frequency_10', 'bin_equal_frequency_10', 'bin_equal_frequency_10']

===================
kaggle_notebooks\ahmetcankaraolan_diabetes-prediction-using-machine-learning.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\aiswaryaramachandran_english-to-hindi-neural-machine-translation.ipynb
['drop_duplicates']

===================
kaggle_notebooks\aitude_ashrae-kfold-lightgbm-without-leak-1-08.ipynb
['norm_log']

===================
kaggle_notebooks\akashkotal_heart-disease-eda-with-7-machine-learning-model.ipynb
['norm_min_max']

===================
kaggle_notebooks\akdagmelih_five-personality-clusters-k-means.ipynb
['norm_min_max']
Processing 300

<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:38: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:49: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:50: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
kaggle_notebooks\akshitmadan_zomato-data-set-analysis-visualization.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\alaasedeeq_house-price-prediction-top-8.ipynb
['norm_log']

===================
kaggle_notebooks\aleaiest_lb-0-945-qwen2-5-32b-gptq.ipynb
['drop_duplicates']

===================
kaggle_notebooks\aleksandrmorozov123_machine-learning-excercises.ipynb
['bin_equal_frequency_10']

===================
kaggle_notebooks\alexioslyon_lgbm-baseline.ipynb
['norm_min_max', 'norm_min_max']
Processing 350

<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\alfathterry_customer-churn-analysis.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\alfathterry_house-price-prediction.ipynb
['IQR', 'norm_min_max', 'IQR', 'norm_min_max']

===================
kaggle_notebooks\alfathterry_telco-customer-churn-analysis.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_min_max', 'norm_min_max']
Processing 400

===================
kaggle_notebooks\alijs1_arc-prize-2024-solution-3rd-place-score-40.ipynb
['drop_duplicates']

===================
kaggle_notebooks\alirezaai_mobile-pricing.ipynb
['IQR']

===================
kaggle_notebooks\alirezahasannejad_data-preprocessing-in-machine-learning.ipynb
['norm_min_max']

===================
kaggle_notebooks\alizgrdede_global-information.ipynb
['norm_min_max']

===================
kaggle_notebooks\alkidiarete_apple-quality-roc-0-97.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\alkidiarete_heartattack-accuracy-92.ipynb
['drop_duplicates', 'IQR', 'norm_min_max']

===================
kaggle_notebooks\allunia_hidden-treasures-in-our-groceries.ipynb
['norm_min_max', 'IQR', 'IQR']

===================
kaggle_notebooks\allunia_m5-sales-uncertainty-prediction.ipynb
['norm_min_max']

<unknown>:6: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\e" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\e"? A raw string is also an option.


===================
kaggle_notebooks\allunia_santander-customer-transaction-eda.ipynb
['bin_equal_frequency_10']

===================
kaggle_notebooks\alnourabdalrahman9_prediction-of-obesity-risk.ipynb
['drop_duplicates']

===================
kaggle_notebooks\alvinai9603_predict-next-point-with-the-imu-data.ipynb
['drop_duplicates']

===================
kaggle_notebooks\alwannabilhanif_prediksi-biaya-asuransi-kesehatan.ipynb
['IQR']

===================
kaggle_notebooks\amalyasser_shhh-i-want-to-sleep.ipynb
['bin_equal_width_2']

===================
kaggle_notebooks\aman9d_data-science-london-scikit.ipynb
['norm_min_max']

===================
kaggle_notebooks\amarpreetsingh_stock-prediction-lstm-using-keras.ipynb
['norm_min_max']
Processing 450

===================
kaggle_notebooks\ambrosm_pss3e11-zoo-of-models.ipynb
['norm_log', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\ambrosm_pss3e23-eda-which-makes-sense.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ambrosm_pss4e4-eda-which-makes-sense.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\ambrosm_tpsapr22-best-model-without-nn.ipynb
['IQR']

===================
kaggle_notebooks\ambrosm_tpsaug22-eda-which-makes-sense.ipynb
['drop_duplicates']

===================
kaggle_notebooks\amerhu_rfm-customer-segmentation.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\amerwafiy_titanic-competition-journey-to-100-accuracy.ipynb
['drop_duplicates', 'drop_duplicates']

<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:38: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:48: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:63: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:77: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:78: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:79: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:88: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:93: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:95: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:96: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:100: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:107: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:109: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:110: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:114: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:115: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:136: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:149: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:157: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:163: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:168: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:169: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:188: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:206: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:213: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:260: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:261: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.


===================
kaggle_notebooks\amirrezaeian_time-series-data-analysis-using-lstm-tutorial.ipynb
['norm_min_max']

===================
kaggle_notebooks\amlanmohanty1_build-web-app-for-heart-disease-with-streamlit.ipynb
['norm_min_max']

===================
kaggle_notebooks\ammarnassanalhajali_riiid-lgbm-bagging2-sakt-0-781.ipynb
['drop_duplicates']

===================
kaggle_notebooks\amrabdelatyfathalla_e-commerce-customer-churn-end-to-end-ml-project.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\analystoleksandra_marketing-analytics-customer-segmentation.ipynb
['IQR']

===================
kaggle_notebooks\analyticaobscura_1st-place-binary-smoke-detector.ipynb
['norm_min_max', 'norm_min_max', 'bin_equal_frequency_5', 'IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\analyticaobscura_mabe-v1-mouse-action-recognition.ipynb
['drop_duplicates']

===================
kaggle_notebooks\analyticaobscura_s5e11-loan-payback-xgb-lgbm-ann.ipynb
['IQR', 'bin_equal_width_10', 'bin_equal_width_10', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\anandaramg_f1-champ-eda-classification-100-accuracy.ipynb
['IQR', 'norm_min_max']
Processing 500

===================
kaggle_notebooks\ananthr1_parkinson-disease-detection-using-xgbooster.ipynb
['norm_min_max']

===================
kaggle_notebooks\anaskad_step-by-step-solving-titanic-problem.ipynb
['norm_min_max']

===================
kaggle_notebooks\anatpeled_spotify-popularity-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\anbarivan_indian-air-quality-analysis-prediction-using-ml.ipynb
['norm_min_max']

===================
kaggle_notebooks\anbarivan_indian-rainfall-analysis-and-prediction.ipynb
['norm_min_max']

<unknown>:23: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\andradaolteanu_wids-datathon-rapids-ensembles-w-b.ipynb
['norm_min_max']

===================
kaggle_notebooks\andreipaulavets_titanic-prediction-0-794-score.ipynb
['norm_log']

===================
kaggle_notebooks\andreshg_timeseries-analysis-a-complete-guide.ipynb
['norm_log', 'norm_min_max']

===================
kaggle_notebooks\angqx95_data-science-workflow-top-2-with-tuning.ipynb
['norm_log']
Processing 550

===================
kaggle_notebooks\aninditapani_will-it-rain-tomorrow.ipynb
['zscore']

===================
kaggle_notebooks\anirbank_callcenteroptimization.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\anlgrbz_data-cleaning-feature-generation-eda-segmentation.ipynb
['drop_duplicates']

===================
kaggle_notebooks\annastasy_brazilian-e-commerce-eda-nlp-ml.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\annastasy_diabetes-prediction-simple-to-advanced.ipynb
['zscore']

===================
kaggle_notebooks\annastasy_mental-health-eda-ensemble.ipynb
['bin_equal_width_10', 'bin_equal_width_10', 'isolationForest']

<unknown>:1: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\annastasy_predicting-students-grades.ipynb
['zscore']

===================
kaggle_notebooks\annastasy_pregnancy-risks-eda-modeling-hypothesis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\annastasy_ps4e8-data-cleaning-and-eda-of-mushrooms.ipynb
['drop_duplicates', 'zscore', 'isolationForest']

===================
kaggle_notebooks\annastasy_sales-forecasting-fighting-data-leakage.ipynb
['norm_min_max', 'norm_min_max', 'zscore', 'norm_min_max']

===================
kaggle_notebooks\anshigupta01_heart-disease-classification.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\anshigupta01_titanic-prediction-top-17.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\anshtanwar_credit-risk-prediction-training-and-eda.ipynb
['norm_min_max']

===================
kaggle_notebooks\anshulranjan2004_worksheet-3b-student-copy.ipynb
['norm_min_max']

===================
kaggle_notebooks\anshuls235_covid19-explained-through-visualizations.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\anubhavgoyal10_customer-churn-prediction-eda-ann.ipynb
['norm_log']

===================
kaggle_notebooks\anubhavgoyal10_laptop-price-prediction.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\anuragraj03_titanic-dataset.ipynb
['drop_duplicates']
Processing 600

===================
kaggle_notebooks\apapiu_regularized-linear-models.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\ar89dsl_predicting-building-damage-from-earthquakes.ipynb
['norm_min_max']

===================
kaggle_notebooks\aradhanapratap_consumer-buying-behavior-analysis.ipynb
['IQR']

<unknown>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:53: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\aremoto_retail-sales-forecast.ipynb
['drop_duplicates']

===================
kaggle_notebooks\arezalo_bank-personal-loan-modeling.ipynb
['norm_min_max']

===================
kaggle_notebooks\arezalo_customer-behaviour-prediction-naive-bayes.ipynb
['norm_min_max']

===================
kaggle_notebooks\arezoodahesh_customer-churn-with-oversampling-techniques.ipynb
['IQR', 'drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\arezoodahesh_heart-failure-prediction-with-ensemble-models.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\arezoodahesh_home-credit-default-risk-part02-boosting-models.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\aritrag_kerascv-starter-notebook-train.ipynb
['drop_duplicates']

===================
kaggle_notebooks\arjunayyangar_asteroidanalysis.ipynb
['norm_min_max']

===================
kaggle_notebooks\arjunjoshua_predicting-fraud-in-financial-payment-services.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\arjunsurendran_using-lstm-on-training-data.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\arootda_the-basic-process-of-classification.ipynb
['bin_equal_frequency_5']

===================
kaggle_notebooks\arootda_titanic-eda-modeling-for-beginners-top3.ipynb
['bin_equal_width_10', 'norm_log']

===================
kaggle_notebooks\artgor_eda-feature-engineering-and-model-interpretation.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\artgor_eda-on-basic-data-and-lgb-in-progress.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']
Processing 650

===================
kaggle_notebooks\artgor_segmentation-in-pytorch-using-convenient-tools.ipynb
['drop_duplicates']

===================
kaggle_notebooks\artgor_where-do-the-robots-drive.ipynb
['drop_duplicates']

===================
kaggle_notebooks\arthurtok_feature-ranking-rfe-random-forest-linear-models.ipynb
['norm_min_max']

===================
kaggle_notebooks\arthurtok_introduction-to-ensembling-stacking-in-python.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\arunjangir245_google-playstore-apps-rating-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\arunklenin_advanced-feature-engg-techniques-beyond-basics.ipynb
['norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_in-depth-analysis-five-anomaly-detection-methods.ipynb
['isolationForest', 'isolationForest']

===================
kaggle_notebooks\arunklenin_ps3e14-blueberry-yield-prediction-challenge.ipynb
['norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_ps3e15-iterative-catboost-imputer-ensemble.ipynb
['norm_min_max', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_ps3e16-eda-feature-engineering-ensemble.ipynb
['norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'drop_duplicates']

===================
kaggle_notebooks\arunklenin_ps3e20-co2-emissions-prediction-regression.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_ps3e23-eda-feature-engineering-ensemble.ipynb
['norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_ps3e24-smoking-cessation-prediction-binary.ipynb
['norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_ps3e25-material-hardness-prediction-with-ml.ipynb
['norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: invalid decimal literal


===================
kaggle_notebooks\arunklenin_ps3e26-cirrhosis-survial-prediction-multiclass.ipynb
['norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_ps4e1-advanced-feature-engineering-ensemble.ipynb
['norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_ps4e3-steel-plate-fault-prediction-multilabel.ipynb
['norm_min_max']
Processing 700

===================
kaggle_notebooks\arunklenin_ps4e4-abalone-age-prediction-regression.ipynb
['norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'isolationForest']

===================
kaggle_notebooks\arunklenin_ps5e3-rainfall-prediction-classification.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\arunklenin_space-titanic-eda-advanced-feature-engineering.ipynb
['norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\arunklenin_wids-datathon-2-metastatic-cancer-diagnosis.ipynb
['norm_min_max', 'norm_min_max', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\ash316_learn-pandas-with-pokemons.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ash316_let-s-play-cricket.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 750

===================
kaggle_notebooks\ashydv_housing-price-prediction-linear-regression.ipynb
['IQR', 'IQR', 'norm_min_max']

===================
kaggle_notebooks\aspillai_obesity-risk-lgb-xgb-cat-92.ipynb
['drop_duplicates']

===================
kaggle_notebooks\atishadhikari_placement-dataanalysis-classification-regression.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\avikumart_classif-hr-why-do-employees-join-the-company.ipynb
['norm_min_max']

<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:59: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.


===================
kaggle_notebooks\avrahamcalev_time-series-models-pamap2-dataset.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\awsaf49_uwmgi-2-5d-infer-pytorch.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\awsaf49_uwmgi-unet-infer-pytorch.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 800

===================
kaggle_notebooks\awsaf49_xgboost-tabular-data-ml-cv-85-lb-787.ipynb
['norm_min_max']

===================
kaggle_notebooks\awwalmalhi_titanic-eda-and-feature-engineering.ipynb
['norm_min_max']

===================
kaggle_notebooks\ayushnitb_cc-fraud-detection-indeptheda-multimod-hyperopt.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\azizozmen_customer-segmentation-cohort-rfm-analysis-k-means.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\azizozmen_heart-failure-predict-8-classification-techniques.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\azizozmen_telco-churn-detailed-eda-8-classification-models.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\baktisiregar_klasifikasi-supervised-learning.ipynb
['norm_min_max']

===================
kaggle_notebooks\balavashan_weather-prediction-ensemble-methods.ipynb
['IQR']

===================
kaggle_notebooks\bandiatindra_telecom-churn-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\barankutluay_titanic-submission.ipynb
['bin_equal_frequency_10']
Processing 850

===================
kaggle_notebooks\batprem_llm-daigt-analyse-edge-cases.ipynb
['drop_duplicates']

===================
kaggle_notebooks\beckpro_previous-2022-xgboost-solution-simple-submission.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\bennyfung_easy-to-use-automl-autogluon-flaml-autosklearn.ipynb
['IQR']

===================
kaggle_notebooks\bennyfung_heart-failure-ensemble-by-voting-auc-85.ipynb
['IQR']

<unknown>:31: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\bennyfung_model-interpretability-xgboost-shap.ipynb
['IQR']

===================
kaggle_notebooks\bennyfung_titanic-random-forest.ipynb
['IQR']

===================
kaggle_notebooks\benroshan_bank-marketing-campaign-predictive-analytics.ipynb
['IQR']

===================
kaggle_notebooks\benroshan_you-re-hired-analysis-on-campus-recruitment-data.ipynb
['IQR']
Processing 900

===================
kaggle_notebooks\bertcarremans_data-preparation-exploration.ipynb
['drop_duplicates']

===================
kaggle_notebooks\bestpredict_location-eda-8eb410.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\bextuychiev_lasso-regression-with-pipelines-tutorial.ipynb
['norm_min_max']

===================
kaggle_notebooks\bhaktipradhan_batch-8-exploratory-data-analysis-tutorial.ipynb
['IQR', 'norm_log']

===================
kaggle_notebooks\bhaktipradhan_exploratory-data-analysis-tutorial.ipynb
['IQR', 'norm_log']

===================
kaggle_notebooks\bhanupratapbiswas_arc-prize-2024-01.ipynb
['drop_duplicates']

<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\bharathraja_statistical-approach-for-predicting-imdb.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\bhatnagardaksh_gradient-descent-from-scratch.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\bhatnagardaksh_pca-and-lda-implementation.ipynb
['norm_min_max']

===================
kaggle_notebooks\bhatnagardaksh_stock-predictions-with-backtesting-arima-and-gru.ipynb
['norm_min_max']

===================
kaggle_notebooks\bhuvanchennoju_women-and-cancer-analysis-and-detection.ipynb
['drop_duplicates', 'drop_duplicates', 'isolationForest', 'drop_duplicates', 'drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\bibanh_lb-0-944-the-art-of-ensemble.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\bibanh_lb-0-947-the-art-of-ensemble-v2.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\bibanh_update-57-4-train-inference-randomforest.ipynb
['norm_min_max', 'drop_duplicates', 'drop_duplicates']
Processing 950

===================
kaggle_notebooks\blurredmachine_titanic-survival-a-complete-guide-for-beginners.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\bminixhofer_5th-place-solution-code.ipynb
['norm_log', 'norm_log']

<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:25: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:60: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:77: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


===================
kaggle_notebooks\bminixhofer_aggregated-features-lightgbm.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\bobber_gpu-rapids-xgb-lgb-cat-nnbase-autoencoder.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\boliu0_monai-3d-cnn-inference.ipynb
['drop_duplicates']

===================
kaggle_notebooks\braquino_convert-to-regression.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\brsdincer_heartbeat-sounds-classification-analysis.ipynb
['norm_min_max']

===================
kaggle_notebooks\bryanb_stock-prices-forecasting-with-lstm.ipynb
['norm_min_max']

===================
kaggle_notebooks\burakergene_titanic-eda-comparing-all-ml-algorithms.ipynb
['bin_equal_width_5']

<unknown>:9: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.

Processing 1000

===================
kaggle_notebooks\caesarlupum_ashrae-start-here-a-gentle-introduction.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\caesarmario_data-to-destiny-titanic-survival-prediction.ipynb
['bin_equal_frequency_5']

===================
kaggle_notebooks\caesarmario_loan-prediction-w-various-ml-models.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\caesarmario_python-magic-big-mart-sales-data-transformed.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'norm_log', 'norm_log']

===================
kaggle_notebooks\carlkirstein_predictive-maintenance-milling-machine-98-6.ipynb
['norm_min_max']

===================
kaggle_notebooks\carlkirstein_unsw-nb15-modelling-97-7.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\carlmcbrideellis_jane-street-eda-of-day-0-and-feature-importance.ipynb
['drop_duplicates']

===================
kaggle_notebooks\carlmcbrideellis_lstm-time-series-stock-price-prediction-fail.ipynb
['norm_min_max']

===================
kaggle_notebooks\casper6290_lung-cancer-prediction-98.ipynb
['drop_duplicates']

===================
kaggle_notebooks\cdabakoglu_time-series-forecasting-arima-lstm-prophet.ipynb
['norm_min_max']
Processing 1050

===================
kaggle_notebooks\cdeotte_deberta-starter-cv-0-930.ipynb
['drop_duplicates']

===================
kaggle_notebooks\cdeotte_ettin-encoder-1b-cv-0-943.ipynb
['drop_duplicates']

===================
kaggle_notebooks\cdeotte_first-place-single-model-cv-1-016-lb-1-016.ipynb
['norm_log']

===================
kaggle_notebooks\cdeotte_gemma2-9b-it-cv-0-945.ipynb
['drop_duplicates']

===================
kaggle_notebooks\cdeotte_modernbert-large-cv-0-938.ipynb
['drop_duplicates']

===================
kaggle_notebooks\cdeotte_recommend-items-purchased-together-0-021.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\cdeotte_tensorflow-gru-starter-0-790.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\cdeotte_xgboost-5000-mutations-200-pdb-files-lb-0-410.ipynb
['drop_duplicates']

===================
kaggle_notebooks\cdeotte_xgboost-starter-0-793.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\chadalee_olympics-data-cleaning-exploration-prediction.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'norm_log', 'norm_log', 'drop_duplicates']

===================
kaggle_notebooks\challenge1a3_convert-to-regression.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\chanakyavivekkapoor_house-price-prediction.ipynb
['norm_log']

===================
kaggle_notebooks\chanchal24_credit-card-fraud-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\chanchal24_diabetes-dataset-eda-prediction-with-7-models.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\chanchal24_heart-disease-eda-prediction-7-models.ipynb
['IQR']

===================
kaggle_notebooks\chanchal24_liver-disease-prediction-using-7-models.ipynb
['drop_duplicates']
Processing 1100

===================
kaggle_notebooks\chandrimad31_flight-passenger-satisfaction-eda-and-prediction.ipynb
['IQR']

===================
kaggle_notebooks\chandrimad31_rainfall-prediction-7-popular-models.ipynb
['IQR']

===================
kaggle_notebooks\chapagain_titanic-solution-a-beginner-s-guide.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\charel_learn-by-example-rnn-lstm-gru-time-series.ipynb
['norm_min_max']

===================
kaggle_notebooks\cheesu_house-prices-1st-approach-to-data-science-process.ipynb
['norm_log']

===================
kaggle_notebooks\chenguangyang_2016-us-presidential-social-media.ipynb
['drop_duplicates']

===================
kaggle_notebooks\chinmayadatt_obesity-risk-prediction-multi-class-0-92160.ipynb
['drop_duplicates']

===================
kaggle_notebooks\chirag9073_airbnb-analysis-visualization-and-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\chleungpolyu_descriptive-statistics-answer.ipynb
['zscore', 'zscore', 'zscore']

===================
kaggle_notebooks\chleungpolyu_mm3425-descriptive-statistics.ipynb
['zscore']

===================
kaggle_notebooks\chocozzz_beginner-challenge-house-prices.ipynb
['norm_log']

===================
kaggle_notebooks\chongzhenjie_ecuador-store-sales-global-forecasting-lightgbm.ipynb
['norm_min_max']

===================
kaggle_notebooks\ChristianDenich_quantile-reg-lr-schedulers-checkpoints.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_min_max']
Processing 1150

===================
kaggle_notebooks\clemchris_pytorch-backfin-convnext-arcface.ipynb
['drop_duplicates']

<unknown>:5: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:25: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
kaggle_notebooks\clkmuhammed_credit-score-classification-part-1-data-cleaning.ipynb
['IQR']

===================
kaggle_notebooks\code1110_jquants-end-to-end-starter.ipynb
['drop_duplicates']

===================
kaggle_notebooks\codingloading_experiment-no-1.ipynb
['drop_duplicates']

===================
kaggle_notebooks\cody11null_squeeze-gbt.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\computervisi_all-best-models.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\computervisi_best-models-spaceship-titanic.ipynb
['norm_log']

===================
kaggle_notebooks\corochann_ashrae-training-lgbm-by-meter-type.ipynb
['norm_log']

===================
kaggle_notebooks\corochann_covid-19-current-situation-in-2021.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\craigmthomas_amp-eda-models.ipynb
['norm_log']

<unknown>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.

Processing 1200

===================
kaggle_notebooks\csanskriti_amazon-sales-data-analysis.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\cv13j0_efficient-prediction-of-smoker-status.ipynb
['drop_duplicates', 'isolationForest']

===================
kaggle_notebooks\daisukelab_cnn-2d-basic-solution-powered-by-fast-ai.ipynb
['norm_min_max']

===================
kaggle_notebooks\damienpark_artificial-neural-network-using-keras.ipynb
['norm_min_max']

===================
kaggle_notebooks\dandrocec_location-eda-with-rusher-features.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\dangnguyen97_0-38006-lightgbm.ipynb
['norm_min_max']

===================
kaggle_notebooks\dangnguyen97_feature-eng-clean-outlier-lgbm-with-optuna.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\danishammar_spam-email-99-accuracy.ipynb
['drop_duplicates']

===================
kaggle_notebooks\danishmubashar_diabetes-hypertension-predict-acc-97.ipynb
['drop_duplicates']

===================
kaggle_notebooks\danishmubashar_heart-disease-prediction-by-danish.ipynb
['IQR', 'IQR', 'norm_min_max']

===================
kaggle_notebooks\danishmubashar_superstore-sales-profit-discount-predict.ipynb
['drop_duplicates']

===================
kaggle_notebooks\danishmubashar_telco-customer-churn-80-accuracy.ipynb
['norm_min_max']

===================
kaggle_notebooks\danishmubashar_titanic-survival-prediction.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\danishmubashar_water-quality-potability.ipynb
['norm_min_max']

===================
kaggle_notebooks\dankok_heart-disease-eda-prediction.ipynb
['IQR']
Processing 1250

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.

Processing 1300

===================
kaggle_notebooks\darkdevil18_0-97362-loan-or-no-loan.ipynb
['drop_duplicates']

===================
kaggle_notebooks\darkdevil18_0-98530-can-you-eat.ipynb
['drop_duplicates']

===================
kaggle_notebooks\darkknight91_predicting-stock-buy-sell-signal-using-cnn.ipynb
['norm_min_max']

===================
kaggle_notebooks\darkside92_detailed-examination-for-house-price-top-10.ipynb
['norm_log']

===================
kaggle_notebooks\darrylljk_data-cleaning.ipynb
['drop_duplicates']

===================
kaggle_notebooks\datafan07_heart-disease-and-some-scikit-learn-magic.ipynb
['isolationForest']

===================
kaggle_notebooks\datafan07_titanic-eda-and-several-modelling-approaches.ipynb
['bin_equal_frequency_5']

===================
kaggle_notebooks\datafan07_what-takes-to-be-a-data-scientist-story-of-robert.ipynb
['drop_duplicates']

===================
kaggle_notebooks\datajmcn_baseline-model-xgboost.ipynb
['bin_equal_frequency_10']

===================
kaggle_notebooks\datatattle_battle-of-ml-classification-models.ipynb
['drop_duplicates']

===================
kaggle_notebooks\datatattle_predicting-loan-default-classification.ipynb
['drop_duplicates', 'drop_duplicates']

<unknown>:16: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\davidcairuz_feature-engineering-lightgbm.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\davidcoxon_titanic-practice-by-davidcoxon.ipynb
['bin_equal_width_5', 'bin_equal_width_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\ddosad_fitness-class-attendance-eda-ml-datacamp.ipynb
['IQR']

===================
kaggle_notebooks\ddosad_ps4e2-visual-eda-lgbm-obesity-risk.ipynb
['drop_duplicates']
Processing 1350

===================
kaggle_notebooks\ddosad_ps4e3-starter-eda-lgbm-steel-plate-defects.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\dejavu23_house-prices-eda-to-ml-beginner.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\dejavu23_house-prices-plotly-pipelines-and-ensembles.ipynb
['norm_log']

===================
kaggle_notebooks\dejavu23_sms-spam-or-ham-beginner.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\dekxrma_no-models-no-algorithms-0-973279.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\derrelldsouza_imdb-sentiment-analysis-eda-ml-lstm-bert.ipynb
['drop_duplicates']

===================
kaggle_notebooks\desalegngeb_student-s-test-performance-eda-and-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\devanshm_zillow-end-to-end-ml-workflow-top-250-0-06416.ipynb
['drop_duplicates']

===================
kaggle_notebooks\devassaxd_student-performance-prediction-complete-analysis.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\devbilalkhan_advanced-predictive-analysis-heart-disease-uci.ipynb
['norm_min_max', 'IQR']

===================
kaggle_notebooks\devbilalkhan_ml-heart-disease-detection-random-forest.ipynb
['IQR']

===================
kaggle_notebooks\devishu14_95-auc-diabetes-prediction-eda.ipynb
['norm_min_max']

===================
kaggle_notebooks\dewminawijekoon_week-03-data-preprocessing.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dgawlik_house-prices-eda.ipynb
['norm_log', 'norm_log']
Processing 1400

===================
kaggle_notebooks\dgluesen_sales-and-workload-in-retail-industry.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\dhamur_machine-learning-in-agriculture.ipynb
['IQR']

===================
kaggle_notebooks\diegoinacio_imdb-genre-based-analysis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dimitreoliveira_model-stacking-feature-engineering-and-eda.ipynb
['drop_duplicates', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\dimitreoliveira_time-series-forecasting-with-lstm-autoencoders.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dimitriosroussis_electricity-price-forecasting-with-dnns-eda.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_min_max', 'norm_min_max']

<unknown>:8: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\diveki_classification-with-nlp-xgboost-and-pipelines.ipynb
['drop_duplicates']

===================
kaggle_notebooks\divyam6969_best-solution-multiclass-obesity-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\divyam6969_easy-solution-91-accuracy-xgboost-optuna.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dkomyagin_predict-future-sales-lightgbm-framework.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dkson1_heart-disease-explainable-catboost-100-recall.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dktalaicha_credit-card-fraud-detection-using-smote-adasyn.ipynb
['norm_log', 'norm_min_max']

===================
kaggle_notebooks\dlaststark_fpe-no-fancy-stuff.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 1450

===================
kaggle_notebooks\dmitrykonovalov_cp3403-cp5634-prac2-v250129a.ipynb
['norm_min_max']

===================
kaggle_notebooks\dmitrykonovalov_fastai-lesson-5-linear-model-and-neural-net.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\dmitrykonovalov_fastai-lesson-6a-random-forests.ipynb
['norm_log']

===================
kaggle_notebooks\dmitryuarov_ps3e20-rwanda-emission-advanced-fe-20-88.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'norm_log']

===================
kaggle_notebooks\docxian_stroke-prediction.ipynb
['bin_equal_frequency_10', 'bin_equal_frequency_10']

===================
kaggle_notebooks\donaldst_stackingandensembling.ipynb
['norm_min_max']

<unknown>:7: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:64: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:92: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:52: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
kaggle_notebooks\donottalk_complete-research-for-icr.ipynb
['drop_duplicates']

===================
kaggle_notebooks\doyouevendata_kiva-exploration-by-a-kiva-lender-and-python-newb.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\dreamingtree_single-nn-with-pairwise-ranking-loss-0-689-lb.ipynb
['norm_log']

===================
kaggle_notebooks\dreamsofbunnies_up-and-running-set-up-eda-and-images-explained.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\dschettler8845_isic-detect-skin-cancer-let-s-learn-together.ipynb
['norm_log']

===================
kaggle_notebooks\dschettler8845_train-sartorius-segmentation-eda-effdet-tf.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dschettler8845_uwm-gi-tract-image-segmentation-eda.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\duncanlarzelere_march-madness-upset-prediction-2024.ipynb
['norm_min_max']

===================
kaggle_notebooks\durgancegaur_a-guide-to-any-classification-problem.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\durgeshrao9993_cycle-case-study-analysis.ipynb
['IQR']

===================
kaggle_notebooks\duygut_airbnb-nyc-price-prediction.ipynb
['norm_log']
Processing 1500

===================
kaggle_notebooks\dvasyukova_a-linear-model-on-apps-and-labels.ipynb
['drop_duplicates']

===================
kaggle_notebooks\dvasyukova_brand-and-model-based-benchmarks.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\dwit392_expanding-on-simple-lgbm.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\ehsanesmaeili_predicting-loan-payback-eda-modeling.ipynb
['norm_log', 'norm_log', 'IQR']

===================
kaggle_notebooks\ehsanesmaeili_road-accident-risk-xbg-lgb-cat.ipynb
['drop_duplicates', 'norm_log', 'norm_log']

===================
kaggle_notebooks\eikedehling_top-10-with-svm-and-linear-regression.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\eisgandar_car-prices-predict-with-ensemble-methods.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\eisgandar_house-prices-predictions-jump-top-1.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\eisgandar_red-wine-quality-eda-classification.ipynb
['norm_min_max']

===================
kaggle_notebooks\eisgandar_sarcastic-headlines-detector-lstm.ipynb
['drop_duplicates']

===================
kaggle_notebooks\eisgandar_smoking-signal-of-body-classification.ipynb
['norm_min_max']

===================
kaggle_notebooks\eisgandar_spam-sms-detector-deep-learning-methods.ipynb
['drop_duplicates']

===================
kaggle_notebooks\eishkaran_shortest-code.ipynb
['norm_log']

===================
kaggle_notebooks\eishkaran_spotify-music-recommendation-system.ipynb
['IQR']

===================
kaggle_notebooks\ekajaya_analysis-dataset-sales-transaction-v-4a-csv.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'IQR', 'IQR', 'IQR', 'IQR', 'IQR', 'drop_duplicates']

===================
kaggle_notebooks\ekrembayar_rfm-analysis-online-retail-ii.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\ekrembayar_store-item-demand-forecasting-with-lgbm.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\ekrembayar_store-sales-ts-forecasting-a-comprehensive-guide.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 1550

<unknown>:18: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


===================
kaggle_notebooks\elikplim_predict-the-burned-area-of-forest-fires.ipynb
['norm_min_max']

===================
kaggle_notebooks\emg826_baseline-for-predicting-cc-strength.ipynb
['norm_min_max']

===================
kaggle_notebooks\emmanueldjegou_house-prices-advanced-regression-techniques.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\equinxx_stock-prediction-gan-twitter-sentiment-analysis.ipynb
['norm_log', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\ericanacletoribeiro_cicids2017-comprehensive-data-processing-for-ml.ipynb
['IQR', 'drop_duplicates']

===================
kaggle_notebooks\erick5_predicting-house-prices-with-machine-learning.ipynb
['norm_log']
Processing 1600

===================
kaggle_notebooks\erikbruin_data-science-bowl-2019-eda-and-baseline.ipynb
['drop_duplicates']

===================
kaggle_notebooks\erikbruin_riiid-comprehensive-eda-baseline.ipynb
['bin_equal_frequency_5']

===================
kaggle_notebooks\esmaascioglu_predicting-good-bad-customers-for-credit-cards.ipynb
['drop_duplicates', 'norm_log']

===================
kaggle_notebooks\evanoleary_gpu-utilization-regression-alibaba.ipynb
['drop_duplicates']

<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\evansussex_rogii-public-score-frontier-lab-visuals.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\fabiendaniel_customer-segmentation.ipynb
['drop_duplicates']

===================
kaggle_notebooks\fahadmehfoooz_heartattack-prediction-with-91-8-accuracy.ipynb
['drop_duplicates']

===================
kaggle_notebooks\fahadmehfoooz_human-activity-recognition-with-neural-networks.ipynb
['norm_min_max']

===================
kaggle_notebooks\fahadmehfoooz_rain-prediction-with-90-65-accuracy.ipynb
['zscore']

===================
kaggle_notebooks\fahadrehman07_salifort-motors-providing-data-driven-suggestions.ipynb
['drop_duplicates', 'IQR']
Processing 1650

===================
kaggle_notebooks\fanvacoolt_tutorial-on-hyperopt.ipynb
['norm_log']

===================
kaggle_notebooks\faraahanwaaar_car-sales-price-prediction.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\fareedalianwar_amazon-delivery.ipynb
['norm_min_max']

<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:36: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.


===================
kaggle_notebooks\faressayah_ibm-hr-analytics-employee-attrition-performance.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\faressayah_lending-club-loan-defaulters-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\faressayah_logistic-regression-for-binary-classification-task.ipynb
['norm_min_max']

===================
kaggle_notebooks\faressayah_predict-employee-absenteeism-from-work.ipynb
['norm_min_max']

===================
kaggle_notebooks\faressayah_stock-market-analysis-prediction-using-lstm.ipynb
['norm_min_max']

===================
kaggle_notebooks\faressayah_support-vector-machine-pca-tutorial-for-beginner.ipynb
['norm_min_max']

===================
kaggle_notebooks\farnazmirfeizi_diabetes-diagnosis-tensorflow-dnn.ipynb
['drop_duplicates']

===================
kaggle_notebooks\faryarmemon_factors-affecting-usa-home-prices.ipynb
['norm_min_max']

===================
kaggle_notebooks\farzadnekouei_customer-segmentation-recommendation-system.ipynb
['drop_duplicates', 'drop_duplicates', 'isolationForest']

===================
kaggle_notebooks\farzadnekouei_gold-price-prediction-lstm-96-accuracy.ipynb
['norm_min_max']

===================
kaggle_notebooks\farzadnekouei_heart-disease-prediction.ipynb
['IQR']

===================
kaggle_notebooks\farzadnekouei_imbalanced-personal-bank-loan-classification.ipynb
['zscore', 'zscore']

===================
kaggle_notebooks\farzadnekouei_polynomial-regression-regularization-assumptions.ipynb
['IQR', 'IQR']
Processing 1700

===================
kaggle_notebooks\fatsaltyfish_convert-to-regression-feature-test.ipynb
['norm_min_max', 'norm_min_max']

<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.


===================
kaggle_notebooks\fccuser_who-does-quincy-larson-talk-to.ipynb
['drop_duplicates']

===================
kaggle_notebooks\feiwenxuan_arc-prize-2024-10-14.ipynb
['drop_duplicates']

===================
kaggle_notebooks\franciscosantos2_is-99-accuracy-good-maybe-not-credit-card-fraud.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\frendyrachman_transjakarta-rfm-segmentation-with-k-means.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\ftmichel_exploratory-study-on-ml-algorithms.ipynb
['norm_log']

===================
kaggle_notebooks\fulrose_kakr-4th-seminar-feature-engineering.ipynb
['norm_min_max']
Processing 1750

===================
kaggle_notebooks\funxexcel_don-t-get-kicked-pipeline-improved.ipynb
['norm_min_max']

===================
kaggle_notebooks\gaborfodor_from-eda-to-the-top-lb-0-367.ipynb
['norm_log']

===================
kaggle_notebooks\gabrielsober_diabetes-eda-prediction.ipynb
['drop_duplicates', 'norm_log', 'drop_duplicates']

===================
kaggle_notebooks\gaganmaahi224_9-clustering-techniques-for-customer-segmentation.ipynb
['IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\gallo33henrique_house-price-ml-regression-lgbm.ipynb
['IQR']

===================
kaggle_notebooks\gcdatkin_top-10-house-price-regression-competition-nb.ipynb
['norm_log']

<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\georgyzubkov_water-quality-exploratory-data-analysis-ml-rf.ipynb
['norm_min_max']

===================
kaggle_notebooks\getanmolgupta01_bank-churn-eda-catboost-lgbm-xgboost.ipynb
['drop_duplicates']
Processing 1800

===================
kaggle_notebooks\getanmolgupta01_regression-model.ipynb
['norm_log']

===================
kaggle_notebooks\getanmolgupta01_unsw-nb15-cybersecurity-threat-detection-ann.ipynb
['drop_duplicates', 'norm_log', 'norm_log']

===================
kaggle_notebooks\ghazouanihaythem_lstm-for-time-series-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\girmdshinsei_for-japanese-beginner-with-wrmsse-in-lgbm.ipynb
['drop_duplicates']

===================
kaggle_notebooks\gobyeonggeon_preprocess-visualize-spatial-data-eda-xgb.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\goyaladi_house-prices-regression-analysis-eda.ipynb
['IQR', 'IQR', 'norm_min_max']

===================
kaggle_notebooks\goyalshalini93_car-price-prediction-linear-regression-rfe.ipynb
['norm_min_max']

<unknown>:12: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.

Processing 1850

===================
kaggle_notebooks\guanlintao_0-814-optuna-xgb-space-titanic.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'isolationForest']

===================
kaggle_notebooks\guanlintao_eda-autogluon-hair-loss-data.ipynb
['norm_min_max']

===================
kaggle_notebooks\guecoraph_titanic-data-cleaning-model-fitting.ipynb
['norm_min_max']

===================
kaggle_notebooks\guesejustin_91-genetic-algorithms-explained-using-geap.ipynb
['norm_min_max']

===================
kaggle_notebooks\gunesevitan_titanic-advanced-feature-engineering-tutorial.ipynb
['bin_equal_frequency_10']

===================
kaggle_notebooks\guslovesmath_tesla-stock-forecasting-multi-step-stacked-lstm.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\habedi_ubaar-starter-kernel.ipynb
['norm_min_max']

<unknown>:25: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:61: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
kaggle_notebooks\hadeux_kor-eng-eda-ensemble-model.ipynb
['norm_min_max']
Processing 1900

===================
kaggle_notebooks\hadeux_titanic-survivor-predict-eda-lightgbm-kor-eng.ipynb
['drop_duplicates']

===================
kaggle_notebooks\hamzaben_eda-feature-eng-and-model-blending-top-20.ipynb
['IQR', 'IQR', 'norm_min_max']

===================
kaggle_notebooks\hamzaben_employee-churn-model-w-strategic-retention-plan.ipynb
['norm_min_max']

===================
kaggle_notebooks\haneenhossam_airline-passengers-using-lstm.ipynb
['norm_min_max']

===================
kaggle_notebooks\hardikgarg03_bank-churn-random-forest-xgboost-and-lightbgm.ipynb
['winsorize', 'winsorize']

===================
kaggle_notebooks\hardikgarg03_house-price-random-forest-linear-regression.ipynb
['norm_log']

===================
kaggle_notebooks\hardikgarg03_obesity-risk-random-forest-xgboost-96-2-accuracy.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\hardikgarg03_smoker-status-signal-80-accuracy.ipynb
['norm_min_max']

===================
kaggle_notebooks\hardikgarg03_software-defects-using-randomforest-xgboost-lgbm.ipynb
['norm_min_max']

===================
kaggle_notebooks\hardikgarg03_table-reservation-xgb-random-forest-and-logistic.ipynb
['winsorize']

===================
kaggle_notebooks\hardikgarg03_titanic-using-random-forest.ipynb
['winsorize', 'winsorize']

===================
kaggle_notebooks\harishnandhakumar_ericsson-cord-19-challenge-task-9.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 1950

===================
kaggle_notebooks\harshkothari21_100-accurate-results-with-eda-all-ml-models.ipynb
['norm_min_max', 'norm_min_max']

<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.


===================
kaggle_notebooks\harvindarjunrai_predicting-house-prices-v1.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\hasanbasriakcay_tps-feb22-eda-ignore-important-cols.ipynb
['drop_duplicates']

===================
kaggle_notebooks\hasanburakavci_titanic-eda-and-classification-top-5.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\hasibalmuzdadid_anime-ratings-analysis-recommender-system.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\hasibalmuzdadid_brain-stroke-analysis-accuracy-96-03.ipynb
['drop_duplicates']

===================
kaggle_notebooks\hasibalmuzdadid_fire-alarm-triggering-analysis-accuracy-100.ipynb
['drop_duplicates']

===================
kaggle_notebooks\hasibalmuzdadid_lung-cancer-analysis-accuracy-96-4.ipynb
['drop_duplicates']

===================
kaggle_notebooks\hely333_eda-regression.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\hely333_explore-avocados-from-all-sides.ipynb
['drop_duplicates']

===================
kaggle_notebooks\hely333_what-is-the-secret-of-academic-success.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 2000

<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\hetulmehta_classification-of-websites.ipynb
['drop_duplicates']

===================
kaggle_notebooks\heyrobin_house-price-prediction-beginner-s-notebook.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\hieyong0302_categoricalfeatureencodingchallenge-jeong.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\himanshunakrani_bitcoin-price-prediction-updated-daily.ipynb
['norm_min_max']

===================
kaggle_notebooks\hjd810_keras-lgbm-aug-feature-eng-sampling-prediction.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\holybread_mm3425-lecture-4-5.ipynb
['zscore']

===================
kaggle_notebooks\holybread_mm3425-tutorial-5-answers.ipynb
['zscore']
Processing 2050

===================
kaggle_notebooks\hoshi7_goodreads-analysis-and-recommending-books.ipynb
['norm_min_max']

<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\hubert101_0-960-phrases-are-keys.ipynb
['drop_duplicates']

===================
kaggle_notebooks\hyunseokc_detecting-early-alzheimer-s.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\hzning_top-3-0-97-introvert-vs-extrovert-eda.ipynb
['drop_duplicates']

===================
kaggle_notebooks\iabhishekofficial_prediction-on-hospital-readmission.ipynb
['drop_duplicates', 'norm_log', 'norm_log', 'drop_duplicates', 'zscore']

===================
kaggle_notebooks\iamleonie_intro-to-time-series-forecasting.ipynb
['norm_log']

===================
kaggle_notebooks\ibtesama_getting-started-with-a-movie-recommendation-system.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ichigoe_en-jp-deck-image-renderer-visual-your-deck-diff.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ididur_nn-based-recommender-engine.ipynb
['drop_duplicates', 'drop_duplicates']
Processing 2100

===================
kaggle_notebooks\ihabsherbiny_lung-cancer-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ikjotsingh221_obesityt2.ipynb
['zscore', 'zscore']

===================
kaggle_notebooks\imaadmahmood_birdclef-2026-perch-v2-protossm-0-925.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\imoore_intro-to-exploratory-data-analysis-eda-in-python.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\imoore_titanic-the-only-notebook-you-need-to-see.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\imtkaggleteam_heart-disease-prediction-ensemble.ipynb
['norm_min_max']

===================
kaggle_notebooks\introverstein_build-neural-network-for-tabular-data-pytorch.ipynb
['drop_duplicates']
Processing 2150

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\iqbalsyahakbar_ps3e12-simple-eda-fe-and-model-for-beginners.ipynb
['drop_duplicates']

===================
kaggle_notebooks\iqbalsyahakbar_ps3e20-time-series-for-beginners.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\iqbalsyahakbar_ps3e25-mohs-hardness-regression-for-beginners.ipynb
['drop_duplicates']

===================
kaggle_notebooks\iqmansingh_bank-churn-kfold-lgbm-cat-xgb-ensemble.ipynb
['drop_duplicates']

===================
kaggle_notebooks\isaienkov_lightgbm-fe-1-19.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\isaienkov_riiid-answer-correctness-prediction-eda-modeling.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ishandutta_petfinder-data-augmentations-master-notebook.ipynb
['norm_min_max']

===================
kaggle_notebooks\ishivinal_tweet-emotions-analysis-using-lstm-glove-roberta.ipynb
['drop_duplicates']

===================
kaggle_notebooks\issacchanjj_anti-money-laundering-detection-with-gnn.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

<unknown>:3: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\ivankhrulenko_pop-rap-or-heavy-metal-lyrics-classifier.ipynb
['drop_duplicates']
Processing 2200

===================
kaggle_notebooks\ivannatarov_amazon-s-books-eda-plotly-hypothesis-test.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\izzettunc_introduction-to-time-series-clustering.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\jacopoferretti_gate-e-learning-analysis-on-customers-conversion.ipynb
['norm_min_max']

===================
kaggle_notebooks\jacopoferretti_tiktok-videos-google-advanced-data-analytics.ipynb
['IQR', 'IQR', 'IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\jaderlima_aplica-es-em-nlp-aula-03.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jagangupta_understanding-approval-donorschoose-eda-fe-eli5.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\jainaru_eda-visualization-and-100-prediction-xgboost.ipynb
['norm_min_max', 'drop_duplicates']

===================
kaggle_notebooks\jakobzerbs_foodprint-dataset.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\janiobachmann_patient-charges-clustering-and-regression.ipynb
['norm_log']

===================
kaggle_notebooks\janiobachmann_s-p-500-time-series-forecasting-with-prophet.ipynb
['norm_log', 'norm_log', 'norm_log']

<unknown>:11: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\jannesklaas_structured-data-code.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\jasonduncanwilson_urination-in-nyc-and-other-fun-exploration.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jaswanthhbadvelu_comparison-of-ml-models-with-rnn.ipynb
['norm_min_max']

===================
kaggle_notebooks\javiermartnezmartnez_concurso-facebook-javimm.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\javiervallejos_titanic-simple-decision-tree-model-score-top-3.ipynb
['bin_equal_width_5']
Processing 2250

===================
kaggle_notebooks\javigallego_top-3-fe-tuning-ensembling.ipynb
['bin_equal_frequency_10', 'bin_equal_frequency_10']

===================
kaggle_notebooks\jaytonde_deepseekmath-7b-lb-0-944.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jdelamorena_recall-97-by-using-undersampling-neural-network.ipynb
['norm_log']

===================
kaggle_notebooks\jeeelsheikh_diabetes-prediction-eda-ml.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\jefferyakuffo_bigmart-sales-forecasting-without-historical-sale.ipynb
['IQR']

===================
kaggle_notebooks\jellyfish0821_wine-reviews-machine-learning-pipeline.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jennifercrockett_marketing-analytics-eda-task-final.ipynb
['drop_duplicates']
Processing 2300

===================
kaggle_notebooks\jerifate_future-sales-time-series-visualization.ipynb
['drop_duplicates', 'norm_log', 'norm_log']

<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\jerryhkl_data-cleansing.ipynb
['drop_duplicates', 'zscore']

===================
kaggle_notebooks\jessemostipak_animal-crossing-villager-analysis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jesucristo_1-house-prices-solution-top-1.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\jesucristo_1-smart-robots-most-complete-notebook.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jhoward_how-random-forests-really-work.ipynb
['norm_log']

===================
kaggle_notebooks\jhoward_linear-model-and-neural-net-from-scratch.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\jhoward_why-you-should-use-a-framework.ipynb
['norm_log']

===================
kaggle_notebooks\jhskaggle_data-preprocessing.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jianlizhou_customer-segmentation-by-rfm-model-and-k-means.ipynb
['drop_duplicates', 'zscore', 'zscore']

===================
kaggle_notebooks\jiashenliu_different-classifier-showcase-and-question.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\jieyima_income-classification-model.ipynb
['bin_equal_width_10', 'bin_equal_width_10']

===================
kaggle_notebooks\jillanisofttech_sleep-health-and-lifestyle-predication-with-94-ac.ipynb
['IQR']
Processing 2350

===================
kaggle_notebooks\jinghanna_ericsson-cord-19-challenge-task-10.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\jingzongwang_usa-car-accidents-severity-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jiweiliu_rapids-cudf-feature-engineering-xgb.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jlfritz_data-analysis-training.ipynb
['norm_min_max']

===================
kaggle_notebooks\jocelyndumlao_cardiovascular-health-analysis.ipynb
['norm_min_max']

===================
kaggle_notebooks\jocelyndumlao_neural-network-analysis.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\joshuajhchoi_titanic-tutorial-for-absolute-beginners-kr-en.ipynb
['bin_equal_frequency_10', 'bin_equal_frequency_10']

===================
kaggle_notebooks\joshuajhchoi_titanic-tutorial-for-beginners-2020.ipynb
['bin_equal_frequency_10', 'bin_equal_frequency_10']

===================
kaggle_notebooks\joshuaswords_awesome-eda-2021-happiness-population.ipynb
['bin_equal_frequency_10', 'bin_equal_frequency_10', 'drop_duplicates']

===================
kaggle_notebooks\joshuaswords_does-hosting-the-olympics-improve-performance.ipynb
['norm_log']

===================
kaggle_notebooks\joshuaswords_netflix-data-visualization.ipynb
['drop_duplicates']
Processing 2400

===================
kaggle_notebooks\joshuaswords_time-series-anomaly-detection.ipynb
['isolationForest', 'isolationForest']

<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\jphoon_bitcoin-time-series-prediction-with-lstm.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\jraddick_2-get-events-and-handedness.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\jrw2200_smart-pricing-with-xgb-rfr-interpretations.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\juanluisrosa_anime-reviews.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\juebrauer_predicting-pump-failure-with-a-mlp.ipynb
['norm_min_max']

===================
kaggle_notebooks\julianguo_fork-of-riiid-lgbm-bagging2-1-471152.ipynb
['drop_duplicates']

===================
kaggle_notebooks\juliencs_a-study-on-regression-applied-to-the-ames-dataset.ipynb
['norm_log']

<unknown>:23: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
kaggle_notebooks\jumpingmandt_sleep-data-study.ipynb
['norm_min_max']

===================
kaggle_notebooks\junhyeok99_titanic-tutorial-for-beginner.ipynb
['norm_min_max', 'norm_log']

===================
kaggle_notebooks\justicevil_short-code-with-detailed-eda-prediction-96.ipynb
['norm_min_max']

===================
kaggle_notebooks\juwonoindo_book-recommendation-system-cbf-and-cf.ipynb
['drop_duplicates']

===================
kaggle_notebooks\jwilda3_classifying-fraud-by-decision-trees.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kabure_almost-complete-feature-engineering-ieee-data.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_log', 'norm_log']

===================
kaggle_notebooks\kabure_credit-card-fraud-prediction-rf-smote.ipynb
['norm_log']
Processing 2450

===================
kaggle_notebooks\kabure_extensive-eda-and-modeling-xgb-hyperopt.ipynb
['norm_log', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\kabure_extensive-usa-youtube-eda.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\kabure_insightful-eda-churn-customers-models-pipeline.ipynb
['norm_log']

===================
kaggle_notebooks\kabure_insightful-eda-modeling-lgbm-hyperopt.ipynb
['norm_min_max', 'drop_duplicates']

===================
kaggle_notebooks\kabure_kickstarter-projects-eda-stat-tests-pipeline.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\kabure_predicting-credit-risk-model-pipeline.ipynb
['norm_log']

<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\kacperrabczewski_horse-health-a-beginner-friendly-guide.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kacperrabczewski_rwanda-co2-step-by-step-guide.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\kadirduran_employee-churn-prediction.ipynb
['drop_duplicates', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\kadirduran_fraud-detection-with-deployment.ipynb
['drop_duplicates', 'zscore', 'IQR', 'IQR']

===================
kaggle_notebooks\kaggleguyreall_predict-time-series-data-with-lstm-autoencoder.ipynb
['norm_min_max']

===================
kaggle_notebooks\kaggleguyreall_predicting-time-series-data-with-tcn.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\kagleo123_student-perform-in-exam-eda-ml-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\kagleo123_titanic-eda-machine-deep-learning.ipynb
['norm_min_max']

===================
kaggle_notebooks\kairosart_machine-learning-for-mental-health-1.ipynb
['norm_min_max']

===================
kaggle_notebooks\kamalchhirang_eda-feature-engineering-lgb-xgb-cat.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\kaniya_covid-global-forecast-sir-xgboost.ipynb
['drop_duplicates']
Processing 2500

===================
kaggle_notebooks\kanuriviveknag_road-accidents-severity-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\kapturovalexander_kapturov-s-solution-of-ps-s3e24.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kapturovalexander_kapturov-s-solution-of-ps-s4e2.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kapturovalexander_kapturov-s-solution-of-ps-s4e3.ipynb
['drop_duplicates']

===================
kaggle_notebooks\karanprinja_neural-network-classification.ipynb
['drop_duplicates', 'IQR', 'norm_log', 'drop_duplicates', 'norm_log']

===================
kaggle_notebooks\karansarpal_insurance-data-science-project-ks.ipynb
['norm_min_max']

===================
kaggle_notebooks\kareem3egm_learn-machine-learning-faster-1.ipynb
['norm_min_max']

===================
kaggle_notebooks\kareemellithy_diabeties-prediction-eda-svm.ipynb
['drop_duplicates']

===================
kaggle_notebooks\karell_xgb-baseline-advanced-feature-engineering.ipynb
['drop_duplicates']

===================
kaggle_notebooks\karelrv_nyct-from-a-to-z-with-xgboost-tutorial.ipynb
['norm_log']

===================
kaggle_notebooks\kartik2112_fraud-detection-on-paysim-dataset.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\kartikdetroja_pandas-tutorial.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kartikpradyumna92_housing-price-prediction.ipynb
['norm_min_max', 'norm_min_max']
Processing 2550

===================
kaggle_notebooks\kashnitsky_topic-6-feature-engineering-and-feature-selection.ipynb
['norm_min_max', 'norm_min_max']

<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\kaushal2896_ashrae-eda-fe-lightgbm-1-12.ipynb
['drop_duplicates', 'norm_log', 'norm_log']

===================
kaggle_notebooks\kcs93023_2019-ml-month-2nd-baseline.ipynb
['norm_log']

===================
kaggle_notebooks\kdsharma_banking-churn-analysis-modeling.ipynb
['norm_log']

===================
kaggle_notebooks\kdsharma_spaceship-titanic-competition-end-to-end-project.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\kdsharma_titanic-machine-learning-from-disaster.ipynb
['norm_log', 'norm_log', 'norm_log', 'drop_duplicates']

===================
kaggle_notebooks\kellibelcher_jpx-stock-market-analysis-prediction-with-lgbm.ipynb
['drop_duplicates']
Processing 2600

===================
kaggle_notebooks\kenjee_titanic-project-example.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\keremkarayaz_obesity-analysis-and-accuracy-96.ipynb
['IQR']

===================
kaggle_notebooks\kerta27_mabe-lgb-xgb-catboost.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\keshavramaiah_hotel-recommender.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\khangtran94vn_classification-of-insurance-cross-selling.ipynb
['norm_min_max']

===================
kaggle_notebooks\khashayarrahimi94_votingclassifier-ensemble-with-just-5-feature.ipynb
['norm_min_max']

===================
kaggle_notebooks\khashayarrahimi94_what-not-to-do-in-titanic-feature-engineering.ipynb
['norm_min_max']

===================
kaggle_notebooks\khoongweihao_data-science-bowl-2019-regression-to-convert-lb.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\khoongweihao_efficientnets-quantile-regression-inference.ipynb
['drop_duplicates']

===================
kaggle_notebooks\khozzy_kobe-shots-show-me-your-best-model.ipynb
['norm_min_max', 'norm_min_max']
Processing 2650

===================
kaggle_notebooks\kimtaehun_nice-eda-and-quick-xgb-baseline-in-2minutes.ipynb
['norm_min_max']

<unknown>:129: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.


===================
kaggle_notebooks\kimtaehun_simple-eda-and-xgb-baseline-you-can-read-in-3min.ipynb
['norm_min_max']

===================
kaggle_notebooks\kingajohnsjoe_covid-19-knowledge-graph-with-bert-weighted-edges.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kirollosashraf_phishing-email-detection-using-deep-learning.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kishanvavdara_map-deepseekmath-7b-it-tpu-train-bf16.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kk0105_cic-ids2017-intrusion-detection.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\kmader_attention-on-pretrained-vgg16-for-bone-age.ipynb
['bin_equal_width_10']

===================
kaggle_notebooks\kmader_deep-learning-skin-lesion-classification.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kmader_inceptionv3-for-retinopathy-gpu-hr.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kmkarakaya_a-baseline-neural-network-model-with-keras-2.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\knowledgegrappler_a-simple-nn-solution-with-keras-0-48611-pl.ipynb
['norm_log', 'norm_min_max']

===================
kaggle_notebooks\kobeerose_bitcoin-trading-bot.ipynb
['norm_min_max']
Processing 2700

<unknown>:54: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:55: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:56: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:57: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:46: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:48: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:49: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\koheimuramatsu_iot-temperature-forecasting.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\kononenko_lgbm-x2-nn-fusion.ipynb
['norm_min_max']

===================
kaggle_notebooks\konradb_ts-0-the-basics.ipynb
['norm_log']

===================
kaggle_notebooks\konstantinmasich_titanic-0-82-0-83.ipynb
['bin_equal_frequency_5']

===================
kaggle_notebooks\kopfstein_house-price-prediction-using-linear-regression.ipynb
['norm_log']

===================
kaggle_notebooks\korfanakis_titanic-a-beginner-friendly-approach-to-top-3.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\kospintr_health-stacked-hgbc-catb-xgb-lgbm-baseline.ipynb
['norm_min_max', 'drop_duplicates']

===================
kaggle_notebooks\krishnaraj30_xgboost-loan-defaulters-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\ksevta_ps4e2-xgb-lgbm-0-92.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kuchhbhi_pandas-zero-to-hero.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\kuldeeprathoree_0-92196-multi-class-obesity.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kulkarnivishwanath_ashrae-great-energy-predictor-iii-eda-model.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\kushagranull_crop-yield-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\kushal1506_titanic-81-1-leader-board-score-guaranteed.ipynb
['bin_equal_frequency_10']
Processing 2750

===================
kaggle_notebooks\kyakovlev_1st-place-solution-part-1-hands-on-data.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kyakovlev_ieee-basic-fe-part-1.ipynb
['drop_duplicates']

===================
kaggle_notebooks\kyakovlev_ieee-fe-for-local-test.ipynb
['drop_duplicates', 'norm_log', 'norm_log']

===================
kaggle_notebooks\kyakovlev_ieee-fe-with-some-eda.ipynb
['drop_duplicates', 'norm_log', 'norm_log']

===================
kaggle_notebooks\kyakovlev_ieee-gb-2-make-amount-useful-again.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\kyakovlev_m5-simple-fe.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\lava18_all-that-you-need-to-know-about-the-android-market.ipynb
['drop_duplicates']

===================
kaggle_notebooks\lavanyashukla01_how-i-made-top-0-3-on-a-kaggle-competition.ipynb
['norm_log']

<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\lavanyashukla01_picking-the-best-model-a-whirlwind-tour-of-model.ipynb
['norm_log']

===================
kaggle_notebooks\ldfreeman3_a-data-science-framework-to-achieve-99-accuracy.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\leekahwin_text-classification-using-n-gram-0-8-f1.ipynb
['drop_duplicates']

===================
kaggle_notebooks\leilahasan_parkinson-prediction-classifiers-neuralnetwork.ipynb
['IQR', 'norm_min_max']
Processing 2800

===================
kaggle_notebooks\leolu1998_lgbm-tabnet-nn-no-leaks-stratifiedgroupkfold.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\levantaokkz_arc-2024-v1-from-arc2020.ipynb
['drop_duplicates']

===================
kaggle_notebooks\liliyak_job-recommendation-analysis.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\limyenwee_stacked-ensemble-models-top-3-on-leaderboard.ipynb
['norm_log', 'norm_log', 'isolationForest']

===================
kaggle_notebooks\linxinzhe_tensorflow-deep-learning-to-solve-titanic.ipynb
['norm_min_max']

<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


===================
kaggle_notebooks\liyenhsu_titanic-neural-network.ipynb
['bin_equal_frequency_5']

===================
kaggle_notebooks\locpham2001_fraud-detection-using-randomforest-smote-tuning.ipynb
['drop_duplicates']

===================
kaggle_notebooks\lovroselic_houseprices-ls.ipynb
['bin_equal_width_2', 'norm_log']

===================
kaggle_notebooks\luaubrey_uspppm-inference.ipynb
['norm_min_max']

===================
kaggle_notebooks\lucabasa_the-data-science-book-of-love.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 2850

===================
kaggle_notebooks\lucamassaron_steel-plate-eda-xgboost-is-all-you-need.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\lucidlenn_data-analysis-and-classification-using-xgboost.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\ludovicocuoghi_detecting-bullying-tweets-pytorch-lstm-bert.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ludovicocuoghi_twitter-sentiment-analysis-with-bert-vs-roberta.ipynb
['drop_duplicates']

===================
kaggle_notebooks\luficergfree_simplicity-is-the-key-to-success.ipynb
['drop_duplicates', 'norm_min_max']

<unknown>:19: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.


===================
kaggle_notebooks\luisresendiz_digit-recognizer-rf-nn.ipynb
['norm_min_max']

===================
kaggle_notebooks\lukhilaksh_customer-behavior-92-prediction-beats.ipynb
['norm_min_max']

===================
kaggle_notebooks\lytvyiv1_bpm-predictions-with-stacking-lgbm-xgb-mlp.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\lytvyiv1_polymer-property-prediction-with-xgb-svr-lgbm.ipynb
['norm_min_max']

===================
kaggle_notebooks\lytvyiv1_single-lgbm-with-feature-engineering.ipynb
['norm_log']

===================
kaggle_notebooks\mahdavi1202_mobile-price-calssification-project.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\maheshdadhich_strength-of-visualization-python-visuals-tutorial.ipynb
['norm_log']

===================
kaggle_notebooks\mahmoudelfahl_cohort-analysis-customer-segmentation-with-rfm.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mahmoudlimam_chronic-kidney-disease-clustering-and-prediction.ipynb
['norm_min_max']
Processing 2900

===================
kaggle_notebooks\mahnazarjmand_mobile-price-prediction-dts-rf-svm.ipynb
['norm_min_max']

===================
kaggle_notebooks\mammadabbasli_bank-marketing-campaign.ipynb
['norm_min_max']

===================
kaggle_notebooks\manifoldix_inceptionv3-for-retinopathy-gpu-hr.ipynb
['drop_duplicates']

===================
kaggle_notebooks\manishkumar7432698_pse17-feature-engineering-tuning-optuna.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\manthanx_employee-s-future-eda-precision-0-97.ipynb
['drop_duplicates']

===================
kaggle_notebooks\marcinrutecki_best-techniques-and-metrics-for-imbalanced-dataset.ipynb
['drop_duplicates']

<unknown>:5: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


===================
kaggle_notebooks\marcinrutecki_clustering-methods-comprehensive-study.ipynb
['isolationForest']

===================
kaggle_notebooks\marcinrutecki_one-hot-encoding-everything-you-need-to-know.ipynb
['drop_duplicates']

===================
kaggle_notebooks\marcinrutecki_outlier-detection-methods.ipynb
['isolationForest']

===================
kaggle_notebooks\marcinrutecki_smote-and-tomek-links-for-imbalanced-data.ipynb
['drop_duplicates']

===================
kaggle_notebooks\marcinrutecki_stacking-classifier-ensemble-for-great-results.ipynb
['drop_duplicates']

===================
kaggle_notebooks\marcinrutecki_telco-churn-eda-model-voting-boosting.ipynb
['drop_duplicates']

===================
kaggle_notebooks\marcovasquez_machine-learning-on-board-titanic-17-algothim.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\margaritakr_project-3-booking-com-margaritak.ipynb
['norm_min_max', 'drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\mariapushkareva_medical-insurance-cost-with-linear-regression.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 2950

===================
kaggle_notebooks\mariushinsberger_xgboost-on-obesity-risk.ipynb
['drop_duplicates']

===================
kaggle_notebooks\marto24_bankruptcy-detection.ipynb
['norm_log']

<unknown>:8: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\marto24_beginners-prediction-top3.ipynb
['norm_log']

===================
kaggle_notebooks\martynovandrey_eda-and-lgb-cat-xgb.ipynb
['norm_log']

===================
kaggle_notebooks\marynaborovska_birdclef-26-two-pass-ssm-advanced-pp.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\massquantity_all-you-need-is-pca-lb-0-11421-top-4.ipynb
['bin_equal_frequency_10', 'norm_log']

===================
kaggle_notebooks\massquantity_end-to-end-process-for-titanic-problem.ipynb
['bin_equal_width_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\mastmustu_saving-fraud-loss-using-machine-learning.ipynb
['norm_log']

===================
kaggle_notebooks\masumrumi_a-detailed-regression-guide-with-house-pricing.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\masumrumi_a-pyspark-tutorial-with-titanic.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mateuszk013_playground-series-s3e20-co2-emission-in-rwanda.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\mateuszk013_playground-series-s3e25-mohs-hardness.ipynb
['norm_min_max']

===================
kaggle_notebooks\mathchi_churn-problem-for-bank-customer.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\mathchi_credit-risk-evaluation.ipynb
['IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\mathchi_diagnostic-a-patient-has-diabetes.ipynb
['IQR']

===================
kaggle_notebooks\matinmahmoudi_complete-guide-to-a-b-testing-a-to-z.ipynb
['drop_duplicates', 'IQR']
Processing 3000

===================
kaggle_notebooks\matinmahmoudi_complete-guide-to-data-quality-part-1.ipynb
['IQR', 'IQR', 'isolationForest']

===================
kaggle_notebooks\matinmahmoudi_complete-guide-to-data-transformation-a-to-z.ipynb
['norm_min_max']

===================
kaggle_notebooks\matinmahmoudi_loan-eda-project-quick-start-for-beginners.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\matinmahmoudi_pandas-mastery-series-ultimate-challenge.ipynb
['IQR', 'isolationForest']

===================
kaggle_notebooks\matthewmcnulty_bank-account-fraud.ipynb
['norm_min_max']

===================
kaggle_notebooks\mattiaangeli_mabe-extra-trees-gpu.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\mattiaangeli_maybe-remix-fps-corrected-v2.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\mattiaangeli_maybe-remix-fps-corrected.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\maunish_osic-super-cool-eda-and-pytorch-baseline.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mauricioasperti_automobile-customer-segmentation-classification.ipynb
['drop_duplicates', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\mauriciofigueiredo_introdu-o-ao-aprendizado-de-m-quina.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\mauriciovellasquez_esg-risk-analysis-insights-from-s-p-500-companies.ipynb
['zscore']

<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\maverickss26_map-charting-student-math-misunderstanding-v1.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\maverickss26_regression-modellng-using-insurance-dataset.ipynb
['drop_duplicates']

===================
kaggle_notebooks\maxmiao_the-importance-of-a-good-title.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mayangrui_lgbm-ffnn.ipynb
['norm_min_max', 'norm_min_max']
Processing 3050

===================
kaggle_notebooks\mayukh18_dinov3-no-tta-postprocess.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mdismielhossenabir_preprocessing-and-prediction-air-quality.ipynb
['norm_min_max']

===================
kaggle_notebooks\mdmahmudferdous_titanic-survivor-prediction-0-804-top-8.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\mdsultanulislamovi_comprehensive-analysis-student-stress-datasets.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\medali1992_hms-resnet1d-gru-train.ipynb
['drop_duplicates']

===================
kaggle_notebooks\meeraajayakumar_spotify-user-behavior-analysis.ipynb
['norm_min_max']

===================
kaggle_notebooks\meetnagadia_bitcoin-price-prediction-using-lstm.ipynb
['norm_min_max']

===================
kaggle_notebooks\mehakiftikhar_iris-multiclass-classification-problem.ipynb
['norm_min_max', 'IQR']

===================
kaggle_notebooks\mehakiftikhar_ml-for-email-spam-detection-nlp-classification.ipynb
['drop_duplicates']

<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:71: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:242: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\mehmetisik_bankas-yar-ma.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mehmetisik_data-science-salary-eda-graphics-prediction.ipynb
['norm_min_max']
Processing 3100

===================
kaggle_notebooks\mehmetisik_titanic-ml-pipeline-34-step-masterclass.ipynb
['norm_log', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\mehrankazeminia_3-3-g6-snap-to-grid-fix-the-timestamps.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\mehrankazeminia_3-arc24-developed-2020-winning-solutions.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mehrankazeminia_ps3e18-gaussiannb.ipynb
['drop_duplicates']

===================
kaggle_notebooks\melissamonfared_diabetes-prediction-eda-logistic-regression.ipynb
['norm_min_max']

<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:45: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:82: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:82: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:88: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:88: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:88: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:88: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:94: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:94: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
kaggle_notebooks\mesutssmn_disease-symptoms-ml-modelling.ipynb
['norm_min_max']

===================
kaggle_notebooks\mfaaris_content-based-and-tensorflow-recommender-system.ipynb
['norm_min_max']

===================
kaggle_notebooks\mfmfmf3_clean-code-detect-ai-generated.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\mfmfmf3_clean-code-voting-regressor-base-3-models.ipynb
['norm_log']

===================
kaggle_notebooks\microtang_predicting-btc-price-using-rnn.ipynb
['norm_min_max']
Processing 3150

===================
kaggle_notebooks\midouazerty_rainfall-prediction-with-6-machine-learn-algo-98.ipynb
['IQR']

===================
kaggle_notebooks\midouazerty_restaurant-recommendation-system-using-ml.ipynb
['drop_duplicates', 'norm_min_max', 'drop_duplicates']

===================
kaggle_notebooks\mihailodin1_101-pandas-the-solution-from-the-documentation.ipynb
['bin_equal_width_10']

===================
kaggle_notebooks\mihirpaghdal_lung-cancer.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\mikeskim_gold-medal-solution-mike-kim.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\mikhailnaumov_loan-approval-ensemble-nn-xgb-lgbm-cat.ipynb
['drop_duplicates', 'norm_log', 'norm_log', 'IQR']

===================
kaggle_notebooks\mikhailnaumov_regression-with-an-insurance-cat-lgb-xgb-hgb-ydf.ipynb
['drop_duplicates', 'norm_log', 'IQR']

===================
kaggle_notebooks\milankalkenings_feature-engineering-tutorial.ipynb
['norm_min_max']

===================
kaggle_notebooks\milanzdravkovic_pharma-sales-data-analysis-and-forecasting.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\miljan_customer-segmentation.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\miljan_predicting-tags-for-stackoverflow.ipynb
['drop_duplicates']

===================
kaggle_notebooks\minanabil11111212_credit-card-fraud-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mistrzuniu1_tutorial-eda-feature-selection-regression.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mlwhiz_feature-selection-using-football-data.ipynb
['norm_min_max']

===================
kaggle_notebooks\mnassrib_convert-to-regression-random-score.ipynb
['norm_min_max', 'norm_min_max']
Processing 3200

===================
kaggle_notebooks\mohaiminul101_avocado-price-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\mohamedelaziz_customer-churn-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\mohamedkhaledelsafty_intrusion-detection-system-with-binary-classifiers.ipynb
['norm_min_max']

===================
kaggle_notebooks\mohamedmohsen3330_data-analysis-students-performance.ipynb
['IQR']

===================
kaggle_notebooks\mohamedsameh0410_eda-rain-prediction-random-forest-xg-boost.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\mohamedsameh0410_eda-random-forest-heart-disease-prediction-98.ipynb
['norm_min_max']

===================
kaggle_notebooks\mohammadfikri_startup-success-prediction-precision-recall-94.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

<unknown>:15: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
kaggle_notebooks\monolith0456_2xlgbm-fnn-ensemble.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\monthepp_house-prices-advanced-regression-techniques.ipynb
['norm_log', 'norm_log', 'norm_log']
Processing 3250

===================
kaggle_notebooks\motono0223_isic-tabular-model-image-model-features.ipynb
['norm_log']

===================
kaggle_notebooks\motono0223_ubc-infer-cnn-crop-resize-thumbnails.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mouadberqia_bank-churn-prediction-beginner-friendly-0-88959.ipynb
['drop_duplicates']

===================
kaggle_notebooks\msjahid_diabetes-risk-analysis-pima-indians-exploration.ipynb
['IQR', 'zscore', 'winsorize']

===================
kaggle_notebooks\msjahid_exploring-laptop-price-trends.ipynb
['IQR', 'zscore']

===================
kaggle_notebooks\msjahid_iris-diversity-analysis-modeling-prediction.ipynb
['IQR', 'zscore', 'winsorize']

===================
kaggle_notebooks\msjahid_loan-status-analysis-exploring-approval-patterns.ipynb
['IQR', 'zscore', 'winsorize']
Processing 3300

===================
kaggle_notebooks\muhammadaammartufail_tips-and-tricks-to-do-eda-in-desi-style-codanics.ipynb
['IQR']

===================
kaggle_notebooks\muhammadahmed68_credit-card-approval-predictions-85-accuracy.ipynb
['norm_min_max']

===================
kaggle_notebooks\muhammadehabmuhammad_forecasting-employee-retention-streamlit-app.ipynb
['drop_duplicates']

===================
kaggle_notebooks\muhammadfaizan65_flight-price-prediction.ipynb
['drop_duplicates', 'IQR', 'norm_log']

===================
kaggle_notebooks\muhammadfaizan65_parkinsons-disease-analysis.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\muhammadfurqan0_heart-disease-prediction-complete-notebook.ipynb
['norm_min_max']

===================
kaggle_notebooks\muhammadimran112233_employee-promotion-end-to-end-solution.ipynb
['drop_duplicates']

===================
kaggle_notebooks\muhammadsaifwaheed_toxicity-unmasked-nlp-for-hate-speech.ipynb
['drop_duplicates']

===================
kaggle_notebooks\muhammedaliyilmazz_full-mobile-pricing-ml-analysis-step-by-step.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mukuldsagupta_riiid-answer-correctness-prediction-lgbm.ipynb
['drop_duplicates']

===================
kaggle_notebooks\mustafagerme_how-to-calculate-clv-using-python.ipynb
['norm_log']

===================
kaggle_notebooks\mvanshika_diabetes-prediction.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\mysarahmadbhat_chances-of-attack.ipynb
['drop_duplicates', 'IQR', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\myzziah_e-commerce-a-b-testing-full-experiment.ipynb
['drop_duplicates']

===================
kaggle_notebooks\nafisur_predictive-maintenance-using-lstm-on-sensor-data.ipynb
['norm_min_max']
Processing 3350

===================
kaggle_notebooks\nandinibagga_apriori-algorithm.ipynb
['drop_duplicates']

===================
kaggle_notebooks\nareshbhat_outlier-the-silent-killer.ipynb
['IQR', 'isolationForest', 'norm_log', 'IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\nasirislamsujan_bank-customer-churn-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\nathanlauga_ethics-and-ai-how-to-prevent-bias-on-ml.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\nazimcherpanov_0-8922-steel-plate-defect-prediction.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\nazimcherpanov_0-96298-loan-approval-prediction.ipynb
['IQR', 'IQR', 'norm_log', 'norm_log']

===================
kaggle_notebooks\neelkudu28_covid-19-visualizations-predictions-forecasting.ipynb
['norm_log', 'norm_log']

<unknown>:40: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\neupane9sujal_ps04e03-steel-plate-defect-xgboost-oof-preds.ipynb
['drop_duplicates']
Processing 3400

===================
kaggle_notebooks\nicholascomuni_notebook15cd36d95a.ipynb
['bin_equal_width_10', 'bin_equal_width_10', 'bin_equal_width_10']

===================
kaggle_notebooks\nicholasdominic_wids2023-data-buddies.ipynb
['IQR']

===================
kaggle_notebooks\nicholasjhana_multi-variate-time-series-forecasting-tensorflow.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\niharika41298_netflix-visualizations-recommendation-eda.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\nikitakudriashov_top-1-titanic-solution.ipynb
['norm_min_max']

===================
kaggle_notebooks\nilaychauhan_etl-pipelines-tutorial-world-bank-datasets.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'IQR', 'IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\nimapourmoradi_healthcare-stroke.ipynb
['norm_min_max']

===================
kaggle_notebooks\nimapourmoradi_red-wine-quality.ipynb
['norm_min_max']

===================
kaggle_notebooks\nimapourmoradi_water-potability.ipynb
['norm_min_max']

===================
kaggle_notebooks\nina2025_birdclef-2026-eos-9.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 3450

===================
kaggle_notebooks\nischaydnk_covid19-week5-visuals-randomforestregressor.ipynb
['norm_min_max']

===================
kaggle_notebooks\niteshx2_top-50-beginners-stacking-lgb-xgb.ipynb
['norm_log']

===================
kaggle_notebooks\niteshyadav3103_hotel-booking-prediction-99-5-acc.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\niteshyadav3103_medical-cost-eda-regression-tensorflow.ipynb
['norm_min_max']

===================
kaggle_notebooks\niteshyadav3103_titanic-eda-prediction-top-8.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\niyamatalmass_machine-learning-for-time-series-analysis.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\nkitgupta_advance-data-preprocessing.ipynb
['norm_log']

===================
kaggle_notebooks\norbertsolymosi_python-course-2024-03.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\norbertsolymosi_python-course-2024-04.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\nrcjea001_lgbm-baseline-no-leaks-stratifiedgroupkfold.ipynb
['norm_min_max', 'norm_min_max']

<unknown>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.

Processing 3500

===================
kaggle_notebooks\nsff591_popular-ml-nn-cnn-rnn-model-code-snippets.ipynb
['norm_min_max']

===================
kaggle_notebooks\nursrijan_pokemon-tcg-eda-deck-engine.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\nvukobrat_mutual-funds-and-etfs-analysis-python.ipynb
['norm_min_max']

===================
kaggle_notebooks\nyanpn_1st-place-public-2nd-place-solution.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_log']

===================
kaggle_notebooks\oaktechacademy_up-to-date-heart-attack-analysis-and-prediction.ipynb
['zscore', 'zscore', 'winsorize', 'IQR', 'IQR', 'winsorize', 'norm_log']

===================
kaggle_notebooks\octavianwr_employee-churn-dibimbing-id.ipynb
['drop_duplicates']

===================
kaggle_notebooks\odaymourad_detailed-and-typical-solution-ensemble-modeling.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\ohseokkim_creditcard-fraud-balance-is-key-feat-pycaret.ipynb
['norm_log']

===================
kaggle_notebooks\ohseokkim_house-price-all-about-house-price.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\ohseokkim_house-price-simple-but-not-simpler.ipynb
['norm_log', 'norm_log', 'norm_log']

<unknown>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\ohseokkim_house-prices-are-you-a-real-estate-agent.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\ohseokkim_predicting-future-by-lstm-prophet-neural-prophet.ipynb
['norm_min_max']
Processing 3550

===================
kaggle_notebooks\omarkhd99_home-credit-default-risk-challeng.ipynb
['norm_min_max']

===================
kaggle_notebooks\omershect_learning-pytorch-lstm-deep-learning-with-m5-data.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\onedatareader_titanic-data-analysis.ipynb
['norm_min_max']

===================
kaggle_notebooks\onydrive_eda-depression-student-dataset.ipynb
['IQR']

===================
kaggle_notebooks\oscarm524_ps-s3-ep16-eda-modeling-submission.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\`" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\`"? A raw string is also an option.


===================
kaggle_notebooks\oscarm524_ps-s3-ep23-eda-modeling-submission.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\oscarm524_ps-s3-ep24-eda-modeling-submission.ipynb
['drop_duplicates']

===================
kaggle_notebooks\oscarm524_ps-s3-ep25-eda-modeling-submission.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 3600

===================
kaggle_notebooks\ozgurhakan_godaddy-eda-xgb-baseline.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\paradiselost_tutorial-automl-capabilities-of-h2o-library.ipynb
['norm_log']

===================
kaggle_notebooks\param302_iitmbs-mlp-oppe-1-mock-1.ipynb
['norm_min_max']

===================
kaggle_notebooks\param302_mlp-session-25-oppe-1-practice-sep-25.ipynb
['norm_min_max']

===================
kaggle_notebooks\param302_practice-oppe-2.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

<unknown>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:79: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:127: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:175: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.


===================
kaggle_notebooks\patelris_crop-yield-eda-viz.ipynb
['norm_min_max']

===================
kaggle_notebooks\patrickgaspar_esg-fund-performance-analysis.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\paulorzp_laborat-rio-12b-usando-lstm-em-s-ries-temporais.ipynb
['norm_min_max']
Processing 3650

===================
kaggle_notebooks\pavansanagapati_ad-ctr-prediction-with-din-model.ipynb
['norm_min_max']

===================
kaggle_notebooks\pavansanagapati_ensemble-learning-techniques-tutorial.ipynb
['bin_equal_frequency_10', 'norm_log']

===================
kaggle_notebooks\pavansanagapati_google-analytics-simple-exploration.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\payamamanat_bank-loan-classification-models-and-dl.ipynb
['norm_min_max']

===================
kaggle_notebooks\payamamanat_tf-idf-countervec-classification-description.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\pedrodematos_titanic-a-complete-approach-for-data-scientists.ipynb
['norm_min_max']

===================
kaggle_notebooks\perryxiao_chapter-11.ipynb
['norm_min_max']

===================
kaggle_notebooks\philbowman212_life-expectancy-exploratory-data-analysis.ipynb
['winsorize']
Processing 3700

===================
kaggle_notebooks\philschmidt_quora-eda-model-selection-roc-pr-plots.ipynb
['norm_min_max', 'norm_min_max']

<unknown>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:59: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:60: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
kaggle_notebooks\piantic_osic-pulmonary-fibrosis-progression-basic-eda.ipynb
['drop_duplicates']

===================
kaggle_notebooks\pierreholat_3d-exploration-of-papers-ranked-by-keyphrases.ipynb
['drop_duplicates']

===================
kaggle_notebooks\pierreholat_keyphrases-ranking-of-data-supplemented-by-api.ipynb
['drop_duplicates']

===================
kaggle_notebooks\pierremegret_gensim-word2vec-tutorial.ipynb
['drop_duplicates']

===================
kaggle_notebooks\pinuto_ai-cyber-threat-detector.ipynb
['norm_log', 'norm_min_max', 'zscore', 'IQR', 'norm_log']

===================
kaggle_notebooks\pkdarabi_multi-text-classification-f1-score-0-95.ipynb
['drop_duplicates']

===================
kaggle_notebooks\pkdarabi_prediction-of-ticket-cancellation-acc-98.ipynb
['drop_duplicates']

===================
kaggle_notebooks\pmarcelino_comprehensive-data-exploration-with-python.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\pmarcelino_data-analysis-and-feature-extraction-with-python.ipynb
['norm_min_max', 'norm_min_max']
Processing 3750

<unknown>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\pouryaayria_a-complete-ml-pipeline-tutorial-acu-86.ipynb
['isolationForest', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\pramodchandrayan_dimensionality-reduction-using-pca.ipynb
['IQR']

===================
kaggle_notebooks\prasadmenonsrees_project-nlp-sentiment-analysis-twitter-us-air.ipynb
['drop_duplicates']

===================
kaggle_notebooks\prasadperera_the-boston-housing-dataset.ipynb
['norm_log']

===================
kaggle_notebooks\prashant111_a-reference-guide-to-feature-engineering-methods.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\prashant111_extensive-analysis-eda-fe-modelling.ipynb
['norm_min_max']

===================
kaggle_notebooks\prashant111_k-means-clustering-with-python.ipynb
['norm_min_max']

===================
kaggle_notebooks\prashant111_logistic-regression-classifier-tutorial.ipynb
['norm_min_max']

===================
kaggle_notebooks\praveengovi_classify-emotions-in-text-with-bert.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\praxitelisk_microsoft-malware-detection-eda-xgboost.ipynb
['drop_duplicates']

===================
kaggle_notebooks\preejababu_titanic-data-science-solutions.ipynb
['bin_equal_width_5']
Processing 3800

===================
kaggle_notebooks\priyang_credit-card-fraud-detect-under-over-sampling.ipynb
['drop_duplicates']

===================
kaggle_notebooks\priyankdl_titanic-eda-demo-for-students.ipynb
['bin_equal_frequency_5']

<unknown>:5: SyntaxWarning: "\!" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\!"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\ptheru_google-stock-price-prediction-rnn.ipynb
['norm_min_max']

===================
kaggle_notebooks\pythonafroz_eda-heart-disease-prediction-roc-pr-curve.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\pythonafroz_evaluation-metrics-used-in-machine-learning.ipynb
['norm_min_max']

===================
kaggle_notebooks\pythonafroz_heart-disease-prediction-using-11-algorithms.ipynb
['drop_duplicates']

===================
kaggle_notebooks\pythonafroz_titanic-survival-prediction-with-20-algorithm.ipynb
['norm_min_max']

===================
kaggle_notebooks\pythonafroz_transformer-fault-prediction-with-99-auc.ipynb
['norm_min_max']
Processing 3850

===================
kaggle_notebooks\rafjaa_dealing-with-very-small-datasets.ipynb
['isolationForest']

<unknown>:31: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
kaggle_notebooks\raghadalharbi_breast-cancer-survival-prediction-acc-0-779.ipynb
['IQR', 'IQR']

===================
kaggle_notebooks\ragnar123_flm-xlmroberta-inference-baseline.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ragnar123_very-fst-model.ipynb
['drop_duplicates']

===================
kaggle_notebooks\rajacsp_pandas-cheatsheet-125-exercises.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\rajacsp_pandas-dundas-challenge-100.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\rajasekharkv_ericsson-cord-19-challenge-task7-ai007model.ipynb
['drop_duplicates']

===================
kaggle_notebooks\rajeevsharma993_battery-health-nasa-dataset.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\rajjain_github-messages-dataset-visualisation.ipynb
['drop_duplicates']

<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
kaggle_notebooks\rajmehra03_a-complete-text-classfication-guide-word2vec-lstm.ipynb
['drop_duplicates']
Processing 3900

===================
kaggle_notebooks\rajnathpatel_multilingual-text-classification.ipynb
['drop_duplicates']

===================
kaggle_notebooks\rakeshkapilavai_predicting-human-personality.ipynb
['IQR']

===================
kaggle_notebooks\rakibhossainsajib_ddos-detection-using-machine-learning.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\ramsesmdlc_titanic-linear-regression-model.ipynb
['norm_min_max']

===================
kaggle_notebooks\ranasabrii_life-expectancy-regression-with-ann.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\randalaidi_femuna-to-be-submit.ipynb
['drop_duplicates']

===================
kaggle_notebooks\randipratama_resource-forecasting-week4.ipynb
['norm_min_max']

===================
kaggle_notebooks\ranjoranjan_stacking-kernels-lb-0-442.ipynb
['norm_min_max']

===================
kaggle_notebooks\rastislav_mri-brain-tumor-survival-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\ratan123_m5-forecasting-lightgbm-with-timeseries-splits.ipynb
['drop_duplicates']

===================
kaggle_notebooks\rattans_logistic-regression-ps-s6e2.ipynb
['norm_min_max']

===================
kaggle_notebooks\rautaki0127_pokemon-data-science-challenge.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']
Processing 3950

===================
kaggle_notebooks\ravaghi_s05e07-personality-type-prediction-ensemble.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ravaghi_social-action-recognition-in-mice-xgboost.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\ravi20076_houseprice-bootstrappingensembles-pipelines.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\ravi20076_sptitanic-bootstrapensemble-pipeline.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ravi20076_tpsapr22-lgbm-lstm.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'IQR']

===================
kaggle_notebooks\ravi20076_tpsaug22-featureengineering.ipynb
['norm_log', 'IQR', 'IQR']

===================
kaggle_notebooks\ravi20076_tpssep22-featureengineeringpipeline.ipynb
['norm_log', 'drop_duplicates']

===================
kaggle_notebooks\ravivarmaodugu_heartdisease-eda-accuracy-log-loss.ipynb
['norm_min_max']

===================
kaggle_notebooks\ravivarmaodugu_salary-classification-eda-modeling.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\rayalizing1_binary-classification-using-smote-lstm.ipynb
['norm_min_max']

===================
kaggle_notebooks\rdhnw1_covid-19-clinical-trial-results-stage-2.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\realtimshady_2lgbm-2nn.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 4000

<unknown>:17: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\;" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\;"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:59: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:60: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\rehamh_credit-card-fraud-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\rehan597r_23f2002420-assisgment-2.ipynb
['IQR']

===================
kaggle_notebooks\renokan_2-deberta-1-roberta-analysis-and-using.ipynb
['norm_min_max']

===================
kaggle_notebooks\rezasemyari_mobile-price-prediction-0-983.ipynb
['drop_duplicates', 'drop_duplicates', 'IQR', 'IQR']

===================
kaggle_notebooks\rezashokrzad_xgboost-mcc-0-985.ipynb
['drop_duplicates', 'IQR', 'IQR']

===================
kaggle_notebooks\rheajgurung_energy-consumption-forecast.ipynb
['norm_min_max', 'norm_min_max']
Processing 4050

===================
kaggle_notebooks\richeyjay_kidney-stone-prediction-eda-binary-classification.ipynb
['IQR']

===================
kaggle_notebooks\richolson_isic-2024-magic-noise-for-lb-overfit.ipynb
['isolationForest', 'norm_log', 'zscore']

===================
kaggle_notebooks\rikdifos_credit-card-approval-prediction-using-ml.ipynb
['norm_log']

===================
kaggle_notebooks\riteshrhyme_starter-credit-card-scoring-bbe98584-0.ipynb
['isolationForest', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\robinteuwens_anomaly-detection-with-auto-encoders.ipynb
['norm_min_max']
Processing 4100

===================
kaggle_notebooks\rockystats_understanding-auto-encoders.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\rodsaldanha_stock-prediction-pytorch.ipynb
['norm_min_max']

===================
kaggle_notebooks\rohan1506_pandas-tips-tricks-tutorial.ipynb
['bin_equal_frequency_5']

===================
kaggle_notebooks\rohanrao_ashrae-divide-and-conquer.ipynb
['norm_log']

===================
kaggle_notebooks\rounakbanik_ted-data-analysis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\rounakbanik_the-story-of-film.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\roydatascience_ashrae-energy-prediction-using-stratified-kfold.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\roydatascience_light-gbm-with-complete-eda.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\rpsuraj_outlier-detection-techniques-simplified.ipynb
['isolationForest']
Processing 4150

<unknown>:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.


===================
kaggle_notebooks\ruslankl_eeg-data-analysis.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\ryanholbrook_creating-features.ipynb
['drop_duplicates']
Processing 4200

===================
kaggle_notebooks\ryannolan1_kaggle-housing-youtube-video.ipynb
['zscore', 'zscore', 'norm_log']

===================
kaggle_notebooks\ryannolan1_titanic-voting-classifier-0-78947.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\ryotasueyoshi_alakazam-deck-best-5th-place.ipynb
['drop_duplicates']

===================
kaggle_notebooks\rzatemizel_lgbm-catb-xgb-nn-voting-or-stacking.ipynb
['norm_min_max']

===================
kaggle_notebooks\saadmuhammad17_a-beginners-guide-to-data-science-top-3.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_width_5', 'bin_equal_width_5']

===================
kaggle_notebooks\safavieh_ultimate-feature-engineering-xgb-lgb-nn.ipynb
['norm_min_max']

<unknown>:7: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\sahidvelji_cleaning-the-ontario-sunshine-list-data.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\sahillyraina_electric-faults-detection-classification.ipynb
['norm_min_max']

===================
kaggle_notebooks\sahityasetu_neural-network-dl-regression-on-car-price.ipynb
['norm_min_max']

===================
kaggle_notebooks\salehahmedrony_hr-analytics-employee-attrition-eda-prediction.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\salimhammadi07_esc-50-environmental-sound-classification.ipynb
['norm_min_max']

===================
kaggle_notebooks\saloni1712_credit-score-classification.ipynb
['norm_min_max']

===================
kaggle_notebooks\samanfatima7_accurate-classification-simplified.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']
Processing 4250

===================
kaggle_notebooks\samanfatima7_heart-health-insights-eda.ipynb
['zscore']

===================
kaggle_notebooks\samanfatima7_visual-insights-into-online-food-taste.ipynb
['drop_duplicates']

===================
kaggle_notebooks\samiraalipour_genomics-of-drug-sensitivity-in-cancer.ipynb
['drop_duplicates', 'IQR', 'IQR', 'IQR', 'IQR', 'norm_log']

===================
kaggle_notebooks\samlakhmani_easy-92-196-single-model.ipynb
['drop_duplicates']

===================
kaggle_notebooks\samratp_beginner-tutorial-using-votingclassifier-82-27.ipynb
['drop_duplicates']

===================
kaggle_notebooks\samuelcortinhas_credit-cards-data-cleaning.ipynb
['drop_duplicates']

===================
kaggle_notebooks\samuelcortinhas_spaceship-titanic-a-complete-guide.ipynb
['norm_log', 'norm_log']

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:28: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:46: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\+" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\+"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:25: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:27: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
kaggle_notebooks\samuelcortinhas_tps-aug-22-failure-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\samuelcortinhas_tps-jan-22-quick-eda-hybrid-model.ipynb
['norm_log']

===================
kaggle_notebooks\samuelcortinhas_tps-sept-22-timeseries-analysis.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\samuelkali_depression-dataset-analysis-and-machine-learning.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\sandeepbhogaraju_word2vec.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sandragracenelson_lung-cancer-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sanjanabasu_tips-dataset.ipynb
['norm_log']
Processing 4300

===================
kaggle_notebooks\sarazahran1_global-career-prediction-ai-system.ipynb
['norm_log', 'bin_equal_width_10', 'norm_log']

===================
kaggle_notebooks\sarazahran1_road-accident-risk-prediction.ipynb
['norm_log']

===================
kaggle_notebooks\sarazahran1_world-cup-2026-match-predictor.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sasakitetsuya_multivariate-time-series-forecasting-with-lstms.ipynb
['norm_min_max']

===================
kaggle_notebooks\satyaprakashshukl_droput-graduate-analysis.ipynb
['norm_log']

===================
kaggle_notebooks\satyaprakashshukl_h2o-automl-academic-performance.ipynb
['norm_log']

<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\satyaprakashshukl_used-car-price-prediction.ipynb
['IQR']

===================
kaggle_notebooks\saurabhbadole_cardiovascular-disease-prediction.ipynb
['IQR']
Processing 4350

===================
kaggle_notebooks\saurabhbadole_wholesale-customer-purchasing-behavior.ipynb
['IQR']

===================
kaggle_notebooks\saurabhbagchi_fmst-semiconductor-manufacturing-project.ipynb
['IQR', 'IQR', 'isolationForest']

===================
kaggle_notebooks\saurabhshahane_stock-prices-predictions-eda-lstm-deepexploration.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\saurav9786_imdb-score-prediction-for-movies.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sayanroy729_titanic-eda-model-building.ipynb
['norm_min_max']

===================
kaggle_notebooks\seifmechi_credit-default-risk.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\seijoh_wc-gan.ipynb
['norm_min_max']

===================
kaggle_notebooks\selener_multi-class-text-classification-tfidf.ipynb
['drop_duplicates']

===================
kaggle_notebooks\seohyeondeok_yolov3-rsna-starting-notebook.ipynb
['drop_duplicates']

<unknown>:34: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:25: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:42: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.


===================
kaggle_notebooks\sergiosaharovskiy_tps-nov-2021-a-complete-guide.ipynb
['norm_log']

===================
kaggle_notebooks\serigne_stacked-regressions-top-4-on-leaderboard.ipynb
['norm_log']

===================
kaggle_notebooks\serkanpeldek_ev-fiyatlar-n-n-tahmini.ipynb
['norm_min_max']

===================
kaggle_notebooks\serkanpeldek_object-oriented-titanics.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\seuwenfei_online-payment-fraud-detection.ipynb
['bin_equal_width_5', 'bin_equal_width_5', 'bin_equal_width_5']

===================
kaggle_notebooks\seyered_eda-novozymes-enzyme-stability.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\sgedela_30-days-of-ml-competition.ipynb
['IQR', 'drop_duplicates']

===================
kaggle_notebooks\shabnamranjbari_shabnamranjbari-datascience2024.ipynb
['drop_duplicates', 'IQR']
Processing 4400

===================
kaggle_notebooks\shanth84_rnn-detailed-explanation-0-2246.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\sharanharsoor_ctr-analysis-of-different-ml-models.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\sharmasanthosh_exploratory-study-of-ml-algorithms-1.ipynb
['norm_min_max']

===================
kaggle_notebooks\sharmasanthosh_exploratory-study-of-ml-algorithms.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\sharmasanthosh_exploratory-study-on-feature-selection.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\sharmasanthosh_exploratory-study-on-ml-algorithms.ipynb
['norm_log']

===================
kaggle_notebooks\shayanzk_chocolate-sales-complete-eda-ml-pipeline.ipynb
['drop_duplicates']

<unknown>:10: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\sheepwang_leaf-classification-eda-model.ipynb
['drop_duplicates']

===================
kaggle_notebooks\shep312_deep-learning-in-tf-with-upsampling-lb-758.ipynb
['norm_min_max']

===================
kaggle_notebooks\sherinclaudia_movie-rating-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\shilongzhuang_attack-on-titanic-solution-no-data-leakage.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\shilongzhuang_space-titanic-a-beginner-guide-80-24-acc.ipynb
['norm_log', 'norm_log']
Processing 4450

===================
kaggle_notebooks\shiratorizawa_nyse-stock-price-prediction-and-transfer-learning.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\shiratorizawa_wcgan.ipynb
['norm_min_max']

===================
kaggle_notebooks\shirellamosi_sentiment-analysis-nlp.ipynb
['norm_min_max']

===================
kaggle_notebooks\shivamb_exploratory-analysis-ga-customer-revenue.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\shivamb_in-depth-analysis-visualisations-avito.ipynb
['norm_log']

===================
kaggle_notebooks\shivamb_semi-supervised-classification-using-autoencoders.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\shivampanchal_learning-from-the-disaster-99-accuracy.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\shivan118_fifa-world-cup-data-analysis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\shivan118_student-performance-analisys.ipynb
['norm_min_max']

===================
kaggle_notebooks\shivansh002_hit-and-trial-2.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\shivavashishtha_zomato-eda-tutorial.ipynb
['drop_duplicates']

===================
kaggle_notebooks\shivmalhotra26_titanic-survival-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\shree1992_predicting-house-price.ipynb
['drop_duplicates']
Processing 4500

===================
kaggle_notebooks\shrutimechlearn_step-by-step-pca-with-iris-dataset.ipynb
['norm_min_max']

===================
kaggle_notebooks\shtrausslearning_building-an-asset-trading-strategy.ipynb
['norm_min_max']

===================
kaggle_notebooks\shtrausslearning_eda-perth-housing-price-prediction.ipynb
['IQR', 'IQR', 'drop_duplicates']

===================
kaggle_notebooks\shtrausslearning_geospatial-data-visualisation-australia.ipynb
['drop_duplicates']

===================
kaggle_notebooks\shtrausslearning_perth-housing-price-prediction-models.ipynb
['IQR', 'drop_duplicates']

===================
kaggle_notebooks\shtrausslearning_twitter-emotion-classification.ipynb
['norm_min_max']

<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:8: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\shubhlaxmi_liver-disease-competetion-includ-ensembl-adaboost.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\shyamsondagar_beginner-regression-modelling-top-15.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\siavrez_2020fatures.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\sid321axn_gold-price-prediction-using-machine-learning.ipynb
['norm_min_max']

===================
kaggle_notebooks\sid321axn_stacked-ensemble-for-heart-disease-classification.ipynb
['zscore', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\siddheshpujari_eda-and-prediction-of-house-price.ipynb
['norm_log', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\simgeerek_churn-prediction-using-machine-learning.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_10', 'IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\simgeerek_diabetes-prediction-using-classification-models.ipynb
['IQR', 'IQR', 'IQR']

===================
kaggle_notebooks\sinakhorami_titanic-best-working-classifier.ipynb
['bin_equal_width_5']
Processing 4550

<unknown>:32: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:35: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\smokingkrils_indian-e-commerce-analysis-and-customer-retention.ipynb
['norm_min_max']

===================
kaggle_notebooks\smrime_lstm-for-time-series-forecasting.ipynb
['norm_min_max']

===================
kaggle_notebooks\snanilim_video-games-sales-analysis-and-visualization.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\snehagilada_nyc-taxi-fare-eda-random-forest.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\snmahsa_breast-cancer-analysis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\snnclsr_tabular-ensemble-lgbm-catboost.ipynb
['norm_log']

===================
kaggle_notebooks\soham1024_titanic-data-science-eda-with-meme-solution.ipynb
['bin_equal_width_5']
Processing 4600

===================
kaggle_notebooks\someadityamandal_bitcoin-time-series-forecasting.ipynb
['norm_min_max']

===================
kaggle_notebooks\sonalisingh1411_customer-churn-eda-top-5-models-95.ipynb
['norm_min_max']

===================
kaggle_notebooks\sonalisingh1411_detailed-eda-placements-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sonalisingh1411_employee-trends-storytelling-predictive-insights.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sonalisingh1411_red-wine-quality-prediction-87-accuracy-using-pca.ipynb
['drop_duplicates', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\songulerdem_health-insurance-cross-sell-prediction-xgboost.ipynb
['drop_duplicates', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\soumya044_disease-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sowmya96_spotify-song-prediction-and-recommendation-system.ipynb
['norm_min_max']

===================
kaggle_notebooks\srinivas24_predict-the-burned-area-of-forest-fires.ipynb
['norm_min_max']

===================
kaggle_notebooks\startupsci_titanic-data-science-solutions.ipynb
['bin_equal_width_5']

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:14: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:28: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:30: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:37: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:38: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:45: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:46: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:47: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:62: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:63: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:65: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:66: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:67: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:68: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:69: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:70: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:71: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:87: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:89: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:91: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:95: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:96: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:104: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:105: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:108: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:109: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:111: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:112: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:113: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:114: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:115: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:116: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:117: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:118: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:119: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:120: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:122: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:123: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:124: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:127: SyntaxWarning: "\|" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\|"? A raw string is also an option.
<unknown>:128: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:129: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:130: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:131: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:132: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:133: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:134: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:135: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:136: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:137: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:138: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:139: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:140: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:141: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:142: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:143: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:144: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:145: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:146: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:147: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:148: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:149: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:150: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:151: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:152: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:153: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:154: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:155: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:156: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:157: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:162: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:163: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:166: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:167: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:168: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:169: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:170: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:171: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:172: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:173: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:174: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:175: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:176: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:177: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:178: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:179: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:180: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:181: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:182: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
<unknown>:183: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:184: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:185: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:186: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:187: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:188: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:189: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:190: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:191: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:192: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:193: SyntaxWarning: "\^" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\^"? A raw string is also an option.
<unknown>:194: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:195: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:196: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:197: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:198: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:199: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:200: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:201: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:202: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:203: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:204: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:205: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:206: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:207: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:208: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:209: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:210: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:211: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:212: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:213: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:214: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<unknown>:217: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:218: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:220: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:221: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.


===================
kaggle_notebooks\stefanozakher94_eda-and-forecasting-with-rfregressor-final-updated.ipynb
['bin_equal_width_5']
Processing 4650

===================
kaggle_notebooks\sudalairajkumar_simple-exploration-baseline-ga-customer-revenue.ipynb
['norm_log']

===================
kaggle_notebooks\sudalairajkumar_where-do-people-learn-ml-ds.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\sudhanshu2198_oil-spill-classification.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\sudhirnl7_linear-regression-tutorial.ipynb
['norm_log']

===================
kaggle_notebooks\sugataghosh_e-commerce-text-classification-tf-idf-word2vec.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sugghi_training-3rd-place-solution.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\sulaniishara_lightgbm-unleashed-premiums-decoded.ipynb
['norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\sulaniishara_plant-health-prediction-with-ml.ipynb
['zscore']

===================
kaggle_notebooks\sulaniishara_student-stress-performance-insights.ipynb
['isolationForest']

===================
kaggle_notebooks\sumedh1507_predicting-phone-addiction-level.ipynb
['drop_duplicates']

===================
kaggle_notebooks\sumitchavhan7_student-performance-dataset.ipynb
['drop_duplicates', 'bin_equal_width_5']

===================
kaggle_notebooks\suneelpatel_graduate-admission-analysis-and-prediction.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\supawitongkariyapong_project-credit-score-classification.ipynb
['drop_duplicates', 'IQR', 'norm_min_max']

<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.


===================
kaggle_notebooks\supreethrao_bert-s-a-stock-market-guru-86-22-huggingface.ipynb
['drop_duplicates']

===================
kaggle_notebooks\suprematism_ml-house-prices-top-7-encoding-techniques.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\suraj520_40-pandas-functions-their-polars-equivalent.ipynb
['drop_duplicates']
Processing 4700

===================
kaggle_notebooks\surekharamireddy_e-commerce-data-set.ipynb
['drop_duplicates']

===================
kaggle_notebooks\surya635_house-price-prediction.ipynb
['norm_log']

===================
kaggle_notebooks\suryanshsharma1_lgbm-nn-fusion-xgb-ensemble.ipynb
['norm_min_max']

===================
kaggle_notebooks\suvroo_complete-nlp-pipeline.ipynb
['drop_duplicates']

===================
kaggle_notebooks\suvroo_ps4e7-optuna-xgboost-klib.ipynb
['zscore', 'drop_duplicates']

===================
kaggle_notebooks\swandipsingha_s5e3-eda-xgb-lgbm-cnn.ipynb
['drop_duplicates']

===================
kaggle_notebooks\swathiunnikrishnan_consumer-behaviour-analysis-of-amazon-a-study.ipynb
['norm_min_max']

===================
kaggle_notebooks\syedali110_car-price-prediction-and-visualization.ipynb
['IQR']

===================
kaggle_notebooks\sz8416_6-ways-for-feature-selection.ipynb
['norm_min_max']

===================
kaggle_notebooks\szhou42_predict-future-sales-top-11-solution.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tahmidmir_dark-triad.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tahmidmir_house-prices-advanced-regression-techniques.ipynb
['norm_min_max']

===================
kaggle_notebooks\tahmidmir_predicting-heart-disease.ipynb
['norm_min_max']

<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.

Processing 4750

===================
kaggle_notebooks\tanayatipre_stress-level-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tanmay111999_avocado-price-forecast-arima-sarima-detailed.ipynb
['norm_log']

===================
kaggle_notebooks\tanmay111999_clustering-pca-k-means-dbscan-hierarchical.ipynb
['norm_min_max']

===================
kaggle_notebooks\tanmay111999_diabetes-classification-xgb-lgbm-stack-smote.ipynb
['norm_min_max']

===================
kaggle_notebooks\tanmay111999_heart-failure-prediction-cv-score-90-5-models.ipynb
['norm_min_max']

===================
kaggle_notebooks\tanmay111999_heart-failure-prediction-eda-model-comparison.ipynb
['norm_min_max']

===================
kaggle_notebooks\tanmay111999_hr-analytics-data-leakage-eda-f1-score-80.ipynb
['norm_min_max']

===================
kaggle_notebooks\tanmay111999_stroke-prediction-effect-of-data-leakage-smote.ipynb
['norm_min_max']

===================
kaggle_notebooks\tanmay111999_telco-churn-eda-cv-score-85-f1-score-80.ipynb
['norm_min_max']

===================
kaggle_notebooks\tanmay111999_unsupervised-learning-3-6-clusters-k-means-eda.ipynb
['norm_min_max']

<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.


===================
kaggle_notebooks\tarekhassan024_panda-cheat-shit-for-insurance-prediction.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\tarekmuhammed_classification-project-titanic-dataset.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tarequebasharovi_random-forest-classifier-ovi.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tarkkaanko_diabetes-feature-engineering-prediction.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\taronzakaryan_predicting-stock-price-using-lstm-model-pytorch.ipynb
['norm_min_max']

===================
kaggle_notebooks\tarundirector_backpack-pred-baseline-ensemble-eda.ipynb
['norm_log', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\tarundirector_binary-classification-bank-churn-eda.ipynb
['norm_log', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\tarundirector_rev-rain-pred-eda-time-series-ai-news.ipynb
['norm_log', 'norm_min_max']
Processing 4800

<unknown>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:61: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:54: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:128: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:195: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:381: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\tarunpaparaju_vsb-competition-attention-bilstm-with-features.ipynb
['norm_min_max']

===================
kaggle_notebooks\tatudoug_stock-embedding-ffnn-features-of-the-best-lgbm.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\tatudoug_stock-embedding-ffnn-my-features.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\tcmaso_mnist-guide-cnn-augmentation-tuning-99-5.ipynb
['norm_min_max']

===================
kaggle_notebooks\teckmengwong_tps2201-hybrid-time-series.ipynb
['norm_log', 'norm_min_max']

===================
kaggle_notebooks\tejasurya_wind-power-generation-in-germany.ipynb
['IQR']

===================
kaggle_notebooks\terryyue_data-analysis-tutorial-for-beginners.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\tetsutani_ps3e13-eda-decomposition-ensemble-rankpredict.ipynb
['norm_min_max']

<unknown>:81: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:87: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.


===================
kaggle_notebooks\tetsutani_ps3e16-eda-ensemble-ml-pipeline.ipynb
['norm_min_max']

===================
kaggle_notebooks\tetsutani_ps3e17-eda-ensemble-ml-pipeline-shap.ipynb
['norm_min_max']

===================
kaggle_notebooks\tetsutani_ps3e18-eda-ensemble-ml-pipeline-binarypredictict.ipynb
['norm_min_max', 'norm_min_max', 'drop_duplicates']

===================
kaggle_notebooks\tetsutani_ps3e19-eda-ensemble-ml-pipeline-rnn-by-skorch.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\tetsutani_ps3e8-xgb-lgbm-cat-ensemble-baseline.ipynb
['drop_duplicates']

===================
kaggle_notebooks\thanepi_journal-clean-logistic-regression-decision-tree.ipynb
['drop_duplicates']

===================
kaggle_notebooks\thangnm1_baseline-public-0-44.ipynb
['drop_duplicates']

===================
kaggle_notebooks\thebrownviking20_intro-to-recurrent-neural-networks-lstm-gru.ipynb
['norm_min_max']
Processing 4850

===================
kaggle_notebooks\theeyeschico_crop-analysis-and-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\theoviel_it-s-that-time-of-the-year-again.ipynb
['drop_duplicates']

===================
kaggle_notebooks\theoviel_using-last-year-s-2nd-place.ipynb
['drop_duplicates']

===================
kaggle_notebooks\therealsampat_early-stage-diabetes-prediction.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\thiagomantuani_carprice-eda-model-get-started.ipynb
['drop_duplicates']

===================
kaggle_notebooks\thiagomantuani_ps4e03-steel-plate-defect-for-beginners.ipynb
['norm_log']

===================
kaggle_notebooks\thiagomantuani_rohlik-orders-2024-eda-modeling-get-started.ipynb
['drop_duplicates']

===================
kaggle_notebooks\thiagomantuani_wids-2025-baseline.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\thiagopanini_e-commerce-sentiment-analysis-eda-viz-nlp.ipynb
['drop_duplicates']

===================
kaggle_notebooks\thiagopanini_exploring-and-modeling-housing-prices.ipynb
['norm_log']

===================
kaggle_notebooks\thiagopanini_insights-from-netflix-the-show-must-go-on.ipynb
['norm_log']

<unknown>:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:29: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<unknown>:9: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.


===================
kaggle_notebooks\thisishusseinali_malicious-url-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\thomasmeiner_ps4e2-eda-feature-engineering-modelling.ipynb
['isolationForest']
Processing 4900

===================
kaggle_notebooks\timolee_a-home-for-pandas-and-sklearn-beginner-how-tos.ipynb
['norm_log']

===================
kaggle_notebooks\todnewman_keras-neural-net-for-champs.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tolgahancepel_lightgbm-single-model-and-feature-engineering.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\tomasmantero_predicting-house-prices-keras-ann.ipynb
['norm_min_max']

===================
kaggle_notebooks\tombresee_next-gen-eda.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\tomooinubushi_postprocessing-based-on-leakage.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\toshimelonhead_ncaa-march-madness-sabermetric-spin.ipynb
['drop_duplicates', 'norm_min_max']

<unknown>:25: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<unknown>:39: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<unknown>:40: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<unknown>:24: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<unknown>:22: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:11: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:15: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.


===================
kaggle_notebooks\travelcodesleep_end-to-end-regression-pipeline-using-scikitlearn.ipynb
['IQR']
Processing 4950

===================
kaggle_notebooks\tronrover_among-the-elite-top-100-spaceship-titanic.ipynb
['norm_log']

===================
kaggle_notebooks\trupologhelper_boosting-synergy-six-model-blend-for-loan-predict.ipynb
['norm_log', 'bin_equal_frequency_5', 'norm_log', 'bin_equal_frequency_5', 'bin_equal_width_5']

===================
kaggle_notebooks\tshephisho_ecommerce-behaviour-using-xgboost.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\tumpanjawat_coffee-eda-geo-cluster-regression.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\tumpanjawat_diabetes-eda-random-forest-hp.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tumpanjawat_ds-salary-full-eda-geo-cluster-xgboost.ipynb
['norm_min_max', 'IQR']

===================
kaggle_notebooks\tumpanjawat_eda-and-handling-missing-value.ipynb
['norm_log', 'drop_duplicates']

===================
kaggle_notebooks\tumpanjawat_heart-attack-eda-cluster-8-ml-models.ipynb
['drop_duplicates']

===================
kaggle_notebooks\tumpanjawat_heart-disease-eda-fe-resam-xgboost.ipynb
['drop_duplicates', 'drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\tumpanjawat_ps3e16-eda-cluster-ensemble-xg-cat.ipynb
['norm_min_max']

===================
kaggle_notebooks\tumpanjawat_stroke-prediction-eda-resampling-xgboost.ipynb
['IQR', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\tunguz_simple-linear-regression-benchmark.ipynb
['norm_log']

===================
kaggle_notebooks\tuosun493_credit-card-approval-prediction-eda.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ulrich07_osic-multiple-quantile-regression-starter.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ulyanovantonamaranta_birdclef-2026-gate-fake008-head0015.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\umerkk12_credit-card-predictive-analysis.ipynb
['drop_duplicates', 'norm_min_max']

<unknown>:47: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.

Processing 5000

===================
kaggle_notebooks\uomislab_instacart-xgboost-gridsearch-notebook.ipynb
['drop_duplicates']

===================
kaggle_notebooks\uomislab_mbads-2023-24.ipynb
['drop_duplicates']

===================
kaggle_notebooks\utcarshagrawal_water-quality-prediction-using-sparkml.ipynb
['drop_duplicates']

===================
kaggle_notebooks\utkarshm25_data-preprocessing-basics.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\valkling_mercari-rnn-2ridge-models-with-notes-0-42755.ipynb
['norm_log']

<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\varunsaikanuri_chennai-houses-sales-analysis-and-prediction.ipynb
['norm_min_max']

===================
kaggle_notebooks\varunsaikanuri_financial-crisis-analysis-and-prediction.ipynb
['norm_min_max']
Processing 5050

===================
kaggle_notebooks\varunsaikanuri_flight-fare-prediction-10-ml-models.ipynb
['norm_min_max']

===================
kaggle_notebooks\varunsaikanuri_spotify-data-visualization.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vbmokin_20-models-for-cardiovascular-disease-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vbmokin_50-advanced-tips-data-science-for-tabular-data.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vbmokin_50-tips-data-science-tabular-data-for-beginner.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\vbmokin_autoselection-from-20-classifier-models-l-curves.ipynb
['norm_min_max']

===================
kaggle_notebooks\vbmokin_convert-to-regression-with-tuning.ipynb
['norm_min_max']

===================
kaggle_notebooks\vbmokin_crypto-btc-advanced-analysis-forecasting.ipynb
['norm_min_max']

===================
kaggle_notebooks\vbmokin_heart-disease-automatic-adveda-fe-20-models.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\vbmokin_higher-lb-score-by-tuning-mloss-upgrade-visual.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vbmokin_stock-embedding-ffnn-upgrade-3d.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\vencerlanz09_electric-cars-eda-with-feature-engineering.ipynb
['drop_duplicates']

<unknown>:5: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\venky12347_insurance-premium.ipynb
['drop_duplicates']

===================
kaggle_notebooks\venky73_icc-cricket-world-cup-2019-analysis.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\verracodeguacas_7-se7en-prompts.ipynb
['drop_duplicates']

===================
kaggle_notebooks\verracodeguacas_spacy-linguistic-features-svr-and-optuna.ipynb
['norm_min_max']

===================
kaggle_notebooks\vetrirah_beginner-10-step-solution-janatahack-healthcare.ipynb
['drop_duplicates', 'norm_log', 'norm_log']

===================
kaggle_notebooks\vetrirah_beginner-time-series-in-iot-top-10-solution.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vetrirah_post-hackathon-analysis-3-ml-models-in-gpu.ipynb
['drop_duplicates', 'norm_log', 'norm_min_max']

===================
kaggle_notebooks\vetrirah_top-5-winning-solution-customer-segmentation.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vexxingbanana_sartorius-mmdetection-training.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vicsonsam_sst-eda-17-models-dl-top-7.ipynb
['norm_min_max']
Processing 5100

===================
kaggle_notebooks\victorambonati_unsupervised-anomaly-detection.ipynb
['isolationForest']

===================
kaggle_notebooks\vikasukani_detecting-parkinson-s-disease-machine-learning.ipynb
['norm_min_max']

===================
kaggle_notebooks\viktortaran_space-titanic.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'isolationForest']

<unknown>:2: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:43: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:59: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<unknown>:95: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.


===================
kaggle_notebooks\vinayak123tyagi_damage-propagation-modeling-for-aircraft-engine.ipynb
['norm_min_max']

===================
kaggle_notebooks\vinayakshanawad_industrial-safety-complete-solution.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vinayshaw_airfare-price-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vincentgupo_classifying-cyberbullying-94-accuracy.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vincentschuler_enefit-baseline-cross-validation.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vipin20_breast-cancer-classification-eda-with-score-0-99.ipynb
['IQR']

===================
kaggle_notebooks\vipin20_heart-attack-analysis-prediction-eda.ipynb
['drop_duplicates', 'norm_min_max']

===================
kaggle_notebooks\vishnu123_tps-aug-22-top-2-logistic-regression-cv-fe.ipynb
['norm_log']

===================
kaggle_notebooks\vishnupriyagarige_forecasting-sticker-sales.ipynb
['drop_duplicates', 'norm_log']

===================
kaggle_notebooks\vishnupriyagarige_obesity-risk.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vishnupriyagarige_used-car-price-prediction.ipynb
['drop_duplicates']
Processing 5150

===================
kaggle_notebooks\vjgupta_reach-top-10-with-simple-model-on-housing-prices.ipynb
['norm_log']

===================
kaggle_notebooks\volhaleusha_titanic-tutorial-encoding-feature-eng-81-8.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\volodymyrgavrysh_bank-marketing-campaigns-dataset-analysis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vslaykovsky_train-pytorch-aux-targets-weighted-loss-thres.ipynb
['bin_equal_frequency_10']

===================
kaggle_notebooks\vuppalaadithyasairam_98-test-accuracy-thyroid-prediction.ipynb
['drop_duplicates']

===================
kaggle_notebooks\vyankteshdwivedi_birdclef-2026-onnx-perch-sequence-modeling.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\waalbannyantudre_bigmart-sales-prediction-project.ipynb
['IQR']

===================
kaggle_notebooks\waalbannyantudre_crab-age-predictions-eda-f-e-modeling-10th.ipynb
['drop_duplicates']

===================
kaggle_notebooks\wassimderbel_nasa-predictive-maintenance-rul.ipynb
['norm_min_max']

<unknown>:18: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<unknown>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:23: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:41: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:53: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.

Processing 5200

===================
kaggle_notebooks\wchan757_achieving-lb-0-47-with-just-lightgbm-detail.ipynb
['drop_duplicates']

===================
kaggle_notebooks\werooring_ch7-modeling.ipynb
['norm_min_max']

===================
kaggle_notebooks\werooring_top-3-5-lightgbm-with-feature-engineering.ipynb
['drop_duplicates']

===================
kaggle_notebooks\what0919_intrusion-detection-classification-by-jinner.ipynb
['drop_duplicates']

===================
kaggle_notebooks\wikaiqi_titaniclearningqi.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\williamsabodunrin_kernel174ab58047.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\willkoehrsen_a-complete-introduction-and-walkthrough.ipynb
['norm_min_max']

===================
kaggle_notebooks\willkoehrsen_start-here-a-gentle-introduction.ipynb
['norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\winfredmumbingure_c02-emission-africa.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\winternguyen_used-car-price-estimation-96-accuracy.ipynb
['norm_log']

===================
kaggle_notebooks\winternguyen_water-pump-maintenance-shutdown-prediction.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\wissams_titanic-competition-step-by-step-using-xgboost.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\wltjd54_insurance-prediction-full-ver.ipynb
['IQR']

===================
kaggle_notebooks\wonghoitin_centralized-examples-not-federated.ipynb
['norm_log']
Processing 5250

===================
kaggle_notebooks\x1wello1x_prediction-of-strokes.ipynb
['drop_duplicates']

===================
kaggle_notebooks\xiangmeng123_transformers-gbm-ensemble.ipynb
['drop_duplicates']

===================
kaggle_notebooks\xiaocao123_lb-0-45.ipynb
['drop_duplicates']

===================
kaggle_notebooks\xiefei_01-titanic.ipynb
['bin_equal_width_5']

===================
kaggle_notebooks\xiyuewang_lol-how-to-win.ipynb
['norm_min_max']

===================
kaggle_notebooks\yaaangzhou_pg-s3-e22-eda-modeling.ipynb
['drop_duplicates']

===================
kaggle_notebooks\yaaangzhou_pg-s3-e24-eda-modeling-ensemle-nn.ipynb
['isolationForest']

===================
kaggle_notebooks\yaaryiitturan_credit-score-prediction-using-ann-smote.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\yacermeftah_email-spam-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\yaheaal_loan-status-with-different-models.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\yahiaomar_now-you-can-eat-mushrooms-acc-0-9922.ipynb
['IQR']

===================
kaggle_notebooks\yairhadad1_cnn-for-handwritten-alphabets.ipynb
['norm_min_max']

<unknown>:4: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:5: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:6: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<unknown>:31: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:32: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:33: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:34: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.


===================
kaggle_notebooks\yannisp_sf-crime-analysis-prediction.ipynb
['drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\yantxx_xgboost-binary-classifier-machine-failure.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\yash9439_loan-prediction-and-visualisation.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\yashvi_vehicle-insurance-eda-and-boosting-models.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\yasserhessein_classification-maternal-health-5-algorithms-ml.ipynb
['drop_duplicates']
Processing 5300

===================
kaggle_notebooks\yasserh_breast-cancer-diagnosis-best-ml-algorithms.ipynb
['drop_duplicates']

===================
kaggle_notebooks\yasserh_email-spam-detection-comparing-best-ml-models.ipynb
['drop_duplicates']

===================
kaggle_notebooks\yasserh_housing-price-prediction-best-ml-algorithms.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\yasserh_insurance-claim-prediction-top-ml-models.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\yasserh_online-customer-segmentation-clustering-approach.ipynb
['norm_log']

===================
kaggle_notebooks\yasserh_song-popularity-prediction-best-ml-models.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\yasserh_uber-fare-prediction-comparing-best-ml-models.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\yasserh_walmart-sales-prediction-best-ml-algorithms.ipynb
['drop_duplicates', 'IQR']

<unknown>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:13: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:17: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:20: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:16: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:18: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:19: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<unknown>:21: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.

Processing 5350

===================
kaggle_notebooks\yeechern_reddit-depression-classification-cnn-lstm-rf.ipynb
['drop_duplicates']

===================
kaggle_notebooks\yekenot_explore-ts-with-lstm.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\yekenot_mcts-deeptables-nn.ipynb
['norm_min_max']

===================
kaggle_notebooks\yiqingge_m15-prediction-final-version.ipynb
['drop_duplicates']

===================
kaggle_notebooks\yixinchen1_ashrae-1-1-to-1-06-with-ucl.ipynb
['norm_log']

===================
kaggle_notebooks\ykojima1989_fine-tune-bert-use-on-sklearn-pipeline.ipynb
['norm_min_max']

===================
kaggle_notebooks\yogidsba_predict-used-car-prices-linearregression.ipynb
['norm_log']

===================
kaggle_notebooks\yogidsba_travelpackageprediction-ensemble-techniques.ipynb
['IQR']

===================
kaggle_notebooks\yoohwanseol_pandas-cheatsheet-125-exercises.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

<unknown>:10: SyntaxWarning: "\/" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\/"? A raw string is also an option.
<unknown>:10: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:7: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<unknown>:26: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:44: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:55: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.


===================
kaggle_notebooks\yossefmohammed_true-and-fake-news-lstm-accuracy-97-90.ipynb
['drop_duplicates']

===================
kaggle_notebooks\youhanlee_stratified-sampling-for-regression-lb-1-4627.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log']
Processing 5400

===================
kaggle_notebooks\yousefalbasel_oulad-personalized-learning-path-recommender-sys.ipynb
['norm_min_max']

===================
kaggle_notebooks\yousefmohamed20_titanic.ipynb
['norm_min_max']

===================
kaggle_notebooks\youssefaboelwafa_hotel-booking-cancellation-multiple-models.ipynb
['IQR']

===================
kaggle_notebooks\ysjf13_cis-fraud-detection-visualize-feature-engineering.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\ysthehurricane_bitcoin-dogecoin-etc-price-prediction-xgboost.ipynb
['norm_min_max']

===================
kaggle_notebooks\ysthehurricane_stock-market-predictions-with-5-algorithms.ipynb
['norm_min_max']

===================
kaggle_notebooks\ysthehurricane_tesla-stock-price-prediction-using-gru-tutorial.ipynb
['norm_min_max']

===================
kaggle_notebooks\yufengsui_ml-project-bank-telemarketing-analysis.ipynb
['zscore']

===================
kaggle_notebooks\yunsuxiaozi_cnn-lstm.ipynb
['norm_min_max']

===================
kaggle_notebooks\yunsuxiaozi_isic-2024-starter.ipynb
['norm_log', 'norm_log', 'norm_log', 'norm_log', 'norm_log']

===================
kaggle_notebooks\zabihullah18_car-price-prediction.ipynb
['norm_log']
Processing 5450

===================
kaggle_notebooks\zabihullah18_email-spam-detection.ipynb
['drop_duplicates']

===================
kaggle_notebooks\zain280_bank-customer-churn-prediction-analysis.ipynb
['IQR', 'norm_min_max']

===================
kaggle_notebooks\zaralavii_loan-credit-prediction.ipynb
['norm_log']

===================
kaggle_notebooks\zeeshanlatif_brain-tumor-segmentation-using-u-net.ipynb
['norm_min_max']

===================
kaggle_notebooks\zeeshanlatif_pandas-tutorial.ipynb
['drop_duplicates']

===================
kaggle_notebooks\zeeshanyounas001_heart-disease-uci.ipynb
['IQR']

===================
kaggle_notebooks\zeeshanyounas001_titanic-dataset-analysis.ipynb
['IQR']

===================
kaggle_notebooks\zephyrwang666_riiid-lgbm-bagging2-1.ipynb
['drop_duplicates']

===================
kaggle_notebooks\zhejing178_descriptive-statistics-answers.ipynb
['zscore']

===================
kaggle_notebooks\zhukovoleksiy_5-solution-ps3e13-ensemble.ipynb
['norm_min_max']

===================
kaggle_notebooks\zhukovoleksiy_icr-stacking-xgb-models.ipynb
['norm_log', 'norm_log']

===================
kaggle_notebooks\zhukovoleksiy_ps-s3e14-simple-eda-ensemble.ipynb
['norm_min_max']

===================
kaggle_notebooks\zhukovoleksiy_ps-s3e22-eda-preprocessing-ensemble.ipynb
['drop_duplicates']

===================
kaggle_notebooks\zhukovoleksiy_ps-s3e23-explore-data-stacking-ensemble.ipynb
['drop_duplicates']

===================
kaggle_notebooks\zhy450324080_death-prediction-battle-analysis.ipynb
['drop_duplicates']

===================
kaggle_notebooks\ziadaymantesla_horses-max-score-0-83536.ipynb
['drop_duplicates', 'IQR']

===================
kaggle_notebooks\zikazika_analysis-of-world-crime.ipynb
['drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates', 'drop_duplicates']

===================
kaggle_notebooks\zikazika_using-rnn-and-arima-to-predict-bitcoin-price.ipynb
['norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\zlatankr_titanic-random-forest-82-78.ipynb
['bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5', 'bin_equal_frequency_5']

===================
kaggle_notebooks\zolboo_recommender-systems-knn-svd-nn-keras.ipynb
['drop_duplicates']

===================
kaggle_notebooks\zoupet_neural-network-model-for-house-prices-tensorflow.ipynb
['isolationForest', 'norm_min_max', 'norm_min_max', 'norm_min_max', 'isolationForest', 'norm_min_max', 'norm_min_max', 'norm_min_max']

===================
kaggle_notebooks\zsinghrahulk_crop-yield-estimation-regression.ipynb
['norm_min_max']

<unknown>:11: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:52: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:59: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<unknown>:12: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<unknown>:3: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.

"""
list_strings = re.findall(r"^\[.*\]$", text, flags=re.MULTILINE)

# Convert them into actual Python lists
lists = [ast.literal_eval(s) for s in list_strings]

# print("Lists:")
# print(lists)

print(f"found {len(lists)} pipes")
# Average list size
avg_size = sum(len(lst) for lst in lists) / len(lists) if lists else 0

print(f"\nAverage list size: {avg_size:.2f}")

# Save to CSV (one row per list)
df = pd.DataFrame({
    "list": [str(lst) for lst in lists],
    "size": [len(lst) for lst in lists]
})
df.to_csv("pipelinesDataPrep_kaggle.csv", index=False)

print("\nSaved to lists.csv")

In [ ]:
(
    transform_probabilities_combined,
    transition_probabilities_combined,
) = analyze_corpus(["kaggle_notebooks", "notebooks"])

In [ ]:
prob_dict_combined = {}
eps = 1e-10
for transform_op in transformations:
    prob_dict_combined[transform_op] = transform_probabilities_combined.get(transform_op, eps)

prob_dict_combined['zscore_clip_3'] = transform_probabilities_combined.get('zscore', eps)
prob_dict_combined['zscore_filter_3'] = transform_probabilities_combined.get('zscore', eps)
print(prob_dict_combined)
